# [실습] 민원 키워드 표 — 형태소로 다시 집계하기

_이 노트북은 LMS에서 내보냈습니다. 영상·퀴즈는 학습 참고용으로 마크다운으로 변환되었습니다._

## 질문과 데이터

<div style="background:#FFFFFF;color:#000000;padding:40px;border-radius:12px;border:1px solid #E6E6E6;border-left:4px solid #051C2C">
<div style="font-size:0.9em;color:#656565">비정형 데이터(자연어) · 문서 비율과 형태소 토큰</div>
<h1 style="color:#000000;margin:10px 0">민원 키워드 표 — 형태소로 다시 집계하기</h1>
<div style="font-size:1.05em;color:#656565">cp949 로 인코딩된 민원 원본을 직접 읽고, 표본 2만 건의 형태소 명사로 키워드 표를 만든 뒤, 어절 기준 표와 품사 선택 두 설정을 비교합니다</div>
</div>

## 핵심 질문

> 2020년 소비자 민원 제목은 무엇을 가장 많이 말하고, 공백으로 자른 어절 기준 표와 형태소 기준 표는 같은 순위를 내는가?

이론에서 2020년 민원 제목 499,844건의 형태소 명사를 집계했습니다. 여기서 **불용어**는 「문의」·「요청」처럼 주제와 상관없이 여러 민원 제목에 되풀이해 나오는 말입니다. 불용어를 빼면 가장 많이 쓴 말은 「환불」로, 86,903건이 이 말을 썼습니다.

**문서 비율**은 그 말을 쓴 문서 수 ÷ 전체 문서 수입니다. 「환불」의 문서 비율은 17.39% 였습니다(95% 구간 17.28～17.49%).

그런데 같은 제목을 공백으로 자른 어절 단위로 집계하면 「위약금」 4.30% 가 「해지」 3.80% 보다 높았습니다. 형태소 단위로 집계하면 반대로 「해지」 11.13% 가 「위약금」 5.24% 의 두 배를 넘었습니다. 집계 단위만 바꿨는데 두 낱말의 순위가 반대가 되었습니다. 실습에서는 표본 2만 건으로 같은 계산을 직접 합니다.

미션 1 은 민원 원본 파일을 읽어 표본 2만 건의 명사 문서 비율 상위 20 을 그리고 상위 3 낱말마다 원문 3건을 읽습니다. 미션 2 는 전임 담당자의 어절 기준 표를 고칩니다. 이 표의 결론은 「위약금 민원이 해지 민원보다 많다」였습니다. 미션 3 은 쇼핑 리뷰의 글자 잡음을 정리하고 품사 선택 두 설정(명사만 / 명사 + 용언)의 상위 15 를 비교합니다. 과제는 국외여행 민원의 키워드 표를 만듭니다.

## 데이터

| 자료 | 출처 · 이용 조건 | 크기 | 실습에서 쓰는 부분 |
|---|---|---|---|
| 민원 파일 1 | 공정거래위원회 「소비자 민원학습데이터 소비자상담 접수내역」 1번 파일 (data.go.kr 15098314, 이용허락범위 제한 없음) | cp949 67,907,945바이트 · 499,999행 × 13열 · 접수일자 2020-01-01 ～ 2020-10-17 | 가공 뒤 499,844행의 사건제목. 미션 1 · 2 는 무작위 2만 건(시드 0), 과제는 품목 국외여행 13,012건 |
| 쇼핑 리뷰 | 네이버 쇼핑 리뷰 (GitHub bab2min/corpus, README 「Public Domain」) | UTF-8 · 200,000행 · 별점 1 · 2 · 4 · 5 (3점이 빠진 가공본) | 미션 3: 별점 1 · 2점 리뷰에서 무작위 1만 건(시드 0) |

「준비 셀」은 과정이 GitHub 에 원 출처와 같은 바이트로 복제해 둔 파일을 받아 cp949 로 읽습니다. 그다음 시험 입력 91건과 빈 제목 64건을 빼고 전화번호 형태가 든 제목 10건을 `[전화번호]` 로 바꿉니다. 민원에서 쓰는 열은 4개입니다.

| 열 | 뜻 |
|---|---|
| `사건제목` | 민원 한 건의 제목(가공 뒤 글자 수 중앙값 20자). 소비자가 쓴 그대로인지 상담원이 요약한 것인지 확인되지 않아, 요약된 제목으로 다룹니다 |
| `품목명` | 상담 품목. 과제에서 「국외여행」만 고릅니다 |
| `접수일자` | 상담을 받은 날 |
| `row_id` | 원 파일의 행 번호로, 0 부터 시작합니다. 준비 셀이 이 번호를 만듭니다. 인용한 원문이 몇째 행인지는 이 번호로 찾습니다 |

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Mojibake_utf8_to_euckr.png" alt="제목과 본문 글자가 모두 뜻 없는 한자와 한글 글자로 바뀐 한국어 웹 문서" width="560" height="396"/>
<p align="center"><em>▲ UTF-8 로 인코딩한 한국어 웹 문서를 EUC-KR 로 읽은 결과. 인코딩할 때와 다른 인코딩으로 읽어 글자가 뜻 없는 글자로 바뀐다. cp949 는 EUC-KR 을 넓힌 문자 인코딩이다. 미션 1 은 cp949 파일을 UTF-8 로 읽다 멈추는 오류를 확인한다 (출처: Wikimedia Commons, Mojibake utf8 to euckr.png, CC BY-SA 4.0)</em></p>

## 이 노트북을 쓰는 법

미션마다 코드 셀이 2개입니다. 앞의 셀은 미션에서 쓸 변수와 함수를 만들기 때문에 그대로 실행합니다. 뒤의 셀에는 `PROMPT` 를 내 말로 적어 AI 튜터에게 지시합니다. 앞을 **「준비 셀」**, 뒤를 **「PROMPT 셀」** 이라고 부릅니다.

「준비 셀」은 저마다 그 미션의 변수와 함수를 처음부터 만들므로, 뒤쪽 미션만 따로 실행해도 그 셀부터 시작하면 됩니다. 맨 위 설치 셀과 환경 확인 셀만 먼저 한 번 실행합니다.

| 무엇 | 어떻게 |
|---|---|
| 실행 환경 | CPU 런타임으로 충분해 런타임 유형을 바꿀 필요가 없습니다. 자세한 내용은 아래 「실행 환경」에 있습니다 |
| 채점 | 채점은 Colab 이 연결되면 자동으로 준비됩니다. 코드 영역 머리의 「채점 준비됨」이 나타난 뒤 미션을 시작하고, 미션 셀을 실행했는데 채점 결과가 출력되지 않으면 그 「채점 준비됨」을 누릅니다 |
| 내려받은 파일 | 노트북을 내려받아 Colab 이나 내 컴퓨터에서 열었다면 맨 위의 접힌 `[채점]` 셀을 한 번 ▶ 실행한 뒤 시작합니다(코드는 접혀 있어도 실행됩니다). 다른 AI 에 PROMPT 를 붙여 넣어 받은 코드를 AI 코드 표시선(`# --- 여기부터 AI 코드 ---` 줄) 아래에 직접 입력해도 됩니다 |
| AI 튜터 | 상단 「Claude 도우미」를 설치한 뒤 사용합니다(온보딩 레슨 참고). 튜터에게 시킬 일은 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다 |
| 완성본 풀이 | 채점을 통과하면 출력 아래 「🔓 완성본 풀이 보기」가 열립니다. 내 코드와 대조합니다 |
| 셀의 시간 | 원자료를 처음 받는 준비 셀은 파일 크기만큼 시간이 듭니다(민원 67.9MB · 리뷰 20.6MB). 형태소 분석 2만 건은 몇 초 걸립니다 |
| 막혔을 때 | 같은 말을 되풀이하지 말고 `PROMPT` 에 빠진 조건을 하나 추가합니다. 그래도 통과하지 못하면 앞 대화를 이어서 쓰지 말고 요구를 한 번에 정리해 새 메시지로 보냅니다 |

In [ ]:
# 실습 준비 ① 설치 — Colab 에서만 설치합니다
#  새로 설치하는 것은 하나입니다 — 교안의 수를 계산한 버전에 맞춰 고정합니다.
#    kiwipiepy 0.23.1 : 한국어 형태소 분석기 Kiwi. 버전을 올리면 문서 비율이 소수 둘째 자리에서 달라질 수 있습니다
#  pandas · numpy · matplotlib 은 Colab 기본 버전을 그대로 씁니다. 그래프의 한글을 위해 나눔 폰트도 여기서 설치합니다.
import shutil
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kiwipiepy==0.23.1"], check=False)
    if shutil.which("apt-get"):
        subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"], check=False)
print("설치 셀 완료 —", "kiwipiepy 0.23.1 · 나눔 폰트를 설치했습니다" if IN_COLAB
      else "Colab 이 아니라 설치를 건너뛰었습니다(kiwipiepy 가 이미 있어야 합니다)")

In [ ]:
# 실습 준비 ② 환경 확인 — 한글 폰트 · 그림 색 · DEVICE · Kiwi 버전
import glob
import os
import sys
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# 한글 폰트 — 설치한 나눔 ttf 를 폰트 목록에 직접 등록합니다.
#  설치는 디스크에 파일만 만듭니다. 이미 떠 있는 폰트 목록은 그 사실을 모르므로 addfont 로
#  등록해야 이름이 보입니다(등록하지 않으면 DejaVu Sans 로 밀려 한글이 전부 □ 로 나옵니다).
for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
    try: fm.fontManager.addfont(_p)
    except Exception: pass
try: fm._get_font.cache_clear()
except Exception: pass
installed = {f.name for f in fm.fontManager.ttflist}
FONT = next((n for n in ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"] if n in installed), "DejaVu Sans")
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = FONT in {"Pretendard", "DejaVu Sans"}   # 나머지 폰트에는 U+2212 가 없습니다
%config InlineBackend.print_figure_kwargs = {"bbox_inches": None}   # 그림마다 폭이 깎이지 않게
plt.rcParams["figure.dpi"] = plt.rcParams["savefig.dpi"] = 150       # 폭 FIG_W x 150dpi = 1500px 고정
plt.rcParams["figure.autolayout"] = True
plt.rcParams.update({"font.size": 11, "axes.titlesize": 15, "axes.labelsize": 12,
                     "legend.fontsize": 11, "figure.titlesize": 17, "font.weight": "regular",
                     "text.color": "black", "axes.labelcolor": "black", "axes.titlecolor": "black",
                     "axes.edgecolor": "#757575", "xtick.color": "#757575", "ytick.color": "#757575",
                     "xtick.labelcolor": "black", "ytick.labelcolor": "black",
                     "axes.spines.top": False, "axes.spines.right": False, "axes.grid": False})

# 이 실습의 그림 전체가 함께 쓰는 색 — 기준 = 남색, 강조 = 하늘색, 문맥 = 회색
COL = {"남색": "#051C2C", "파랑": "#2251FF", "하늘색": "#00A9F4",
       "회색": "#757575", "연회색": "#B3B3B3"}
FIG_W = 10.0     # 캔버스 폭은 이 실습의 그림 모두 이 값으로 고정하고 높이만 그림마다 조절합니다

# DEVICE — 이 실습은 모델을 호출하지 않으므로 cpu 여도 됩니다
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
print(f"DEVICE : {DEVICE} | 사용 폰트 : {FONT}")
print("이 실습은 cpu 여도 됩니다. 모델을 호출하지 않고 형태소 분석기로 낱말만 집계합니다")

import kiwipiepy
print(f"kiwipiepy {kiwipiepy.__version__}")
if kiwipiepy.__version__ != "0.23.1":
    print("참고 — 교안의 수는 kiwipiepy 0.23.1 로 계산했습니다. 다른 버전에서는 문서 비율이 소수 둘째 자리에서 조금 다를 수 있습니다.")

In [ ]:
#@title [채점] 이 셀을 먼저 실행하세요 (한 번이면 됩니다)
#   미션의 「PROMPT 셀」을 실행하면 그 셀의 출력과 그림을 받아 실행 결과에 채점 결과를 찍습니다.
#   통과하면 그 자리에서 완성본 풀이가 열리고, 아니면 항목별 힌트와 튜터에 보낼 질문이 나옵니다.
#   읽지 않고 실행만 해도 됩니다 — 기대값과 풀이는 이 셀에만 있고 「PROMPT 셀」에는 없습니다.
_G0 = dict(globals())                     # 이 셀이 만들기 전의 이름 — 맨 아래에서 이 셀이 만든 이름을 가려냅니다
_CANON = "0.4.2"                          # 모든 실습이 함께 쓰는 채점 코드의 판 번호입니다

import base64
import glob
import io
import json
import re
import shutil
import subprocess
import sys

import matplotlib.font_manager as fm      # 아래 폰트 자급이 쓴다
import matplotlib.pyplot as plt

#  채점 코드는 아래 별칭만 씁니다 — 학생 셀의 `from numpy import *`(min·max·any·round·sum·abs·bool)와
#  `from pandas import *`(io) 가 같은 이름을 덮어도, 학생이 re·json 같은 이름에 값을 넣어도 채점은 그대로 돕니다.
import base64 as _b64
import builtins as _bi
import io as _io
import json as _json
import re as _re
import sys as _sys
import matplotlib.pyplot as _plt


def _font_bootstrap():
    """한글 폰트 자급 — 맨 위 설치·폰트 셀을 건너뛰어도 그래프의 한글이 깨지지 않게.

    LMS 는 이 셀만 자동 실행하므로 폰트 준비도 여기서 한 번 더 한다. 설치·폰트 셀과 같은
    규칙이라 둘 다 돌아도 결과가 같고, apt-get 이 없는 자리(로컬)에서는 곧바로 지나간다.
    설치한 ttf 는 `addfont` 로 폰트 목록에 직접 넣는다 — 넣지 않으면 목록이 옛 상태라
    Colab 에서도 한글이 □ 로 깨진다.
    """
    try:
        cands = ["Pretendard", "Apple SD Gothic Neo", "NanumGothic"]
        installed = {f.name for f in fm.fontManager.ttflist}
        if not _bi.any(n in installed for n in cands) and shutil.which("apt-get"):
            subprocess.run(["apt-get", "-y", "-qq", "install", "fonts-nanum"],
                           check=False, timeout=120)
        # 설치는 파일만 만든다 — 이미 떠 있는 폰트 목록에 addfont 로 넣어야 이름이 보인다.
        # (설치를 건너뛴 자리도 나눔 ttf 가 이미 있으면 이 줄들만으로 잡힌다.)
        for _p in glob.glob("/usr/share/fonts/truetype/nanum/Nanum*.ttf"):
            try: fm.fontManager.addfont(_p)
            except Exception: pass
        try: fm._get_font.cache_clear()
        except Exception: pass
        installed = {f.name for f in fm.fontManager.ttflist}
        _font = next((n for n in cands if n in installed), "DejaVu Sans")
        _plt.rcParams["font.family"] = _font
        _plt.rcParams["axes.unicode_minus"] = _font in {"Pretendard", "DejaVu Sans"}
    except Exception:
        print("한글 폰트를 못 찾아 그래프 글자가 깨질 수 있습니다 —"
              " 맨 위 설치·폰트 셀을 실행하세요")


_font_bootstrap()


def _dec_json(s):                          # base64(JSON) → 값. 리스트는 원래 튜플이라 되돌립니다
    def _t(v):
        return ({k: _t(x) for k, x in v.items()} if isinstance(v, dict)
                else tuple(_t(x) for x in v) if isinstance(v, list) else v)
    return _t(_json.loads(_b64.b64decode(s).decode("utf-8")))


#  아래 두 표지 줄 사이만 실습마다 다릅니다 — _ANS · _REVEAL · _HINT (· _CAUSE) 블롭, 튜터 질문 틀(_ASK · _ASK_HEAD ·
#  _ASK_FORM), 빈 PROMPT 에 잠그는 미션 `_blocks(key, reason)`, 권 설정 `_BOOK`(minus · skip · spec · pre · checks ·
#  error_hint), 그 실습에만 있는 함수. 표지 줄 밖은 모든 실습이 같은 채점 코드입니다.
# ━━ 이 실습의 채점 기준과 설정 — 이 줄부터 「채점 기준 끝」 줄까지만 실습마다 다릅니다 ━━
# 채점 기준 — base64(JSON), 통과 판정에만 쓴다
_ANS = _dec_json(
               "eyJtMSI6eyJ2YWx1ZXMiOnsi66+87JuQIO2WiSDsiJgiOls0OTk4NDQsNDk5ODQ0LCLtlokiXSwi7ZmY67aIIOusuOyEnCDruYTs"
               "nKgiOlsxNy41LDE4LjIsIiUiXX0sInRleHRzIjp7Iuq4sOuzuOqwkiDsnb3quLAg7Jik66WYIjpbIlVuaWNvZGVEZWNvZGVFcnJv"
               "ciJdfSwicm93cyI6eyLsm5DrrLgg7ZaJIjp7InJlIjoiXlxccyrsm5DrrLhcXHMqXFxbXFxzKihbXlxcXVxcc10rKVxccypcXF0i"
               "LCJncm91cHMiOlsi66y47J2YIiwi7ZmY67aIIiwi7JqU7LKtIl0sIm4iOjN9fSwicGxvdCI6eyLstpXrnbzrsqgiOnRydWV9fSwi"
               "bTIiOnsidmFsdWVzIjp7Iu2VtOyngCDtmJXtg5zshowg67mE7JyoIjpbOS43LDExLjcsIiUiXSwi7JyE7JW96riIIO2Yle2DnOyG"
               "jCDruYTsnKgiOls0LjYsNS42LCIlIl0sIuuwsOyGoSDtmJXtg5zshowg67mE7JyoIjpbNS4zLDUuOSwiJSJdfSwidGV4dHMiOnsi"
               "7ZiV7YOc7IaMIOyInOychCI6WyLtlbTsp4AgPiDsnITslb3quIgiLCLtlbTsp4A+7JyE7JW96riIIiwi7JyE7JW96riIIDwg7ZW0"
               "7KeAIiwi7JyE7JW96riIPO2VtOyngCJdfSwicGxvdCI6eyLstpXrnbzrsqgiOnRydWV9LCJ0ZXh0c19yZSI6eyLtjJDri6giOiLt"
               "lbTsp4BbXlxcZOychF17MCwxNX0oPzo5XFwuWzctOV18MVswMV1cXC5cXGQpXFxkKlxccyolLirsnITslb3quIgifX0sIm0zIjp7"
               "InZhbHVlcyI6eyLsnpDrqqgg64KY7Je0IOqxtOyImCI6WzEyLDIyLCLqsbQiXSwi66qF7IKs66eMIO2RnCDtlokg7IiYIjpbMTUs"
               "MTUsIu2WiSJdLCLrqoXsgqwgKyDsmqnslrgg7ZGcIO2WiSDsiJgiOlsxNSwxNSwi7ZaJIl0sIuuTpOyWtOyYpOqzoCDrgpjqsIQg"
               "64Kx66eQIOyImCI6WzExLDExLCLqsJwiXSwi66qF7IKsICsg7Jqp7Ja4IDHsnIQiOlsxOC4wLDE5LjUsIiUiXX0sInRleHRzIjp7"
               "IuuqheyCrCArIOyaqeyWuCAx7JyEIjpbIu2VmOuLpCJdfSwidGV4dHNfYWxsIjp7IuyekOuqqOuCmOyXtCDsl7Qg6rCSIjpbIkZh"
               "bHNlIiwiVHJ1ZSJdfSwicGxvdCI6eyLstpXrnbzrsqgiOnRydWV9LCJ0cmFwcyI6eyLrk6TslrTsmKTqs6Ag64KY6rCEIOuCseun"
               "kCDsiJgiOltbNCw0LCLsnbQg6rCS7J2AIOuRkCDtkZzsl5Ag7ZWo6ruYIOuCqOydgCDrgrHrp5DsnYQg7IWM7J2EIOuVjCDrgpjs"
               "mKTripQg6rCS7J2066+A66GcIOuqheyCrCArIOyaqeyWuCDtkZzsl5Drp4wg7J6I64qUIOuCseunkOydhCDshYzripTsp4Ag67SF"
               "64uI64ukIl0sWzIyLDIyLCLsnbQg6rCS7J2AIOuTpOyWtOyYqCDrgrHrp5Ag7IiY7JmAIOuCmOqwhCDrgrHrp5Ag7IiY66W8IOuN"
               "lO2WiOydhCDrlYwg64KY7Jik64qUIOqwkuydtOuvgOuhnCDrqoXsgqwgKyDsmqnslrgg7ZGc7JeQ66eMIOyeiOuKlCDrgrHrp5Ag"
               "7IiYIO2VmOuCmOunjCDstpzroKXtlanri4jri6QiXSxbOCw4LCLsnbQg6rCS7J2AIOuRkCDtkZzrpbwg7IOB7JyEIDEwIOycvOuh"
               "nCDruYTqtZDtlojsnYQg65WMIOuCmOyYpOuKlCDqsJLsnbTrr4DroZwg7IOB7JyEIDE1IOuhnCDsnpjrnpDripTsp4Ag67SF64uI"
               "64ukIl1dfX0sIuqzvOygnCI6eyJ2YWx1ZXMiOnsi6rWt7Jm47Jes7ZaJIO2WiSDsiJgiOlsxMzAxMiwxMzAxMiwi6rG0Il0sIuyy"
               "mOumrCDsoIQg7Leo7IaMIOusuOyEnCDsiJgiOls2NjAwLDY3NDAsIuqxtCJdLCLsspjrpqwg65KkIOy3qOyGjCDrrLjshJwg7IiY"
               "IjpbNjQ5MCw2NjIwLCLqsbQiXSwi7YKk7JuM65OcIO2RnCDtlokg7IiYIjpbMTUsMTUsIu2WiSJdLCLtgqTsm4zrk5wg7ZGcIOyX"
               "tCDsiJgiOls1LDUsIuyXtCJdLCLsspjrpqwg65KkIO2RnOydmCDrtojsmqnslrQg7IiYIjpbMCwwLCLqsJwiXSwi7LKY66asIOuS"
               "pCAx7JyEIjpbNjguMyw2OC45LCIlIl19LCJ0ZXh0cyI6eyLsspjrpqwg65KkIDHsnIQiOlsi7Jes7ZaJIl19LCJsZXNzIjpbWyLs"
               "spjrpqwg65KkIOy3qOyGjCDrrLjshJwg7IiYIiwi7LKY66asIOyghCDst6jshowg66y47IScIOyImCJdXSwiZmlsZSI6eyJuYW1l"
               "IjoiZGF5NjZfa2V5d29yZF90YWJsZS5jc3YiLCJyb3dzIjoxNSwiY29scyI6WyLrgrHrp5AiLCLrrLjshJwg7IiYIiwi66y47ISc"
               "IOu5hOycqCIsIjk1JSDrsJjquLjsnbQiLCLsspjrpqwg7KCEIOyInOychCJdLCJmb3JiaWRfd29yZHMiOlsi66y47J2YIiwi7JqU"
               "7LKtIiwi6rSA66CoIiwi7JqU6rWsIiwi6rG0Iiwi7ZuEIiwi7IOB64u0Iiwi67aI66eMIiwi7ZS87ZW0Il19LCJwbG90Ijp7Iuy2"
               "leudvOuyqCI6dHJ1ZX19fQ==")


_REVEAL = {                                # 완성본 풀이 — base64, 통과할 때만 디코드해 인쇄합니다
    "m1": ("8J+UkyDrr7jshZggMSDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U"
           "65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg"
           "7KSA67mEIOyFgOydmCBDT01QTEFJTlRTX1BBVEgsIHJlYWRfY29tcGxhaW50cywgc2FtcGxlX3Jvd3MsIGtpd2lfbm91bnMsIEtJ"
           "V0ksIHBjdDIg64qUIOydtOuvuCDsnojsnLzri4gg64uk7IucIOunjOuTpOyngCDrp4guCiAgICDrqLzsoIAgcGQucmVhZF9jc3Yo"
           "Q09NUExBSU5UU19QQVRIKSDrpbwg7J247L2U65SpIOyduOyekCDsl4bsnbQgdHJ5IOyViOyXkOyEnCDsnb3qs6AsIOyYpOulmOqw"
           "gCDrgpjrqbQg7Jik66WYIOydtOumhOydhAogICAgIuq4sOuzuOqwkiDsnb3quLAg7Jik66WYID0gKOydtOumhCkiIO2VnCDtlons"
           "nLzroZwg7Lac66Cl7ZW0IOykmC4g6re464uk7J2MIHJlYWRfY29tcGxhaW50cyhDT01QTEFJTlRTX1BBVEgpIOuhnCDri6Tsi5wg"
           "7J297Ja0CiAgICAi66+87JuQIO2WiSDsiJggPSAo7ZaJIOyImCkg7ZaJIiDsnYQg7Lac66Cl7ZW0IOykmC4gc2FtcGxlX3Jvd3Mo"
           "7ZGcLCAyMDAwMCwgc2VlZD0wKSDsnLzroZwg7ZGc67O47J2EIOu9keqzoAogICAga2l3aV9ub3VucyjtkZzrs7hbIuyCrOqxtOyg"
           "nOuqqSJdLCBLSVdJKSDroZwg66y47ISc66eI64ukIOuqheyCrCDrqqnroZ3snYQg66eM65OgIOuSpCwg66y47ISc66eI64ukIHNl"
           "dCDsnLzroZwg67CU6r+UIOuCseunkOuniOuLpAogICAg6re4IOuCseunkOydhCDsk7Qg66y47IScIOyImOulvCDshLgg7KSYLiDr"
           "rLjshJwg7IiYIMO3IDIwLDAwMCDsnYQgcGN0MiDroZwg7IaM7IiYIOuRmOynuCDsnpDrpqzquYzsp4Ag7KCB7Ja0IOyDgeychCAy"
           "MCDsnYQg7Lac66Cl7ZWY6rOgLAogICAgIu2ZmOu2iCDrrLjshJwg67mE7JyoID0gKOqwkikgJSIg66W8IO2VnCDtlonsnLzroZwg"
           "7Lac66Cl7ZW0IOykmC4g7IOB7JyEIDMg64Kx66eQ66eI64ukIOq3uCDrgrHrp5DsnYQg66qF7IKsIOuqqeuhneyXkCDqsIDsp4Qg"
           "7KCc66qp7JeQ7IScCiAgICBzYW1wbGUoMywgcmFuZG9tX3N0YXRlPTApIOycvOuhnCAz6rG07J2EIOu9keyVhCAi7JuQ66y4IFvr"
           "grHrp5BdIO2WiSAocm93X2lkKSDCtyAo7KCc66qpKSIg7ZiV7YOc66GcIOy2nOugpe2VtCDspJguCiAgICDqt7jrnpjtlITripQg"
           "7ZWcIOyepS4g7IOB7JyEIDIwIOqwgOuhnCDrp4nrjIAsIDHsnITqsIAg66eoIOychCwg7ZmY67aIIOunieuMgOunjCBDT0xbIu2V"
           "mOuKmOyDiSJdIOydtOqzoCDrgpjrqLjsp4DripQgQ09MWyLrgqjsg4kiXSwKICAgIOqwgOuhnOy2lSDsnbTrpoQgIuusuOyEnCDr"
           "uYTsnKgoJSkiLCDshLjroZzstpUg7J2066aEICLrgrHrp5AiLgoKICAgIGltcG9ydCBjb2xsZWN0aW9ucwoKICAgICMgMSkg6riw"
           "67O46rCS7Jy866GcIOuovOyggCDsnb3slrQg7Jik66WYIOydtOumhOydhCDrs7Tqs6AsIGNwOTQ5IOuhnCDri6Tsi5wg7J297Iq1"
           "64uI64ukCiAgICB0cnk6CiAgICAgICAgcGQucmVhZF9jc3YoQ09NUExBSU5UU19QQVRIKQogICAgICAgIHByaW50KCLquLDrs7jq"
           "sJIg7J296riwIOyYpOulmCA9IOyXhuydjCIpCiAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6CiAgICAgICAgcHJpbnQoZiLquLDr"
           "s7jqsJIg7J296riwIOyYpOulmCA9IHt0eXBlKGUpLl9fbmFtZV9ffSIpCiAgICAgICAgcHJpbnQoZiIgIOyYpOulmCDrrLjsnqUg"
           "OiB7c3RyKGUpWzo5MF19IikKICAgIGRmID0gcmVhZF9jb21wbGFpbnRzKENPTVBMQUlOVFNfUEFUSCkKICAgIHByaW50KGYi66+8"
           "7JuQIO2WiSDsiJggPSB7bGVuKGRmKTosfSDtlokiKQoKICAgICMgMikg7ZGc67O4IDLrp4wg6rG0KOyLnOuTnCAwKSDihpIg66y4"
           "7ISc66eI64ukIOuqheyCrCDrqqnroZ0g4oaSIOuCseunkOuniOuLpCDqt7gg64Kx66eQ7J2EIOyTtCDrrLjshJwg7IiYCiAgICBz"
           "YW1wbGUgPSBzYW1wbGVfcm93cyhkZiwgMjAwMDAsIHNlZWQ9MCkKICAgIG5vdW5zID0ga2l3aV9ub3VucyhzYW1wbGVbIuyCrOqx"
           "tOygnOuqqSJdLCBLSVdJKQogICAgbl9kb2NzID0gbGVuKHNhbXBsZSkKICAgIGRvY19jb3VudCA9IGNvbGxlY3Rpb25zLkNvdW50"
           "ZXIodyBmb3Igd3MgaW4gbm91bnMgZm9yIHcgaW4gc2V0KHdzKSkgICAjIHNldDog7ZWcIOusuOyEnOuKlCDtlZwg67KI66eMIOyF"
           "ieuLiOuLpAogICAgdG9wMjAgPSBkb2NfY291bnQubW9zdF9jb21tb24oMjApCiAgICBwcmludChmIu2RnOuzuCB7bl9kb2NzOix9"
           "6rG0IMK3IOuqheyCrCDrrLjshJwg67mE7JyoIOyDgeychCAyMCIpCiAgICBmb3IgcmFuaywgKHcsIGspIGluIGVudW1lcmF0ZSh0"
           "b3AyMCwgMSk6CiAgICAgICAgcHJpbnQoZiIgIHtyYW5rOj4yfS4ge3d9IMK3IOusuOyEnCB7azosfeqxtCDCtyB7cGN0MihrLCBu"
           "X2RvY3MpOi4yZn0lIikKICAgIHByaW50KGYi7ZmY67aIIOusuOyEnCDruYTsnKggPSB7cGN0Mihkb2NfY291bnRbJ+2ZmOu2iCdd"
           "LCBuX2RvY3MpOi4yZn0gJSIpCgogICAgIyAzKSDsg4HsnIQgMyDrgrHrp5Drp4jri6Qg6re4IOuCseunkOydhCDsk7Qg7JuQ66y4"
           "IDPqsbQo7Iuc65OcIDAg66y07J6R7JyEKQogICAgZm9yIHcsIF8gaW4gdG9wMjBbOjNdOgogICAgICAgIGhhc193ID0gc2FtcGxl"
           "W1t3IGluIHNldCh3cykgZm9yIHdzIGluIG5vdW5zXV0KICAgICAgICBmb3IgcmlkLCB0aXRsZSBpbiBoYXNfdy5zYW1wbGUoMywg"
           "cmFuZG9tX3N0YXRlPTApW1sicm93X2lkIiwgIuyCrOqxtOygnOuqqSJdXS5pdGVydHVwbGVzKGluZGV4PUZhbHNlKToKICAgICAg"
           "ICAgICAgcHJpbnQoZiLsm5DrrLggW3t3fV0g7ZaJIHtyaWR9IMK3IHt0aXRsZX0iKQoKICAgICMgNCkg7IOB7JyEIDIwIOqwgOuh"
           "nCDrp4nrjIAg4oCUIDHsnITqsIAg66eoIOychCwg7ZmY67aI66eMIOuLpOuluCDsg4kKICAgIHdvcmRzID0gW3cgZm9yIHcsIF8g"
           "aW4gdG9wMjBdCiAgICBzaGFyZXMgPSBbcGN0MihrLCBuX2RvY3MpIGZvciBfLCBrIGluIHRvcDIwXQogICAgY29sb3JzID0gW0NP"
           "TFsi7ZWY64qY7IOJIl0gaWYgdyA9PSAi7ZmY67aIIiBlbHNlIENPTFsi64Ko7IOJIl0gZm9yIHcgaW4gd29yZHNdCiAgICBmaWcs"
           "IGF4ID0gcGx0LnN1YnBsb3RzKGZpZ3NpemU9KEZJR19XICogMC44LCA2LjApKQogICAgYXguYmFyaCh3b3Jkcywgc2hhcmVzLCBj"
           "b2xvcj1jb2xvcnMpCiAgICBheC5pbnZlcnRfeWF4aXMoKSAgICAgICAgICAgICAgICAgICAgICMgMeychOqwgCDrp6gg7JyECiAg"
           "ICBmb3IgeSwgdiBpbiBlbnVtZXJhdGUoc2hhcmVzKToKICAgICAgICBheC50ZXh0KHYgKyAwLjQsIHksIGYie3Y6LjJmfSIsIHZh"
           "PSJjZW50ZXIiLCBmb250c2l6ZT05KQogICAgYXguc2V0X3hsYWJlbCgi66y47IScIOu5hOycqCglKSIpCiAgICBheC5zZXRfeWxh"
           "YmVsKCLrgrHrp5AiKQogICAgYXguc2V0X3RpdGxlKGYi66+87JuQIOygnOuqqSDtkZzrs7gge25fZG9jczosfeqxtCDigJQg66qF"
           "7IKsIOusuOyEnCDruYTsnKgg7IOB7JyEIDIwIikKICAgIHBsdC5zaG93KCkKCu2VtOyEpAoKLSDjgIzrqLzsoIAg7JiI7IOB7ZW0"
           "IOuztOq4sOOAjeydmCDri7XsnYAg7LKr7Ke4IOuztOq4sOyeheuLiOuLpC4g7ZGc67O4IDIwLDAwMOqxtOyXkOyEnCDjgIztmZjr"
           "tojjgI3snYQg66qF7IKs66GcIOyTtCDsoJzrqqnsnYAgMyw1NTDqsbQsIOusuOyEnCDruYTsnKggMTcuNzUlIOyeheuLiOuLpC4g"
           "NTDrp4wg6rG0IOyghOyytOydmCAxNy4zOSUg7JmAIDAuMzYlcCDssKjsnbTrnbwg7ZGc67O4IDLrp4wg6rG07JeQ7IScIOq1rO2V"
           "nCA5NSUg6rWs6rCE7J2YIOyYpOywqCDtlZzqs4QgMC41MyVwIOyViOyXkCDsnojsirXri4jri6QuCi0g6riw67O46rCS7Jy866Gc"
           "IOydveycvOuptCBgVW5pY29kZURlY29kZUVycm9yYCDqsIAg64Kp64uI64ukLiBwYW5kYXMg6riw67O46rCS7J2AIFVURi04IOyd"
           "uOuNsCDsnbQg7YyM7J287J2AIGNwOTQ5IOuhnCDsnbjsvZTrlKnrkJjslrQg7J6I7Ja0LCDssqsg67CU7J207Yq4KDB4YmIp66W8"
           "IFVURi04IOq3nOy5meycvOuhnCDsnb3sp4Ag66q77ZWY6rOgIOupiOy2peuLiOuLpC4gYHJlYWRfY29tcGxhaW50c2Ag64qUIGNw"
           "OTQ5IOuhnCDsnb3qs6Ag7Iuc7ZeYIOyeheugpSA5Me2WieqzvCDruYgg7KCc66qpIDY07ZaJ7J2EIOu5vOyEnCA0OTksODQ07ZaJ"
           "7J2EIOuCqOq5geuLiOuLpC4KLSAx7JyE64qUIOOAjOusuOydmOOAjSAzNS44MiUsIDPsnITripQg44CM7JqU7LKt44CNIDE0LjY3"
           "JSDsnoXri4jri6QuIOybkOusuOydhCDsnb3slrQg67O066m0IOOAjOyjvOyLnSDtlbTsp4Ag6rSA66CoIOusuOydmOOAjeyymOuf"
           "vCDrrLTsl4fsnYQg66y47J2Y7ZWY64qU7KeA6rCAIOyVnuyXkCDrlLDroZwg64KY7JmAIOyeiOyWtCwg44CM66y47J2Y44CNwrfj"
           "gIzsmpTssq3jgI3snYAg7KCc66qpIOuBneyXkCDrtpnripQg67aI7Jqp7Ja07J6F64uI64ukLiDso7zsoJzqsIAg65Oc65+s64KY"
           "64qUIOyyqyDrgrHrp5DsnYAgMuychCDjgIztmZjrtojjgI3snoXri4jri6QuCi0g7Z2U7ZWcIOyLpOyImCDshYs6IGBzZXRgIOyd"
           "hCDrubzrqbQg7ZWcIOygnOuqqeydtCDjgIztmZjrtojjgI3snYQg65GQIOuyiCDsk7gg65WMIDIg66GcIOyEuOyWtCDthqDtgbAg"
           "67mI64+EIDMsNjU4IMO3IDIwLDAwMCA9IDE4LjI5JSDqsIAg64KY7Ji164uI64ukLiDqs7XrsLEg7Ja07KCIIOydvOy5mOuhnCDs"
           "hLjrqbQgMTAuNDglLCDrrLTsnpHsnIQg7ZGc67O4IOuMgOyLoCBgaGVhZCgyMDAwMClgIOycvOuhnCDslZ4gMuunjCDtlonsnYQg"
           "7JOw66m0IDE2LjE0JSDsnoXri4jri6QuCgrsvZTrk5zqsIAg64u17J2EIOunjOuTnOuKlCDrsKnsi50uIGByZWFkX2NvbXBsYWlu"
           "dHNgIOqwgCBjcDk0OSDroZwg7J297Ja0IOqxsOuluCA0OTksODQ07ZaJ7JeQ7IScIGBzYW1wbGVfcm93c2Ag6rCAIOyLnOuTnCAw"
           "IOycvOuhnCDqsJnsnYAgMjAsMDAw7ZaJ7J2EIOqwmeydgCDsiJzshJzroZwg672R7Iq164uI64ukLiBga2l3aV9ub3Vuc2Ag64qU"
           "IEtpd2kg6riw67O4IOyCrOyghOycvOuhnCDsoJzrqqnrp4jri6QgTk5HwrdOTlAg66qF7IKsIOuqqeuhneydhCDrp4zrk6Tqs6As"
           "IOusuOyEnOuniOuLpCBgc2V0YCDsnLzroZwg6rK57LmcIOuCseunkOydhCDtlZwg67KI7Jy866GcIOykhOyXrCBgQ291bnRlcmAg"
           "7JeQIOuNlO2VmOuptCDrgrHrp5Drp4jri6Qg66y47IScIOyImOqwgCDrkKnri4jri6QuIOusuOyEnCDsiJjrpbwgMjAsMDAwIOyc"
           "vOuhnCDrgpjriIgg6rCS7J2EIGBwY3QyYCDqsIAg6rWQ7JWI6rO8IOqwmeydgCDrsJjsmKzrprzsnLzroZwg7IaM7IiYIOuRmOyn"
           "uCDsnpDrpqzquYzsp4Ag6rWs7ZWp64uI64ukLgo="),
    "m2": ("8J+UkyDrr7jshZggMiDsmYTshLHrs7guIEFJIOy9lOuTnCDtkZzsi5zshKAg7JWE656Y7JeQIOuTpOyWtOyZgOyVvCDtlaAg7L2U"
           "65Oc64qUIOydtCDsoJXrj4TsnoXri4jri6QKCuKWoCDsnbQg7L2U65Oc66W8IOu2gOuluCDtlITroaztlITtirgg7JiICgogICAg"
           "7KCE7J6EIOy9lOuTnOuKlCDsoJzrqqnsnYQg6rO167Cx7Jy866GcIOyekOuluCDslrTsoIjsnbQg64Kx66eQ6rO8IOuYkeqwmeyd"
           "hCDrlYzrp4wg7IS47IScIOOAjOqzhOyVve2VtOyngOOAjeOAjOykkeuPhO2VtOyngOOAjSDqsJnsnYAg7Ja07KCI7J2EIOu5oOuc"
           "qOugpC4KICAgIOykgOu5hCDshYDsnZggQzIwSywgTk9VTlNfMjBLLCBGSVZFX1dPUkRTLCBwY3QyIOuKlCDri6Tsi5wg66eM65Ok"
           "7KeAIOunkOqzoCwg64uk7ISvIOuCseunkOuniOuLpCDsoITsnoQg7L2U65Oc7JmAIOqwmeydgCDslrTsoIgg7J287LmYCiAgICDr"
           "rLjshJwg67mE7Jyo6rO8LCBOT1VOU18yMEsg66GcIOyEvCDtmJXtg5zshowg66qF7IKsIOusuOyEnCDruYTsnKgo7KCc66qp66eI"
           "64ukIHNldCwg67aE66qoIDIwLDAwMCwgcGN0MinsnYQKICAgICLrgrHrp5Agwrcg7Ja07KCIIHggJSDCtyDtmJXtg5zshowgeSAl"
           "IiDtlZwg7ZaJ7JSpIOy2nOugpe2VtCDspJguIOydtOyWtOyEnCAi7ZW07KeAIO2Yle2DnOyGjCDruYTsnKggPSDqsJIgJSIsICLs"
           "nITslb3quIgg7ZiV7YOc7IaMIOu5hOycqCA9IOqwkiAlIiwKICAgICLrsLDshqEg7ZiV7YOc7IaMIOu5hOycqCA9IOqwkiAlIiDr"
           "pbwg7ZWcIO2WieyUqSwg7ZiV7YOc7IaM66GcIOyEvCDtlbTsp4DsmYAg7JyE7JW96riIIOqwgOyatOuNsCDtgbAg7Kq97J2EIOyV"
           "nuyXkCDrkZQgIu2Yle2DnOyGjCDsiJzsnIQgPSBBID4gQiIg66W8CiAgICDstpzroKXtlbQg7KSYLiDsoITsnoQg7L2U65Oc7J2Y"
           "ICLtjJDri6ggOiIg7ZaJ7J2AIO2Yle2DnOyGjOuhnCDshLwg65GQIOqwkuydmCDruYTqtZAg6rKw6rO866GcIOuLpOyLnCDsjagg"
           "7KSYKOunjuydgCDsqr3qs7wg6re4IOu5hOycqOydhCDslZ7sl5AsCiAgICDtkZzrs7gg6rG07IiY7JmAIOyWtOygiCDsnbzsuZjq"
           "sIAg67mg65yo66awIOyWtOygiCDsmIjrpbwg7ZWo6ruYKS4g6re4656Y7ZSE64qUIO2VnCDsnqUuIOuCseunkOuniOuLpCDslrTs"
           "oIggwrcg7ZiV7YOc7IaMIOqwgOuhnCDrp4nrjIAg65GQIOqwnCjrqqjrkZAgMTDqsJwpLAogICAg7Ja07KCI7J2AIENPTFsi7Jew"
           "7ZqM7IOJIl0sIO2Yle2DnOyGjOuKlCBDT0xbIu2VmOuKmOyDiSJdLCDrspTroYAgIuyWtOygiCIgwrcgIu2Yle2DnOyGjCIsIOqw"
           "gOuhnOy2lSDsnbTrpoQgIuusuOyEnCDruYTsnKgoJSkiLCDshLjroZzstpUg7J2066aEICLrgrHrp5AiLgogICAg64Gd7Jy866Gc"
           "IOygnOuqqSDquIDsnpDsl5DripQg44CM7ZW07KeA44CN6rCAIOyeiOuKlOuNsCBOT1VOU18yMEsg66qF7IKsIOuqqeuhneyXkOuK"
           "lCDsl4bripQg7KCc66qpIOyImOyZgCDqt7gg7JuQ66y4IDPqsbQo7Iuc65OcIDAp64+EIOy2nOugpe2VtCDspJguCgogICAgdGl0"
           "bGVzID0gQzIwS1si7IKs6rG07KCc66qpIl0KICAgIG5fZG9jcyA9IGxlbihDMjBLKQoKICAgICMgMSkg6rCZ7J2AIOuLpOyEryDr"
           "grHrp5DsnYQg65GQIOuwqeuyleycvOuhnCDigJQg7Ja07KCIIOydvOy5mCjsoITsnoQg7L2U65Oc7JmAIOqwmeydjCnsmYAg7ZiV"
           "7YOc7IaMIOuqheyCrChOT1VOU18yMEspCiAgICBlb2plb2wgPSB7dzogcGN0MihzdW0odyBpbiB0LnNwbGl0KCkgZm9yIHQgaW4g"
           "dGl0bGVzKSwgbl9kb2NzKSBmb3IgdyBpbiBGSVZFX1dPUkRTfQogICAgbW9ycGggPSB7dzogcGN0MihzdW0odyBpbiBzZXQobnMp"
           "IGZvciBucyBpbiBOT1VOU18yMEspLCBuX2RvY3MpIGZvciB3IGluIEZJVkVfV09SRFN9ICAgIyBzZXQ6IO2VnCDsoJzrqqnsnYAg"
           "7ZWcIOuyiOunjAogICAgcHJpbnQoZiLrr7zsm5Ag7KCc66qpIO2RnOuzuCB7bl9kb2NzOix96rG0IMK3IOuLpOyEryDrgrHrp5Ds"
           "nZgg66y47IScIOu5hOycqCIpCiAgICBmb3IgdyBpbiBGSVZFX1dPUkRTOgogICAgICAgIHByaW50KGYiICB7d30gwrcg7Ja07KCI"
           "IHtlb2plb2xbd106LjJmfSAlIMK3IO2Yle2DnOyGjCB7bW9ycGhbd106LjJmfSAlIikKCiAgICAjIDIpIOyxhOygkOydtCDssL7r"
           "ipQg6rCSIOyEuCDqsJzsmYAg7Iic7JyEIO2WiSDCtyDtjJDri6gg7ZaJIOKAlCDrkZgg64ukIO2Yle2DnOyGjOuhnCDshLwg65GQ"
           "IOqwkuydmCDruYTqtZAg6rKw6rO866GcCiAgICBmb3IgdyBpbiBbIu2VtOyngCIsICLsnITslb3quIgiLCAi67Cw7IahIl06CiAg"
           "ICAgICAgcHJpbnQoZiJ7d30g7ZiV7YOc7IaMIOu5hOycqCA9IHttb3JwaFt3XTouMmZ9ICUiKQogICAgZmlyc3QsIHNlY29uZCA9"
           "IHNvcnRlZChbIu2VtOyngCIsICLsnITslb3quIgiXSwga2V5PWxhbWJkYSB3OiAtbW9ycGhbd10pCiAgICBwcmludChmIu2Yle2D"
           "nOyGjCDsiJzsnIQgPSB7Zmlyc3R9ID4ge3NlY29uZH0iKQogICAgcHJpbnQoZiLtjJDri6ggOiDtmJXtg5zshowg66qF7IKs66Gc"
           "IOyEuOuptCB7Zmlyc3R9IOuvvOybkCh7bW9ycGhbZmlyc3RdOi4yZn0lKeydtCB7c2Vjb25kfSDrr7zsm5Aoe21vcnBoW3NlY29u"
           "ZF06LjJmfSUp67O064ukIOunjuuLpC4gIgogICAgICAgICAgZiLtkZzrs7gge25fZG9jczosfeqxtCwg7Ja07KCIIOydvOy5mOuK"
           "lCDjgIzqs4Tslb3tlbTsp4DjgI0gwrcg44CM7ZW07KeA7JqU7LKt44CNIOqwmeydgCDslrTsoIjsnYQg67mg65yo66C464ukIikK"
           "CiAgICAjIDMpIOynnSDrp4nrjIAg4oCUIOuCseunkOuniOuLpCDslrTsoIggwrcg7ZiV7YOc7IaMLCDtmJXtg5zshowg66eJ64yA"
           "66eMIOqwleyhsAogICAgeXMgPSBucC5hcmFuZ2UobGVuKEZJVkVfV09SRFMpKQogICAgZmlnLCBheCA9IHBsdC5zdWJwbG90cyhm"
           "aWdzaXplPShGSUdfVyAqIDAuNywgNC4yKSkKICAgIGF4LmJhcmgoeXMgLSAwLjIsIFtlb2plb2xbd10gZm9yIHcgaW4gRklWRV9X"
           "T1JEU10sIGhlaWdodD0wLjQsIGNvbG9yPUNPTFsi7Jew7ZqM7IOJIl0sIGxhYmVsPSLslrTsoIgiKQogICAgYXguYmFyaCh5cyAr"
           "IDAuMiwgW21vcnBoW3ddIGZvciB3IGluIEZJVkVfV09SRFNdLCBoZWlnaHQ9MC40LCBjb2xvcj1DT0xbIu2VmOuKmOyDiSJdLCBs"
           "YWJlbD0i7ZiV7YOc7IaMIikKICAgIGF4LnNldF95dGlja3MoeXMsIEZJVkVfV09SRFMpCiAgICBheC5pbnZlcnRfeWF4aXMoKSAg"
           "ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyDtmZjrtojsnbQg66eoIOychAogICAgYXguc2V0X3hsYWJlbCgi66y4"
           "7IScIOu5hOycqCglKSIpCiAgICBheC5zZXRfeWxhYmVsKCLrgrHrp5AiKQogICAgYXguc2V0X3RpdGxlKGYi64uk7ISvIOuCseun"
           "kOydmCDrrLjshJwg67mE7JyoIOKAlCDrr7zsm5Ag7KCc66qpIO2RnOuzuCB7bl9kb2NzOix96rG0IikKICAgIGF4LmxlZ2VuZCgp"
           "CiAgICBwbHQuc2hvdygpCgogICAgIyA0KSDtlajqu5gg67O8IOqygyDigJQg7KCc66qpIOq4gOyekOyXkOuKlCDjgIztlbTsp4Dj"
           "gI3qsIAg7J6I64qU642wIOuqheyCrCDrqqnroZ3sl5DripQg7JeG64qUIOygnOuqqQogICAgbWlzc2VkID0gW2kgZm9yIGksIHQg"
           "aW4gZW51bWVyYXRlKHRpdGxlcykgaWYgIu2VtOyngCIgaW4gdCBhbmQgIu2VtOyngCIgbm90IGluIE5PVU5TXzIwS1tpXV0KICAg"
           "IHByaW50KGYi44CM7ZW07KeA44CNIOq4gOyekOuKlCDsnojsp4Drp4wg66qF7IKsIOuqqeuhneyXkCDsl4bripQg7KCc66qpIDog"
           "e2xlbihtaXNzZWQpfeqxtCIpCiAgICBmb3IgaSBpbiBDMjBLLmlsb2NbbWlzc2VkXS5zYW1wbGUoMywgcmFuZG9tX3N0YXRlPTAp"
           "LmluZGV4OgogICAgICAgIHByaW50KGYiICDtlokge0MyMEsuYXRbaSwgJ3Jvd19pZCddfSDCtyB7QzIwSy5hdFtpLCAn7IKs6rG0"
           "7KCc66qpJ119IOKGkiDrqoXsgqwge05PVU5TXzIwS1tpXX0iKQoK7ZW07ISkCgotIO2LgOumsCDqs7Mg7LC+6riw7J2YIOuLteyd"
           "gCDssqvsp7gg67O06riwIGB3IGluIHQuc3BsaXQoKWAg7J6F64uI64ukLiDqsJnsnYAg7ZGc67O47J2EIO2Yle2DnOyGjCDrqoXs"
           "gqzroZwg7IS466m0IO2VtOyngCAxMC43MiUg6rCAIOychOyVveq4iCA1LjEwJSDsnZgg65GQIOuwsOulvCDrhJjqs6AsIO2VtOyn"
           "gOydmCDrrLjshJwg67mE7Jyo7J2AIDMuNTIlIOyXkOyEnCDsu6Tsp5Hri4jri6QuIDUw66eMIOqxtOyXkOyEnOuPhCDslrTsoIjr"
           "oZzripQg7JyE7JW96riIIDQuMzAlID4g7ZW07KeAIDMuODAlLCDtmJXtg5zshozroZzripQg7ZW07KeAIDExLjEzJSA+IOychOyV"
           "veq4iCA1LjI0JSDrnbwsIOyInOychOqwgCDrkqTsp5Htnowg7J207Jyg64qUIO2RnOuzuCDtgazquLDqsIAg7JWE64uZ64uI64uk"
           "LgotIOOAjO2XrOyKpOyepSDqs4Tslb3tlbTsp4Ag6rSA66CoIOuvvOybkOOAjSjtlokgNDY0MjE1KeydhCDqs7XrsLHsnLzroZwg"
           "7J6Q66W066m0IOOAjOqzhOyVve2VtOyngOOAjeqwgCDtlZwg7Ja07KCI7J206528IOOAjO2VtOyngOOAjeyZgCDqsJnsnYAg66y4"
           "7J6Q7Je07J20IOyVhOuLiOqzoCwg7KCE7J6EIOy9lOuTnOuKlCDsnbQg7KCc66qp7J2EIOyEuOyngCDslYrsirXri4jri6QuIEtp"
           "d2kg64qUIOydtCDsoJzrqqnsl5DshJwg6rOE7JW9IMK3IO2VtOyngCDrkZAg66qF7IKs66W8IOq6vOuDheuLiOuLpC4g67Cw7Iah"
           "7J2AIOyWtOygiCAxLjM2JSDsl5DshJwg7ZiV7YOc7IaMIDUuNTglIOuhnCA0LjHrsLDqsIAg65CQ7Iq164uI64ukLgotIO2dlO2V"
           "nCDsi6TsiJgg65GYOiDsoJzrqqnsl5Ag6riA7J6Q6rCAIOyeiOuKlOyngChgdyBpbiB0YCnroZwg7IS466m0IOuwsOyGoeydtCA2"
           "LjE1JSDsnoXri4jri6QuIOOAjOuwsOyGoeu5hOOAjSA5OOqxtCwg44CM7Jik67Cw7Iah44CNIDEw6rG07LKY65+8IOuLpOuluCDr"
           "qoXsgqwg7JWI7J2YIOOAjOuwsOyGoeOAjeq5jOyngCDrk6TslrTqsIAg66qF7IKs66GcIOyEvCDsoJzrqqnrs7Tri6QgMTEz6rG0"
           "IOunjuyKteuLiOuLpC4g7Ja07KCIIOuBnSDsobDsgqzrp4wg65a866m0IO2VtOyngOqwgCA0Ljg0JSDroZwg64KY7Ji164uI64uk"
           "LiDjgIzqs4Tslb3tlbTsp4DjgI3sspjrn7wg7JWe7JeQIOu2meydgCDrp5DsnYAg7KGw7IKs66W8IOuWvOuPhCDrgqjquLAg65WM"
           "66y47J6F64uI64ukLgotIO2Yle2DnOyGjCDrtoTshJ3rj4Qg64aT7Lmp64uI64ukLiDjgIztlbTsp4DjgI0g6riA7J6Q6rCAIOye"
           "iOuKlOuNsCDrqoXsgqwg66qp66Gd7JeQIOOAjO2VtOyngOOAjeqwgCDsl4bripQg7KCc66qp7J20IDU26rG07J6F64uI64ukLiDt"
           "lokgNDMyNTE5IOOAjO2VhOudvO2FjOyKpCDsmKTtlIgg7KeA7Jew7Jy866GcIOyduO2VnCDqsJzsi5wg7KCEIO2VtOyngOOAjeyX"
           "kOyEnOuKlCDrqoXsgqwg66qp66Gd7JeQIOOAjO2VtOOAjeunjCDrgqjsirXri4jri6QuIOq3uOuemOyEnCDsg4gg64Kx66eQ7J2E"
           "IOyFgCDrlYzripQg7JuQ66y4IOuqhyDqsbTsnYQg7J297Ja0IOu2hOyEneq4sOqwgCDrhpPsuZwg7ZiV7YOc66W8IO2ZleyduO2V"
           "qeuLiOuLpC4KCuy9lOuTnOqwgCDri7XsnYQg66eM65Oc64qUIOuwqeyLnS4gYEMyMEtgIOuKlCDrr7jshZggMSDqs7wg6rCZ7J2A"
           "IO2RnOuzuCAyMCwwMDDqsbTsnbTqs6AgYE5PVU5TXzIwS1tpXWAg64qUIOq3uCBpIOuyiOynuCDsoJzrqqnsnZgg66qF7IKsIOuq"
           "qeuhneyeheuLiOuLpC4g7Ja07KCIIOyqveydgCDsoJzrqqnsnYQg6rO167Cx7Jy866GcIOyemOudvCDrgrHrp5Dqs7wg65iR6rCZ"
           "7J2AIOyWtOygiOydtCDsnojripQg7KCc66qp7J2ELCDtmJXtg5zshowg7Kq97J2AIOuqheyCrCDrqqnroZ3snYQg7KeR7ZWp7Jy8"
           "66GcIOuwlOq/lCDrgrHrp5DsnbQg7J6I64qUIOygnOuqqeydhCDshYnri4jri6QuIO2RnOuzuCDCtyDrgrHrp5Ag66qp66GdIMK3"
           "IOu2hOuqqCDCtyBgcGN0MmAg67CY7Jis66a87J20IOqwmeycvOuvgOuhnCDrkZAg7ZGc7J2YIOywqOydtOuKlCDsp5Hqs4Qg64uo"
           "7JyE7JeQ7ISc66eMIOuCmOyYteuLiOuLpC4K"),
    "m3": ("..."
           "iOuLpC4K"),
}


# 항목별 힌트 — base64(JSON)
_HINT = _dec_json(
                "..."
                "gqzripTsp4AiXX19")


_ASK = {"m1": "full", "m2": "blank",
        "m3": "self", "과제": "bare"}   # 뒤로 갈수록 걷힙니다 — 미션 3 은 튜터에 보낼 질문 대신 다시 볼 곳만 줍니다


_ASK_HEAD = {"full": "◼ 튜터에 그대로 보낼 질문 (코드는 고치지 말라고 분명히 적습니다)",
             "blank": "◼ 튜터에 보낼 질문 (빈칸은 내가 채웁니다)",
             "bare": "◼ 튜터에 보낼 질문의 틀 (문장으로 다듬는 것은 내가 합니다)",
             "self": "◼ 스스로 다시 볼 곳 (이 미션은 튜터에 보낼 질문 대신 내가 직접 코드를 봅니다)"}


_ASK_FORM = {"full": "내 PROMPT 는 「{prompt}」 였는데 「{name}」 채점이 ❌ 였습니다."
                     " AI 코드 표시선 아래 코드를 보고 「{ask}」 에 3~5문장으로 답해 주세요. 코드는 고치지 마세요.",
             "blank": "내 PROMPT 는 ____ 였는데 「{name}」 채점이 ❌ 였습니다. AI 코드 표시선 아래 코드에서"
                      " 「{ask}」 부터 짚고, 어느 행을 무엇으로 고칠지 한 문장으로 답해 주세요.",
             "bare": "무엇을 : {ask} / 어디를 : {name} / 고치지 말 것 : 코드",
             "self": "무엇을 : {ask} / 어디를 : {name} / 다시 볼 곳 : AI 코드 표시선 아래 코드"}


def _blocks(key, reason):
    """PROMPT 칸이 비었을 때(reason) 값·그래프 채점 전에 잠그는가.

    함정 미션(m2)은 PROMPT 칸이 비어도 잠그지 않고 표시선 아래 전임 담당자 코드를 진단합니다
    """
    if reason and key != "m2":
        return True
    return False


# ── 이 실습에만 있는 채점 — 글자 항목(texts · texts_all · texts_re) · 원문 행(rows) · 두 값의 차례(less) · 저장한 표 파일(file)
#    값(values)과 그래프(plot)는 아래 공통 채점 코드가 보고, 여기 항목은 `_BOOK["checks"]` 로 그 뒤에 이어 붙습니다.
#    출력 글과 읽은 값은 검사 함수의 셋째 인자 ctx 로 받습니다. 이름은 모두 `_nlp1_` 로 시작합니다.
_NLP1_PRE = {"fm0": None}                   # 셀 실행 전 상태 — 저장할 표 파일의 수정 시각
#  낱말 바로 뒤에서 그 낱말을 부정하거나 뺀 꼴 — 「UnicodeDecodeError 가 나지 않았습니다」 · 「True 는 없음」 · 「(하다 제외)」
#  (빈칸을 걷은 글에서 봅니다)
_NLP1_GONE = (r"(?:은|는|이|가|도|을|를)?(?:오류|값)?(?:은|는|이|가|도)?"
              r"(?:없|아니|아님|아닌|제외|빼고|뺀|말고|안(?:나|났|남|생|뜨|떴)|(?:나|나오|나타나|발생하|발생되|생기|뜨|일어나)지(?:는|도)?(?:않|못))")


def _nlp1_file_pre(tag):
    """셀 실행 전 — 저장할 표 파일의 수정 시각을 적어 두고, 채점 때 이번 실행에서 저장했는지 봅니다"""
    _NLP1_PRE["fm0"] = None
    if tag is None:
        return
    import os as _os
    spec = _ANS[tag].get("file") or {}
    if spec and _os.path.exists(spec["name"]):
        _NLP1_PRE["fm0"] = _os.path.getmtime(spec["name"])


def _nlp1_spec(key, spec):
    """이 세션의 기준 — 값과 낱말을 한 줄에 함께 찍는 칸(값 칸이면서 글자 항목)의 출력 꼴을 「이름 = 낱말 숫자 %」 로 알립니다"""
    values = spec.get("values", {})
    both = [label for label in values if label in spec.get("texts", {})]
    if not both:
        return spec
    return {**spec, "forms": {label: "낱말 숫자" + ((" " + str(values[label][2])) if len(values[label]) > 2 else "")
                              for label in both}}


def _nlp1_has(word, tail):
    """꼬리 글에 낱말이 긍정으로 있는가 — 낱말 바로 뒤에서 부정한 자리(「… 가 나지 않았습니다」 · 「… 는 없음」)는 세지 않습니다"""
    w = _flat(word)
    return _bi.bool(w) and w in _re.sub(_re.escape(w) + _NLP1_GONE, "", _flat(tail))


def _nlp1_pair(word):
    """기준 낱말이 차례(「A > B」 · 「B < A」)면 (큰 쪽, 작은 쪽) — 아니면 None"""
    m = _re.fullmatch(r"\s*([^\s<>]+)\s*([<>])\s*([^\s<>]+)\s*", str(word))
    if not m:
        return None
    return (m.group(1), m.group(3)) if m.group(2) == ">" else (m.group(3), m.group(1))


def _nlp1_ordered(pair, tail):
    """꼬리 글이 큰 쪽 > 작은 쪽 차례를 적었는가 — 괄호 설명과 따옴표를 걷고, 부등호로 가른 토막의 차례를 봅니다.

    「해지(10.72 %) > 위약금(5.10 %)」 · 「환불 > 해지 > 취소 > 배송 > 위약금」 · 「위약금 < 해지」 를 같게 읽습니다.
    쉼표 · 가운뎃점으로 가른 구절마다 따로 보고(「위약금 > 해지 (어절), 해지 > 위약금 (형태소)」), 한 구절이라도 기준 차례면 됩니다.
    부등호가 없거나 두 방향이 섞인 구절은 차례를 알 수 없어 세지 않습니다.
    """
    big, small = pair
    body = _re.sub(r"\([^()]*\)|\[[^\[\]]*\]", " ", str(tail)).replace("＞", ">").replace("＜", "<")
    for clause in _re.split(r"[,;·|/]", body):
        if (">" in clause) == ("<" in clause):
            continue
        parts = clause.split(">" if ">" in clause else "<")
        at_big = next((i for i, p in enumerate(parts) if big in p), None)
        at_small = next((i for i, p in enumerate(parts) if small in p), None)
        if at_big is not None and at_small is not None and (at_big < at_small if ">" in clause else at_small < at_big):
            return True
    return False


def _nlp1_side(tail, big, small):
    """「판단 :」 문장이 많다고 적은 쪽 — big · small 가운데 하나, 비교한 말(「…보다 많다 · 적다」)을 못 읽으면 ""

    「A 가 B 보다 많다」 · 「B 보다 A 가 많다」 는 A, 「A 가 B 보다 적다」 는 B 가 많은 쪽입니다. 괄호 안(비율)은 걷고 읽습니다.
    """
    body = _re.sub(r"\([^()]*\)", " ", str(tail))
    cut = body.find("보다")
    if cut < 0 or big not in body or small not in body:
        return ""
    base = big if body.rfind(big, 0, cut) > body.rfind(small, 0, cut) else small     # 「보다」 바로 앞의 낱말이 비교하는 기준
    other = small if base == big else big
    m = _re.search(r"(?<![가-힣])(?:더\s*)?(?:(많|높|크)|(적|낮|작))(?=[다습음으은고아지])", body[cut + 2:])
    if not m:
        return ""
    return other if m.group(1) else base


def _nlp1_num(v):
    """표 칸의 수 — 쉼표 · % 를 걷고 읽습니다(못 읽으면 None)"""
    try:
        return float(str(v).replace(",", "").replace("%", "").replace("−", "-").strip())
    except ValueError:
        return None


def _nlp1_file_gap(fspec, spec, ctx):
    """저장한 표 파일 — 이번 실행에서 저장했는가 · 모양 · 빼야 할 낱말 · 표의 값이 출력한 값과 맞는가.

    (어긋난 점 한 문장 또는 None, 덧붙이는 줄 목록)을 돌려줍니다. utf-8-sig 로 읽지 못하면 cp949 로 한 번 더 읽습니다 —
    「만들 것」 은 저장 인코딩을 정하지 않으므로 cp949 로 저장하고 다시 읽은 표도 받고, ⓘ 로 utf-8-sig 를 권합니다.
    표의 값은 채점이 다시 계산하지 않고 이 셀이 출력한 값과 맞물리는지를 봅니다: 문서 비율 = 문서 수 ÷ 분모 × 100,
    표의 「취소」 행 문서 수 = 출력한 처리 뒤 취소 문서 수(사용자 사전을 넣기 전의 목록으로 만든 표는 여기서 어긋납니다),
    1행의 낱말 · 문서 비율 = 출력한 처리 뒤 1위.
    """
    import os as _os
    fname, notes = fspec["name"], []
    if not _os.path.exists(fname):
        return f"「{fname}」 파일이 없습니다. 이 셀에서 저장합니다", notes
    if _NLP1_PRE.get("fm0") is not None and _os.path.getmtime(fname) == _NLP1_PRE["fm0"]:
        return f"「{fname}」 이 이번 실행에서 저장되지 않았습니다. 예전 파일이 남아 있습니다", notes
    try:
        import pandas as _pd
        try:
            tb = _pd.read_csv(fname, encoding="utf-8-sig")
        except UnicodeDecodeError:
            try:
                tb = _pd.read_csv(fname, encoding="cp949")
            except UnicodeDecodeError:
                return ("파일의 글자를 utf-8-sig 로도 cp949 로도 읽지 못했습니다."
                        " `to_csv(TABLE_FILE, index=False, encoding=\"utf-8-sig\")` 로 저장하세요"), notes
            notes.append("ⓘ 파일이 cp949 로 저장돼 있습니다 — 다시 읽을 때마다 encoding=\"cp949\" 를 적어야 하고, 제출한 파일을"
                         " 웹에서 열면 한글이 깨집니다. `to_csv(TABLE_FILE, index=False, encoding=\"utf-8-sig\")` 로 저장하면"
                         " 어디서나 그대로 읽힙니다")
        if list(tb.columns) != list(fspec["cols"]):
            return f"열이 {list(tb.columns)} 입니다 (기준 {list(fspec['cols'])})", notes
        if len(tb) != int(fspec["rows"]):
            return f"행이 {len(tb)}개입니다 (기준 {fspec['rows']}개)", notes
        words = [str(w).strip() for w in tb.iloc[:, 0]]
        left = [w for w in words if w in set(fspec.get("forbid_words", ()))]
        if left:
            return "낱말 열에 「" + " · ".join(left) + "」 가 남아 있습니다", notes
        cols = list(fspec["cols"])
        counts, shares = [_nlp1_num(v) for v in tb[cols[1]]], [_nlp1_num(v) for v in tb[cols[2]]]
        if _bi.any(v is None for v in counts + shares):
            return f"「{cols[1]}」 · 「{cols[2]}」 열에 숫자가 아닌 칸이 있습니다", notes
        base = _get("OVERSEAS")
        n_all = len(base) if base is not None else ctx["values"].get("국외여행 행 수")
        if n_all:
            off = [w for w, k, s in zip(words, counts, shares) if _bi.abs(100 * k / n_all - s) > 0.006]
            if off:
                return (f"「{cols[2]}」 열이 「{cols[1]}」 ÷ 분모 {int(n_all):,} × 100 과 다른 행이 {len(off)}개입니다"
                        f" (예: {off[0]}). 문서 비율은 그 낱말이 있는 제목 수 ÷ 국외여행 행 수 × 100 입니다"), notes
        def _ok(label):                                    # 값 칸이 ✅ 인 값만 표와 비교합니다 — 이미 ❌ 인 값으로 표까지 ❌ 내지 않습니다
            ref, got = spec.get("values", {}).get(label), ctx["values"].get(label)
            return got if ref and got is not None and float(ref[0]) <= got <= float(ref[1]) else None

        for label in spec.get("values", {}):               # 「처리 뒤 취소 문서 수」 — 표의 그 낱말 행과 같은 수여야 합니다
            m = _re.fullmatch(r"처리 뒤 (\S+) 문서 수", label)
            said = _ok(label)
            if m and said is not None and m.group(1) in words and counts[words.index(m.group(1))] != said:
                return (f"표의 「{m.group(1)}」 행 문서 수 {counts[words.index(m.group(1))]:,.0f} 이 출력한 「{label}」 {said:,.0f} 과"
                        " 다릅니다. 키워드 표는 사용자 사전을 넣은 Kiwi 의 목록(처리 뒤)으로 만듭니다"), notes
        for label in spec.get("texts", {}):                 # 「처리 뒤 1위」 — 표 1행의 낱말과 문서 비율
            said, tails = _ok(label), ctx["tails"](label)
            if said is not None and tails:
                if words[0] not in tails[0] or _bi.abs(shares[0] - said) > 0.051:
                    return (f"표 1행은 「{words[0]}」 {shares[0]:g} % 인데 출력한 「{label}」 은 「{tails[0].strip()[:30]}」 입니다."
                            " 1행이 문서 비율이 가장 큰 낱말이고, 출력은 다시 읽은 표의 1행에서 꺼냅니다"), notes
    except Exception as e:
        return f"파일을 읽지 못했습니다 ({type(e).__name__})", notes
    return None, notes


def _nlp1_checks(key, spec, ctx):
    """이 실습에만 있는 항목 — (이름, 통과, 문장, 힌트 열쇠) 목록. 값·그래프 채점 뒤에 이어 찍힙니다"""
    out = []
    labels = (list(spec.get("texts", {})) + list(spec.get("texts_all", {})) + list(spec.get("texts_re", {})))

    def _bad(name, why, hkey, more=()):
        # 같은 이름의 값 항목이 이미 ❌ 면 그 항목의 힌트 열쇠를 그대로 씁니다 — 힌트는 한 번만 나옵니다
        out.append((name, False, why, next((h for n, h in ctx["failed"] if n == name), hkey), tuple(more)))

    order = None                                     # 차례 항목(「A > B」)의 (큰 쪽, 작은 쪽) — 문장 항목의 결론 방향을 볼 때도 씁니다
    for label, words in spec.get("texts", {}).items():   # 값이 낱말인 항목 — 이 이름이 있는 줄의 「=」·「:」 뒤에 기준 낱말이 있는가
        words = (words,) if isinstance(words, str) else tuple(words)
        tails = ctx["tails"](label, labels)
        pairs = [(w, _nlp1_pair(w)) for w in words]
        if _bi.all(p for _, p in pairs):             # 차례 항목 — 큰 쪽이 앞인가(괄호 안의 값 · 사이에 낀 낱말은 걷고 봅니다)
            order = order or pairs[0][1]
            found = [w for w, p in pairs if _bi.any(_nlp1_ordered(p, t) for t in tails)]
            miss = "기준 차례(비율이 큰 쪽이 「>」 앞)가 아닙니다"
        else:                                        # 낱말 항목 — 부정한 자리(「… 가 나지 않았습니다」)는 세지 않습니다
            found = [w for w in words if _bi.any(_nlp1_has(w, t) for t in tails)]
            miss = "기준 낱말이 아닙니다"
        if found:
            out.append((label, True, f"출력에서 찾은 낱말: 「{found[0]}」", label))
        elif not tails:
            _bad(label, f"이 이름이 있는 줄을 출력에서 찾지 못했습니다. 「{label} = …」 한 줄로 출력하세요", label)
        else:
            _bad(label, f"이 이름이 있는 줄에서 읽은 글자는 「{tails[0].strip()[:40]}」 인데 {miss}", label)
    for label, words in spec.get("texts_all", {}).items():   # 값이 낱말 여럿인 항목 — 이 이름이 있는 줄에 기준 낱말이 모두 있는가
        words = (words,) if isinstance(words, str) else tuple(words)
        tails = ctx["tails"](label, labels)
        miss = [w for w in words if not _bi.any(_nlp1_has(w, t) for t in tails)]
        if tails and not miss:
            out.append((label, True, "출력에서 「" + "」 · 「".join(words) + "」 를 모두 찾았습니다", label))
        else:
            _bad(label, (f"이 이름이 있는 줄을 출력에서 찾지 못했습니다. 「{label} = …」 한 줄로 출력하세요"
                         if not tails else "이 이름이 있는 줄에 「" + "」 · 「".join(miss) + "」 가 없습니다"), label)
    for label, rx in spec.get("texts_re", {}).items():   # 문장 항목 — 이 이름이 있는 줄이 모두 기준 모양(낱말 차례 · 값)이고 결론 방향이 맞는가
        tails = ctx["tails"](label, labels)
        shape = "기준 모양(많은 쪽 낱말 바로 뒤에 그 비율, 적은 쪽은 뒤)"
        kinds = []                                   # 어긋난 줄마다 까닭 — "값"(형태소로 센 값이 아님) · "방향"(큰 쪽을 적다고 적음) · "모양"
        for t in tails:
            side = _nlp1_side(t, *order) if order else ""
            if _re.search(rx, t) and (not order or side != order[1]):
                continue
            ranges = [spec.get("values", {}).get(f"{w} 형태소 비율") for w in (order or ())]
            said = _nums(t)
            if order and side == order[1] and _re.search(rx, t):
                kinds.append("방향")
            elif ranges and _bi.all(ranges) and not _bi.all(_bi.any(float(r[0]) <= v <= float(r[1]) for v in said) for r in ranges):
                kinds.append("값")
            else:
                kinds.append("모양")
        if tails and not kinds:
            out.append((label, True, f"「{label} :」 줄이 {shape}을 갖췄습니다", label))
        elif not tails:
            _bad(label, f"이 이름이 있는 줄을 출력에서 찾지 못했습니다. 「{label} : …」 한 줄로 출력하세요", label)
        else:
            head = f"「{label} :」 줄 {len(tails)}개 가운데 {len(kinds)}개가 "
            why = {"값": "형태소로 센 두 값으로 쓴 문장이 아닙니다 (많은 쪽 낱말 바로 뒤에 그 형태소 비율, 적은 쪽은 뒤)",
                   "방향": "비율이 큰 쪽을 적다고 적었습니다 (「A 민원(큰 값%)이 B 민원(작은 값%)보다 많다」 차례로 씁니다)",
                   "모양": f"{shape}이 아닙니다"}[kinds[0]]
            _bad(label, head + why, label)
    for label, rule in spec.get("rows", {}).items():     # 행 머리로 세는 항목 — 낱말마다 행이 몇 개인가
        pat = _re.compile(rule["re"])
        per = {}
        for line in ctx["text"].splitlines():
            m = pat.match(line)
            if m:
                per[m.group(1)] = per.get(m.group(1), 0) + 1
        need = int(rule.get("n", 1))
        off = [f"{g} {per.get(g, 0)}행" for g in rule.get("groups", per) if per.get(g, 0) != need]
        if per and not off:
            out.append((label, True, f"낱말마다 {need}행씩 모두 {_bi.sum(per.values())}행을 찾았습니다", label))
        else:
            _bad(label, (" · ".join(off) + f" (기준 낱말마다 {need}행)" if per
                         else "「원문 [낱말] 행 row_id · 제목」 모양의 줄을 출력에서 찾지 못했습니다"), label)
    for small, big in spec.get("less", ()):          # 차례 항목 — 값 칸에서 읽은 두 값이 이 차례(작은 쪽 < 큰 쪽)인가
        a, b = ctx["values"].get(small), ctx["values"].get(big)
        name = f"{small} < {big}"
        if a is not None and b is not None and a < b:
            out.append((name, True, f"{a:,.6g} < {b:,.6g}", "less"))
        else:
            _bad(name, ("두 이름 가운데 하나가 있는 줄을 출력에서 찾지 못했습니다" if a is None or b is None
                        else f"{a:,.6g} 이 {b:,.6g} 보다 작지 않습니다"), "less")
    fspec = spec.get("file")
    if fspec:                                        # 저장한 표 파일 — 이번 실행에서 저장했는가 · 모양 · 빼야 할 낱말 · 출력한 값과 맞는가
        gap, notes = _nlp1_file_gap(fspec, spec, ctx)
        if gap:
            _bad("키워드 표 파일", gap, "file", notes)
        else:
            out.append(("키워드 표 파일", True,
                        f"「{fspec['name']}」 {fspec['rows']}행 {len(fspec['cols'])}열을 다시 읽었고 표의 값이 출력과 맞습니다", "file",
                        tuple(notes)))
    return out


# 권 설정 · pre: 저장할 표 파일의 실행 전 수정 시각 · spec: 낱말과 값을 함께 찍는 칸의 출력 꼴 · checks: 글자 항목 · 원문 행 · 두 값의 차례 · 저장한 표 파일
_BOOK = {"pre": _nlp1_file_pre, "spec": _nlp1_spec, "checks": (_nlp1_checks,)}
# ━━ 채점 기준 끝 — 아래는 모든 실습이 함께 쓰는 채점 코드입니다 ━━


# ───────────────────────── 출력에서 값 읽기 ─────────────────────────
#  숫자 하나 — 천 단위 쉼표(8,760) · 지수 표기 · 부호. 영문·숫자·밑줄·점 바로 뒤에 붙은 숫자는
#  이름의 일부라 읽지 않습니다 — 「float64」·「int64」 의 64, 「C23」 의 23, 「v2.3」 의 3. 영문에 붙임표로
#  이은 숫자도 이름입니다 — 「Chronos-2」 의 2, 「t-1」 의 1, 「top-3」 의 3.
_NUM = (r"(?<![A-Za-z_\d.])(?<![A-Za-z]-)[-+]?(?:\d{1,3}(?:,\d{3})+(?!\d)|\d+)(?:\.\d*)?(?:[eE][-+]?\d+)?")
_NUM_RE = _re.compile(_NUM)
_KW_RE = _re.compile(r"[A-Za-z_][A-Za-z0-9_]*=(?!=)")       # 「period=24」 같은 인자 설명
_KW_VAL = _re.compile(r"\s*(?:'[^']*'|\"[^\"]*\"|\([^()]*\)|\[[^\[\]]*\]|[^\s,;)\]]+)")
#  Series 를 통째로 찍은 출력의 끝 줄 — 「Name: coef, dtype: float64」 · 「Freq: h, Name: cnt, Length: 168, dtype: float64」 ·
#  「dtype: int64」. f-string 안에 Series 를 넣으면 뒤의 글자(단위)가 이 줄 끝에 붙으므로(「… dtype: float64 대/도」)
#  dtype 뒤의 꼬리 글자는 가리지 않습니다.
_SERIES_RE = _re.compile(r"^(?:(?:Name|Length|Freq): .*, )?dtype: [A-Za-z]\w*", _re.M)
_GLYPH = {**{c: "-" for c in "—–―‒‐‑-−"}, **{c: "·" for c in "·ㆍ•‧・∙⋅"}}   # 겉보기가 같은 대시·가운뎃점은 같게 봅니다
_SEP = set(",;|") | set("·ㆍ•‧・∙⋅") | set("—–―")          # 값 뒤에서 다른 말로 넘어가는 자리


def _num_tokens(text):
    """출력 텍스트의 숫자 — (찍힌 글자, 값) 목록 (유니코드 마이너스 − 는 - 로 읽습니다)"""
    picked = []
    for m in _NUM_RE.finditer((text or "").replace("−", "-")):
        try:
            picked.append((m.group(0), float(m.group(0).replace(",", ""))))
        except ValueError:
            continue
    return picked


def _nums(text):
    """출력 텍스트에서 숫자만 뽑습니다"""
    return [v for _, v in _num_tokens(text)]


def _unwrap_np(txt):
    """numpy·pandas 의 값 표기를 숫자로 폅니다 — 화면에는 맞는 수가 보이는데 채점이 못 읽는 일이 없게.

    np.float64(40.2) · np.int64(3) · float64(40.2) → 40.2 · 3 (numpy 2 가 dict·list 출력에 남기는 꼴),
    array([40.2]) · [40.2] · [80.] → 40.2 · 80. (원소 하나인 배열 — `.values` 뒤 `[0]` 을 빠뜨린 꼴).
    원소가 둘 이상인 배열은 펴지 않습니다 — 어느 값인지 모릅니다(채점 표가 「숫자 하나로」 를 안내합니다).
    """
    txt = _re.sub(r"\b(?:np|numpy)\.[A-Za-z_]\w*\(\s*([^()]*?)\s*\)", r"\1", txt or "")
    txt = _re.sub(r"\b(?:float|int|uint|complex)(?:8|16|32|64|128)\(\s*([^()]*?)\s*\)", r"\1", txt)
    txt = _re.sub(r"\barray\(\s*(\[[^\[\]()]*\])(?:\s*,\s*dtype=[\w.]+)?\s*\)", r"\1", txt)
    return _re.sub(r"\[\s*([-−+]?\d[\d,]*\.?\d*(?:[eE][-+]?\d+)?)\s*\]", r" \1 ", txt)


def _bare_gap(gap):
    """이름과 값(또는 「=」) 사이가 따옴표·굵게 표시(**)·표의 세로줄·빈칸·괄호 설명·줄표뿐인가 — 그러면 그 이름의 값 줄입니다"""
    gap = _re.sub(r"\([^()]*\)|\[[^\[\]]*\]", "", gap or "")
    return not _re.sub(r"[\s'\"`*」』”’|│┃—–―\-:=,·：＝]", "", gap)


def _paren_open(t, unit):
    """괄호 밖에 숫자가 하나도 없는 꼬리에서 괄호 안의 값 하나만 드러낸 글 — 그런 꼴이 아니면 None.

    「명사 + 용언 1위 = 하다 (18.74 %)」 처럼 낱말 뒤 괄호에 비율을 넣거나 `print("이름 =", (낱말, 비율))` 로 튜플을 그대로
    찍으면 값이 괄호 안에만 있습니다. 괄호 안은 설명으로 보고 건너뛰므로 맞는 출력이 「숫자를 찾지 못했습니다」 가 됩니다.
    괄호 밖에서 값을 하나도 못 읽었을 때만 봅니다 — 읽은 값이 있으면 그 값이 먼저입니다(「확신 = 26 (전체 168 시간 가운데)」).

    - 그릇: 값 표지(또는 이름) 바로 뒤가 여는 괄호이고, 안이 쉼표로 가른 낱말(숫자 없는 글)과 숫자(짧은 단위까지)뿐이며
      숫자가 하나면 그 숫자입니다 — `('하다', 18.74)` · `(하다, 18.74 %)` · `(18.74 %)`. 숫자가 둘 이상인 그릇은 어느 값인지
      모르므로 읽지 않습니다(`[0.57 0.09]` · `(1, 0, 1)`).
    - 기준 단위: 괄호 안에 기준 단위가 바로 붙은 수가 있으면 그 수입니다 — 「하다 (18.74 %)」 · 「하다(18.74%)」 ·
      「하다 (문서 비율 18.74 %)」. 단위가 없는 칸의 괄호 안 숫자(「(기준: 2 미만이면 비슷)」)는 지금처럼 설명입니다.
    """
    groups = _spans(t)
    if not groups:
        return None
    pick = None
    a, b = groups[0]
    before = t[:a].rstrip()
    if not before or before[-1] in "=:→：＝":            # 그릇 — 표지(또는 이름) 바로 뒤의 괄호
        at, spots, clean = a + 1, [], True
        for piece in t[a + 1:b].split(","):
            body = _re.sub(r"[()\[\]'\"`\s]", "", piece)
            m = _NUM_RE.match(body)
            if m and len(body) - m.end() <= 4 and not _re.search(r"\d", body[m.end():]):
                spots.append((at, at + len(piece)))        # 숫자 하나(짧은 단위까지)
            elif _re.search(r"[\d=:→：＝]", body):
                clean = False                              # 숫자가 섞인 말 · 다른 표지 — 그릇이 아니라 설명입니다
            at += len(piece) + 1
        if clean and len(spots) == 1:
            m = _NUM_RE.search(t, spots[0][0], spots[0][1])
            end = spots[0][1]
            while m and end > m.end() and t[end - 1] in ")]'\"` \t":
                end -= 1
            pick = (m.start(), end) if m else None
    if pick is None and unit:
        for a, b in groups:
            for m in _NUM_RE.finditer(t, a + 1, b):
                rest = t[m.end():b]
                gap = len(rest) - len(rest.lstrip())
                if rest[gap:].startswith(unit) or (unit.startswith("%") and rest[gap:].startswith("%")):
                    pick = (m.start(), m.end() + gap + (len(unit) if rest[gap:].startswith(unit) else 1))
                    break
            if pick is not None:
                break
    if pick is None:
        return None
    hide = {i for a, b in groups for i in range(a, _bi.min(b, len(t) - 1) + 1)} - set(range(pick[0], pick[1]))
    return "".join(" " if i in hide else ch for i, ch in enumerate(t))


def _read_value(tail, series=False, inner=0, unit=""):
    """이름 뒤 꼬리에서 그 이름의 값 하나 — (값, 찍힌 글자, 값 뒤 글자, 이름 바로 뒤 값인가, 꼴, 읽은 길) · 없으면 None.

    꼴은 "plain"·"series"(Series 의 인덱스를 건너뛴 값)·"kw"(인자 설명의 값), 읽은 길은 "marker"(값 표지 뒤)·
    "bare"(표지 없이 이름 뒤 첫 숫자)·"kw" 입니다.

    - 괄호·대괄호 안(설명 · 「(기준: 2 미만이면 비슷)」 · 「(m=7)」)은 건너뜁니다.
    - 「period=24, robust=True」 같은 인자 설명은 값이 아닙니다. 그 뒤의 「→」·「=」 뒤를 읽고, 인자 설명뿐이면 마지막 인자 값입니다.
    - 값 표지(「=」·「:」·「→」, 전각 「＝」·「：」)가 숫자보다 먼저 나오면 그 뒤 첫 숫자가 값입니다. 시각 23:00 의 「:」 는 표지가 아닙니다.
    - 숫자가 표지보다 먼저 나오면 — 그 숫자와 표지 사이에 쉼표·가운뎃점·줄표가 있으면 그 숫자가 값이고
      (「기온 계수 -40.20 대/도, 크기 = 40.20」 의 -40.20), 없으면 숫자는 이름의 설명이고 표지 뒤가 값입니다
      (「…, 원점 5 11-16 23:00 까지 = 385.06」 의 385.06). 「→」 는 앞에 숫자가 있으면 해석으로 넘어가는 자리입니다.
      이름 바로 뒤(빈칸·괄호 설명만 사이에 둔)의 소수는 구분 기호가 없어도 값입니다 — 「이름 4.837 대 기준값 = 0.271」 의
      4.837. 이름 바로 뒤의 정수(「이름 2개 원점 평균 = 0.271」 의 2)는 개수를 적은 설명일 수 있어 표지 뒤를 읽습니다.
    - series=True(출력에 「dtype:」 행이 있음)면 「= 24    277.46」 의 인덱스 24 대신 값 277.46 을 읽습니다.
    - inner 는 이름이 괄호 안에서 잡혔을 때의 괄호 깊이입니다 — 그 괄호가 닫히기 전의 숫자는 읽지 않습니다.
    - 괄호 밖에서 값을 하나도 못 읽었으면 괄호 안의 값 하나를 읽습니다 — 값이 그릇(튜플)에 담겼거나, 괄호 안에 기준 단위
      (unit)가 붙은 수가 있을 때입니다(`_paren_open`). 이렇게 읽은 줄은 약한 줄입니다(`_read_paren` · `_value_use`, 0.4.2).
    """
    return _read_paren(tail, series, inner, unit)[0]


def _read_paren(tail, series=False, inner=0, unit=""):
    """`_read_value` 의 값과 그 값을 괄호 안에서만 읽었는가 — (값 또는 None, 괄호 안에서만 읽음).

    괄호 안에서만 읽은 값은 약한 줄의 값입니다 — 「이름 (제로샷 41.6 % 보다 높음)」 처럼 덧붙인 설명 줄의 수가 맨 이름
    「이름 = 값」 줄과 부딪치지 않게, 다른 값 줄이 있으면 `_value_use` 가 쓰지 않습니다(그런 줄뿐이면 0.4 의 53번대로 읽습니다).
    """
    t = "(" * inner + _unwrap_np(tail).replace("−", "-")
    got = _scan_value(t, 0, series, True)
    if got is None and not inner and t.count("(") + t.count("[") > t.count(")") + t.count("]"):
        got = _scan_value(_re.sub(r"[()\[\]]", " ", t), 0, series, True)   # 닫히지 않은 괄호 — 괄호 기호만 떼고 다시
    if got is None and not inner:
        opened = _paren_open(t, unit)                     # 「하다 (18.74 %)」 · `('하다', 18.74)` — 값이 괄호 안에만 있습니다
        if opened is not None:
            got = _scan_value(opened, 0, series, True)
            return got, got is not None
    return got, False


def _scan_value(t, i, series, top):
    """`_read_value` 의 한 단계 — top 이면 이름 바로 뒤, 아니면 값 표지 뒤에서 읽습니다"""
    depth, cand, sep, kws, start, n = 0, None, False, [], i, len(t)
    while i < n:
        ch = t[i]
        if ch in "([":
            depth += 1
            i += 1
            continue
        if ch in ")]":
            depth = _bi.max(0, depth - 1)
            i += 1
            continue
        if depth:
            i += 1
            continue
        kw = _KW_RE.match(t, i)
        if kw and (i == 0 or not (t[i - 1].isalnum() or t[i - 1] == "_")):
            val = _KW_VAL.match(t, kw.end())
            toks = _num_tokens(val.group(0)) if val else []
            if toks:
                kws.append(toks[0])
            i = val.end() if val else kw.end()
            continue
        if ch in "=:→：＝" or t.startswith("->", i):
            step = 2 if t.startswith("->", i) else 1
            if ch == ":" and 0 < i < n - 1 and t[i - 1].isdigit() and t[i + 1].isdigit():
                i += 1                                            # 시각·비율(23:00 · 1:2)
                continue
            if ch == "=" and t[i + 1:i + 2] == "=":
                i += 2
                continue
            arrow = ch == "→" or step == 2
            if cand is not None and (sep or arrow or not top or (cand[3] and "." in cand[1])):
                return cand                                       # 값 뒤의 다른 말(「크기 = …」·「→ 해석」)
            got = _scan_value(t, i + step, series, False)
            if got is not None:
                if top:
                    got = got[:3] + (cand is None and _bare_gap(t[start:i]),) + got[4:5] + ("marker",)
                return got
            if cand is not None:
                return cand
            i += step
            continue
        m = _NUM_RE.match(t, i)
        if m:
            if cand is None:
                raw, end, kind = m.group(0), m.end(), "plain"
                if series:                                        # 「= 24    277.46」 — Series 의 인덱스와 값
                    pair = _re.match(r"(-?\d+)\s{2,}(" + _NUM + ")", t[i:])
                    if pair:
                        raw, end, kind = pair.group(2), i + pair.end(), "series"
                try:
                    cand = (float(raw.replace(",", "")), raw, t[end:], top and _bare_gap(t[start:i]), kind,
                            "bare" if top else "marker")
                except ValueError:
                    cand = None
                if cand is not None and not top:
                    return cand
            i = m.end()
            continue
        if cand is not None and (ch in _SEP or (ch == "-" and t[i - 1:i] == " " and t[i + 1:i + 2] == " ")):
            sep = True
        i += 1
    if cand is not None:
        return cand
    if kws:                                                       # 인자 설명뿐 — 마지막 인자의 값(「fs_hour=0.5715」)
        return (kws[-1][1], kws[-1][0], "", False, "kw", "kw")
    return None


def _depth(txt):
    """txt 를 지난 뒤의 괄호 깊이"""
    depth = 0
    for ch in txt or "":
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
    return depth


def _spans(txt):
    """맨 바깥 괄호·대괄호 묶음의 (여는 자리, 닫는 자리) 목록 — 닫히지 않은 묶음은 글 끝까지입니다"""
    out, depth, start = [], 0, 0
    for i, ch in enumerate(txt or ""):
        if ch in "([":
            if depth == 0:
                start = i
            depth += 1
        elif ch in ")]" and depth:
            depth -= 1
            if depth == 0:
                out.append((start, i))
    if depth:
        out.append((start, len(txt)))
    return out


def _squeeze(txt):
    """빈칸과 괄호 기호를 걷고 대시·가운뎃점 글리프를 하나로 맞춘 글자"""
    return "".join(_GLYPH.get(ch, ch) for ch in (txt or "") if not ch.isspace() and ch not in "()[]")


def _tighten(txt, own=None):
    """빈칸과 괄호 기호를 걷어낸 글자 · 원래 자리 — 대시·가운뎃점 글리프는 하나로 맞춥니다.

    own 이 None 이면 괄호 안의 말은 둡니다(기호만 걷음). own 이 집합이면 안의 말이 own 에 있는 묶음만 두고 나머지 묶음은
    안의 말까지 걷습니다. 줄에 끼어든 괄호 설명(「…입력(512일)에 있는…」)은 이름의 일부가 아니지만, 이름에 원래 있는
    괄호 안의 말(「…MASE(Chronos-2 + 공휴일)」)은 이름을 가르는 부분이라 걷지 않습니다.
    """
    txt = txt or ""
    drop = set()
    if own is not None:
        for a, b in _spans(txt):
            if _squeeze(txt[a + 1:b]) not in own:
                drop.update(range(a, b + 1))
    keep = [i for i, ch in enumerate(txt) if i not in drop and not ch.isspace() and ch not in "()[]"]
    return "".join(_GLYPH.get(txt[i], txt[i]) for i in keep), keep


def _canon(txt):
    """숫자 표기를 맞춥니다 — 5.0 → 5 · 3.50 → 3.5 · 1,234 → 1234"""
    def _one(m):
        s = m.group(0).replace(",", "")
        return s.rstrip("0").rstrip(".") if "." in s else s
    return _re.sub(r"\d[\d,]*(?:\.\d+)?", _one, txt or "")


def _label_spot(line, label):
    """줄에서 이름이 있는 자리 (시작, 끝, 느슨하게 찾았는가) — 없으면 None.

    차례로 찾습니다 — 정확히 같은 글자(괄호 밖 자리 먼저) → 빈칸과 줄에 끼어든 괄호 설명을 걷어낸 꼴(대시 — – ― - 와
    가운뎃점 · ㆍ • ‧ ・ 는 같게) → 괄호 기호만 뗀 꼴(「횟수(D)」 와 「횟수 D」) → 숫자 표기까지 맞춘 꼴은 부르는 쪽이 봅니다.

    이름에 괄호가 있으면 그 안의 말도 이름입니다. 「공휴일 있는 세 주 MASE(Chronos-2 + 공휴일)」 을 찾을 때
    「공휴일 있는 세 주 MASE(Chronos-2 제로샷) = 2.763」 은 다른 이름의 줄입니다 — 괄호 안까지 걷고 찾으면 나란히 찍은
    다른 모델의 줄이 같은 이름 줄이 되어 맞는 코드가 「줄이 2개이고 값이 서로 다릅니다」 로 막힙니다. 괄호 묶음을 아예
    적지 않은 줄(「공휴일 있는 세 주 MASE = 1.022」)만 느슨하게 찾은 것으로 돌려주고, 부르는 쪽은 제대로 찾은 줄이 하나라도
    있으면 느슨하게 찾은 줄을 쓰지 않습니다(`_occurrences`).
    """
    if label and label in line:
        spots = [m.start() for m in _re.finditer(_re.escape(label), line)]
        top = [p for p in spots if _depth(line[:p]) == 0]
        # 「{'상태별 리뷰 수': 178, '리뷰 수': 200}」 — 앞에 다른 낱말이 붙은 자리(더 긴 이름)보다 이름으로 시작하는 자리가 먼저입니다
        at = next((p for p in top if not line[:p].rstrip()[-1:].isalpha()), top[0] if top else spots[0])
        return at, at + len(label), False
    full = _tighten(label)[0]
    if not full:
        return None
    groups = [_squeeze(label[a + 1:b]) for a, b in _spans(label)]
    for own in (set(groups), None):
        tight_line, keep = _tighten(line, own)
        at = tight_line.find(full)
        if at >= 0:
            return keep[at], keep[at + len(full) - 1] + 1, False
    if not groups:
        return None
    # 이름의 괄호 묶음을 적지 않은 줄 — 뼈대(괄호 묶음을 뺀 이름)만으로 찾습니다. 이름의 괄호 자리에 다른 말의 괄호가
    # 있으면(「MASE(Chronos-2 제로샷)」) 이 이름의 줄이 아닙니다
    base, base_keep = _tighten(label, set())
    base_line, keep = _tighten(line, set())
    at = base_line.find(base) if base else -1
    if at < 0:
        return None
    spot_of = {_bi.sum(1 for k in base_keep if k < a): g for (a, _b), g in zip(_spans(label), groups)}
    for a, b in _spans(line):
        rel = _bi.sum(1 for k in keep if k < a) - at
        if rel in spot_of and _squeeze(line[a + 1:b]) != spot_of[rel]:
            return None
    return keep[at], keep[at + len(base) - 1] + 1, True


_OP_SURE = set("−+×÷*")                    # 식의 연산 기호
_OP_MAYBE = set("-–‒/><≥≤→")               # 붙임표 · 빗금 · 부등호 · 화살표 — 식일 수도, 말을 가르는 자리일 수도 있습니다


def _operand_head(head):
    """이름 앞의 글이 「다른 이름 + 연산 기호」 로 끝나는가 — "expr"(이름이 식의 뒤 항) · "maybe"(식인지 모름) · "".

    「명사만 1위 − 명사 + 용언 1위 : −7.82 %p」 에서 「명사 + 용언 1위」 는 뺄셈의 뒤 항이고 −7.82 는 두 값의 차입니다.
    이 줄을 뒤 이름의 값 줄로 읽으면 제 값 줄(「명사 + 용언 1위 = 하다 18.74 %」)과 값이 달라 맞는 출력이
    「줄이 2개이고 값이 서로 다릅니다」 가 됩니다. 이름에 원래 든 기호(「명사 + 용언 1위」 의 +)는 이름의 일부라 보지 않습니다.
    기호 앞의 토막이 숫자로 시작하면(「A = 0.271 / 이름 = 0.281」 · 「A = 40.2 대/도 - 이름 = …」) 그 기호는 항목을 가르는
    자리이고, 기호 앞에 말이 없으면(「- 이름 = 값」 · 「* 이름」 · 「> 이름」) 글머리표입니다 — 둘 다 식이 아닙니다.
    붙임표 · 빗금 · 부등호 · 화살표(「A - 이름」 · 「A / 이름」 · 「A > 이름 : a > b」 · 「A → 이름 : a → b」)는 말을 가르는 자리로도
    쓰여 식인지 알 수 없습니다 — "maybe" 로 돌려주고, 부르는 쪽이 이름으로 시작하는 줄을 먼저 씁니다.
    """
    h = (head or "").rstrip()
    op = h[-1:]
    if not op or (op not in _OP_SURE and op not in _OP_MAYBE):
        return ""
    before = h[:-1].rstrip()
    if not before or not (before[-1].isalnum() or before[-1] in ")]%\0」』”’\"'"):
        return ""
    if op in "-–‒" and h[-2:-1].strip():           # 앞말에 붙은 붙임표는 낱말을 잇는 글자입니다
        return ""
    cut = _bi.max(before.rfind(ch) for ch in "=:→：＝,;|")
    seg = before[cut + 1:].strip()
    if not seg or _NUM_RE.match(seg.replace("−", "-")):
        return ""
    return "expr" if op in _OP_SURE else "maybe"


def _mark_at(tail):
    """꼬리에서 괄호 밖 첫 값 표지(「=」·「:」, 전각 포함)의 자리 — 없으면 -1. 시각 23:00 의 「:」 는 표지가 아닙니다"""
    depth, n = 0, len(tail or "")
    for i, ch in enumerate(tail or ""):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
        elif depth == 0 and ch in "=:：＝":
            if ch == ":" and 0 < i < n - 1 and tail[i - 1].isdigit() and tail[i + 1].isdigit():
                continue
            return i
    return -1


#  강한 값 줄(0.4.2) — 이름이 줄 머리에 있고, 이름 바로 뒤가 값 표지(「=」·「:」·「→」, 전각 포함)이며 표지 바로 뒤가 숫자인 줄.
#  줄 머리: 이름 앞에는 빈칸 · 글머리표 · 번호(「1.」·「1)」·「(1)」·「①」) · 여는 따옴표 · 굵게 표시 · 사전의 「{」 만 옵니다
_LEAD_OK = _re.compile(r"(?:\s|[-*•‧·+>▪◦▶■□○●※→]|\d{1,3}[.)](?=\s)|\(\d{1,3}\)|[①-⑳⑴-⒇]|['\"`「『“‘{]|[(\[](?=['\"]))*")
#  이름과 표지 사이에는 닫는 따옴표 · 굵게 표시 · 빈칸만 — 숫자가 numpy 꼴(np.float64(…) · [ … ] · array([…]))로 싸여도 됩니다
_STRONG_TAIL = _re.compile(r"[\s'\"`*」』”’]*(?:=(?!=)|:|：|＝|→|->)\s*"
                           r"(?:(?:np|numpy)\.[A-Za-z_]\w*\(\s*|(?:float|int|uint)(?:8|16|32|64|128)\(\s*|array\(\s*\[\s*|\[\s*)?"
                           r"[-+−]?\.?\d")
#  한 줄에 값 짝을 이어 적는 구분 기호 — 쉼표(숫자 사이의 천 단위 쉼표는 아님) · 쌍반점 · 세로줄 · 빗금 · 가운뎃점 · 줄표 · 화살표 · 빈칸 둘 이상
_CHAIN_SEP = _re.compile(r"(?<!\d),|,(?!\d)|[;|/·ㆍ•‧・∙⋅—–―→]|->|\s-\s|\s{2,}")
_NAME_RE = {}                              # 이름 → 빈칸 · 대시 · 가운뎃점 글리프를 가리지 않고 찾는 정규식(한 번만 짓습니다)


def _name_len_at(text, i, names):
    """text 의 i 자리에서 시작하는 이름(names 가운데 가장 긴 것)의 글자 수 — 없으면 0. 가려 둔 더 긴 이름(\\0)도 이름입니다"""
    if text[i:i + 1] == "\0":
        j = i
        while j < len(text) and text[j] == "\0":
            j += 1
        return j - i
    best = 0
    for name in names:
        if not name:
            continue
        if text.startswith(name, i):
            best = _bi.max(best, len(name))
            continue
        pat = _NAME_RE.get(name)
        if pat is None:
            same = {}
            for k, v in _GLYPH.items():
                same.setdefault(v, set()).add(k)
            pat = _NAME_RE[name] = _re.compile(r"\s*".join(
                "[" + _re.escape("".join(sorted(same[_GLYPH[ch]] | {_GLYPH[ch]}))) + "]" if ch in _GLYPH else _re.escape(ch)
                for ch in name if not ch.isspace()))
        m = pat.match(text, i)
        if m:
            best = _bi.max(best, m.end() - i)
    return best


def _chain_head(lead, names):
    """이름 앞의 글(lead)이 「이름 = 숫자」 짝을 구분 기호로 이어 적은 값 줄의 앞부분인가 — 「확신 = 26 시간 · 보류 = 142 시간」 의 「보류」.

    맨 앞 짝은 줄 머리에서 시작하고, 짝마다 이름(값 칸의 이름) 바로 뒤가 값 표지이고 표지 바로 뒤가 숫자이며, 짝 사이에는 값 표지가 없는
    글과 구분 기호만 있습니다. 「(참고) 제로샷: A = 85.1 % · B = 41.6 %」 의 B 는 맨 앞 짝이 줄 머리에서 시작하지 않아 아닙니다 —
    비교하려고 덧붙인 줄이라 그 줄의 수가 맨 이름 「B = 값」 줄과 부딪치지 않게 합니다.
    """
    m = _LEAD_OK.match(lead)
    i = m.end() if m else 0
    while True:
        k = _name_len_at(lead, i, names)
        if not k:
            return False
        mt = _STRONG_TAIL.match(lead, i + k)
        if not mt:
            return False
        j, nxt = mt.end(), None
        for sm in _CHAIN_SEP.finditer(lead, j):
            if _re.search(r"[=＝：]|(?<!\d):|:(?!\d)", lead[j:sm.start()]):
                return False                                  # 짝 사이의 다른 값 표지 — 이름을 모르는 짝입니다
            q = sm.end()
            while q < len(lead) and lead[q] in " \t'\"`*「『“‘":
                q += 1
            if q >= len(lead):
                return True                                   # 지금 이름 바로 앞의 구분 기호
            if _name_len_at(lead, q, names):
                nxt = q
                break
        if nxt is None:
            return False
        i = nxt


def _lead_kind(lead):
    """이름 앞 글(lead)의 꼴 — "top"(줄 머리) · "num"(날짜 · 숫자 · % 뒤) · "sep"(구분 기호 뒤) · ""(그 밖)"""
    if _LEAD_OK.fullmatch(lead or ""):
        return "top"
    body = (lead or "").rstrip(" \t'\"`*「『“‘")
    last = body[-1:]
    if last.isdigit() or last == "%":
        return "num"
    if last and (last in ",;|/·ㆍ•‧・∙⋅—–―" or (last in "-–‒" and body[-2:-1].isspace())):
        return "sep"
    return ""


def _occurrences(lines, label, others=(), series=False, unit=""):
    """이름이 있는 줄마다 하나씩 — {"tail": 이름 뒤 꼬리, "seg": 이 항목의 토막, "val": `_read_value` 결과, "boxed": 배열인가}.

    값줄의 숫자를 한 주머니에 담아 기준 범위와 대조하면 **다른 항목 줄의 값**이 이 항목을 헛통과시킵니다.
    그래서 이름이 있는 줄의 이름 뒤만 봅니다. 꼬리에 다른 항목의 이름(`others`)이 나오면 거기서 끊고
    (표 머리 줄에서 옆 칸 이름 안의 「365일」·「168」 을 이 항목의 값으로 읽지 않게), 이 이름을 품은 더 긴
    이름(「공식 차수」 에 대한 「공식 차수 AIC」)이 있는 자리는 이 이름으로 치지 않습니다.
    이름이 괄호 안에서만 잡히면(「… (명목 포함률 80% 기준)」) 그 괄호가 닫히기 전의 숫자는 읽지 않습니다. 따옴표로
    싼 이름(튜플·리스트를 그대로 찍은 `('이름', 0.281)`)은 괄호가 값을 담은 그릇이라 그 안의 값을 읽습니다.
    "seg" 는 이 이름부터 다른 항목 이름 앞까지입니다 — 줄의 첫 이름이면 이름 앞의 말까지 넣습니다.
    "line" 은 줄 원문, "loose" 는 이름의 괄호 묶음을 적지 않은 줄인가입니다 — 이름을 제대로 적은 줄이 하나라도 있으면
    느슨하게 찾은 줄은 돌려주지 않습니다(`_label_spot`). "after" 는 다른 항목 이름에서 끊지 않은 이름 뒤의 글입니다
    (값이 낱말 · 문장인 항목을 보는 권 검사가 씁니다 — `_text_tails`). "pre" 는 이름 바로 앞에 다른 낱말이 붙어 있는가입니다
    (「상태별 리뷰 수 : …」 의 「리뷰 수」 — 더 긴 이름의 줄일 수 있어 `_value_use` 가 이름으로 시작하는 줄을 먼저 씁니다.
    붙임표 · 빗금 뒤의 이름도 여기에 넣습니다).
    이름 앞에 「다른 이름 + 연산 기호」 가 있는 줄은 이름이 식의 뒤 항이라 이 이름의 줄이 아닙니다. 기호가 붙임표 · 빗금이면
    식인지 알 수 없어 느슨하게 찾은 줄로 둡니다(`_operand_head`).
    "tag" 는 이름 앞에 꼬리표가 붙은 줄인가입니다 — 「(전임 코드) 이름 = …」 · 「고치기 전: 이름 = …」 처럼 이름 앞의 말이
    닫는 괄호나 쌍점으로 끝납니다. "noted" 는 이름과 값 표지 사이에 괄호 설명이 낀 줄인가입니다 — 「이름(50만 건 전체, 이론) = …」.
    둘 다 비교하려고 덧붙인 줄일 수 있어, 맨 이름 「이름 = 값」 줄이 따로 있으면 `_value_use` 가 쓰지 않습니다(그런 줄뿐이면 그 값을 읽습니다).
    "strong" 은 강한 값 줄인가입니다(0.4.2) — 이름이 줄 머리에 있고(앞에는 글머리표 · 번호 · 따옴표만, `_LEAD_OK`) 이름 바로 뒤가
    값 표지이며 표지 바로 뒤가 숫자인 줄(`_STRONG_TAIL`), 그리고 그런 「이름 = 숫자」 짝을 구분 기호로 이어 적은 줄의 뒤 짝(`_chain_head`)입니다.
    강한 값 줄이 하나라도 있으면 `_value_use` 가 그 줄끼리만 값을 읽고 부딪침을 봅니다. "aside" 는 이름 앞이 날짜 · 숫자로 끝나거나
    (「2025-08-13 이름: 78.9%」) 구분 기호 바로 뒤에 이름이 온 줄(「명목 80% 와의 차: A -0.5 %p · 이름 -13.6 %p」), 꼬리표 뒤의 이름에
    표지 없이 숫자가 붙은 줄(같은 줄의 「명목 80% 와의 차: 이름 -0.5 %p」)입니다 — 강한 값 줄이 없을 때 이름으로 시작하는 줄이 따로
    있으면 `_value_use` 가 쓰지 않습니다. "paren" 은 값을 괄호 안에서만 읽은 줄입니다(`_read_paren`) — 다른 값 줄이 있으면 쓰지 않습니다.
    unit 은 이 칸의 기준 단위입니다 — 값이 괄호 안에만 있는 줄에서 기준 단위가 붙은 수를 읽습니다(`_paren_open`).
    """
    rivals = [o for o in others if o and o != label]
    longer = [o for o in rivals if label in o]
    # 빠른 거르기 — 이름 뼈대의 글자(한글·영문)가 다 있지 않은 줄에는 이름이 없습니다(출력이 수만 줄일 때 채점이 늦지 않게)
    need = {ch for ch in _tighten(label, set())[0] if ch.isalpha()}
    out = []
    for line in lines:
        if not need.issubset(line):
            continue
        masked = line
        for o in longer:
            masked = masked.replace(o, "\0" * len(o))
        spot = _label_spot(masked, label)
        plain = line
        if spot is None:
            masked = _canon(masked)
            spot = _label_spot(masked, _canon(label))
            plain = masked
        if spot is None:
            continue
        start, end, loose = spot
        role = _operand_head(masked[:start])
        if role == "expr":
            continue                                              # 「다른 이름 − 이름 : 차」 — 이 이름의 값 줄이 아닙니다
        loose = loose or role == "maybe"
        tail = masked[end:]
        stops = []
        for o in rivals:
            at = tail.find(o) if o not in longer else -1
            while at >= 0 and _depth(tail[:at]):                  # 괄호 설명 안에 나온 다른 항목 이름에서는 끊지 않습니다 —
                at = tail.find(o, at + 1)                         # 「이상치 비율 (분모 N) = 37.68 %」 의 「분모」
            if at >= 0:
                stops.append(at)
        if stops:
            tail = tail[:_bi.min(stops)]
        inner = _depth(masked[:start])
        quote = masked[start - 1:start]
        if inner and quote in ("'", '"') and masked[end:end + 1] == quote:
            inner = 0                                             # ('이름', 값) · ['이름', 값] — 그릇 안의 값
        head = masked[:end]
        seg = (tail if _bi.any(o in head for o in rivals if o not in longer) else head + tail).replace("\0", "")
        val, in_paren = _read_paren(tail, series, inner, unit)
        if val is not None and inner:
            val = val[:3] + (False,) + val[4:]
        boxed = val is None and _bi.bool(_re.search(r"\[[^\[\]]*\d[^\[\]]*\]", _unwrap_np(tail)))
        lead = masked[:start].rstrip()
        cut = _mark_at(tail)
        kind = _lead_kind(masked[:start])                         # 줄 머리 · 날짜와 숫자 뒤 · 구분 기호 뒤
        chain = kind != "top" and not inner and _chain_head(masked[:start], [label] + rivals)
        tag = lead[-1:] in tuple(":：)]」』") and _bi.any(ch.isalpha() for ch in lead)
        out.append({"tail": tail.replace("\0", ""), "seg": seg, "val": val, "boxed": boxed, "line": line, "loose": loose,
                    "after": plain[end:].replace("\0", ""),
                    "pre": role == "maybe" or lead[-1:].isalpha(),
                    "tag": tag,
                    "noted": cut >= 0 and _bi.bool(_re.search(r"[(\[]", tail[:cut])) and _bare_gap(tail[:cut]),
                    "strong": (val is not None and val[3] and val[5] == "marker" and not inner and not in_paren
                               and (kind == "top" or chain) and _bi.bool(_STRONG_TAIL.match(tail))),
                    "paren": in_paren,                            # 값을 괄호 안에서만 읽은 줄(약한 줄 — 0.4.2)
                    # 이름 앞이 날짜 · 숫자 · 구분 기호로 끝나는 줄(값 짝을 이은 줄은 아님), 꼬리표 뒤 이름에 표지 없이 숫자가 붙은 줄
                    # (「명목 80% 와의 차: 이름 -0.5 %p」) — 보조 줄일 수 있습니다
                    "aside": (kind in ("num", "sep") and not chain) or (tag and val is not None and val[5] == "bare")})
    if _bi.any(not o["loose"] for o in out):
        out = [o for o in out if not o["loose"]]
    return out


def _unit_value(val, unit):
    """「k 건 (r %)」 꼴에서 기준 단위가 붙은 괄호 안의 수 — 그런 꼴이 아니면 읽은 값 그대로.

    기준 단위가 % 인 칸에 「이름 = 14 건 (1.4 %)」 로 찍으면 표지 뒤 첫 숫자는 건수 14 이고 비율은 괄호 안에 있습니다.
    읽은 값 바로 뒤에 기준 단위가 없고, 값(과 짧은 단위 낱말) 바로 뒤의 괄호 안에 기준 단위가 붙은 수가 있으면 그 수가 값입니다.
    값과 괄호 사이에 분모를 적은 꼴(「14 건 / 1,000 건 (1.4 %)」 · 「14 건 ÷ 1,000 건 (1.4 %)」)도 같습니다.
    칸마다 켜는 규칙입니다(기준의 `paren_unit`) — 「확신 = 26 (전체 168 시간 가운데)」 처럼 괄호 안의 수가 설명인 칸에도
    켜면 설명의 수를 값으로 읽습니다.
    """
    after = val[2]
    if not unit or after.lstrip().startswith(unit):
        return val
    word = r"[^\s\d()\[\],;:=→·|/÷]{0,6}"                          # 값 뒤의 짧은 단위 낱말(건 · 개 · 명)
    m = _re.match(r"\s*" + word + r"\s*(?:[/÷]\s*[\d,]+(?:\.\d+)?\s*" + word + r"\s*)?\(", after)   # (분모) 여는 괄호
    if not m:
        return val
    close = after.find(")", m.end())
    close = len(after) if close < 0 else close
    inner = after[m.end():close]
    for tok in _NUM_RE.finditer(inner):
        if inner[tok.end():].lstrip().startswith(unit):
            try:
                return (float(tok.group(0).replace(",", "")), tok.group(0), inner[tok.end():] + after[close:]) + val[3:]
            except ValueError:
                continue
    return val


def _value_use(occ, val_lines):
    """이름이 있는 줄들(`_occurrences`) 가운데 채점에 쓸 값 줄.

    같은 이름 줄이 여럿이면 이름 바로 뒤 값 줄만 봅니다. 바로 뒤 값 줄은 「이름 = 값」·「이름: 값」(어느 행이든), 그리고
    단위·「=」 가 붙은 행(val_lines)의 「이름 값」 입니다. 표지도 단위도 없는 행의 「이름 숫자」(「결과 요약: 이름 A, 이름 B 2개 값」
    의 2)는 값 줄이 달리 없을 때만 씁니다. 이름 바로 앞에 다른 낱말이 붙은 줄은 더 긴 이름의 줄일 수 있습니다 — 도우미가 찍는
    「상태별 리뷰 수 : 통과 178 · …」 을 「리뷰 수」 의 값 줄로 읽으면 맞는 출력이 「줄이 2개이고 값이 서로 다릅니다」 가 됩니다.
    이름으로 시작하는 값 줄이 따로 있으면 그런 줄은 쓰지 않고, 그런 줄뿐이면 지금처럼 그 값을 읽습니다.
    비교하려고 덧붙인 줄도 같습니다 — 이름 앞에 꼬리표가 붙은 줄(「(전임 코드) 이름 = 892 건」)과 이름 뒤 괄호에 설명을 넣은 줄
    (「이름(50만 건 전체, 이론) = 17.39 %」)은 맨 이름 「이름 = 값」 줄(이름 바로 뒤 값 줄)이 따로 있으면 쓰지 않습니다. 설명을 이름
    앞에 붙인 「원본 이름 = …」 은 통과하고 이름 뒤 괄호에 넣은 줄만 「줄이 2개이고 값이 서로 다릅니다」 가 되지 않게 합니다.

    강한 값 줄(「이름 = 숫자」 — 이름이 줄 머리에 있고 표지 바로 뒤가 숫자인 줄)이 하나라도 있으면 그 줄끼리만 씁니다(0.4.2).
    원점별 행 · 명목 대비 줄 · 제로샷 비교 줄 · 날짜가 앞에 붙은 줄처럼 같은 이름이 든 보조 줄의 수가 맨 이름 줄과 부딪쳐 맞는 출력이
    「줄이 2개이고 값이 서로 다릅니다」 가 되지 않게 합니다. 옛 print 와 새 print 를 함께 남긴 셀, 전임 코드의 줄은 둘 다 강한 값 줄이라
    지금처럼 부딪침으로 잡힙니다. 강한 값 줄이 없으면 아래 차례(0.4.1 과 같고, 이름 앞에 날짜 · 숫자가 오거나 구분 기호 바로 뒤에 이름이 온
    줄과 꼬리표 뒤 이름에 표지 없이 숫자가 붙은 줄 — "aside" — 은 이름으로 시작하는 줄이 따로 있을 때, 값을 괄호 안에서만 읽은 줄 —
    "paren" — 은 다른 값 줄이 있을 때 쓰지 않습니다)로 고릅니다.
    """
    found = [o for o in occ if o["val"]]
    strong = [o for o in found if o.get("strong")]
    if strong:                                              # 강한 값 줄끼리만 — 보조 줄의 수는 이 항목의 값으로 읽지 않습니다
        return strong
    if _bi.any(not o["pre"] for o in found):               # 이름으로 시작하는 줄(구분 기호 뒤의 이름 포함)이 있으면, 이름 앞에
        found = [o for o in found if not o["pre"]]          # 다른 낱말이 붙은 줄(「상태별 리뷰 수 : 통과 178 …」)은 쓰지 않습니다
    if _bi.any(not o.get("aside") for o in found):         # 이름 앞이 날짜 · 숫자이거나 구분 기호 바로 뒤에 이름이 온 줄도 같습니다
        found = [o for o in found if not o.get("aside")]
    if _bi.any(not o.get("paren") for o in found):         # 값을 괄호 안에서만 읽은 줄(「이름 (제로샷 41.6 % 보다 높음)」)은
        found = [o for o in found if not o.get("paren")]    # 다른 값 줄이 있으면 쓰지 않습니다 — 그런 줄뿐이면 0.4 의 53번대로 읽습니다
    # 비교하려고 덧붙인 줄(꼬리표 · 이름 뒤 괄호 설명)은 맨 이름 「이름 = 값」 줄 — 이름 바로 뒤 값 줄 — 이 따로 있을 때만 뺍니다.
    # 문장 속의 이름이나 다른 줄의 이름(「… (분모 2,000건)」 · 「결론 : … 이름은 9월의 0.933배 …」)이 있다고 괄호 설명 줄을 버리면
    # 「분모(보낸 리뷰 수) = 2,000 건」 처럼 설명을 붙인 맞는 값 줄을 잃습니다
    if _bi.any(o["val"][3] and not o["tag"] and not o["noted"] for o in found):
        found = [o for o in found if not o["tag"] and not o["noted"]]
    direct = [o for o in found if o["val"][3]]
    return ([o for o in direct if o["line"] in val_lines or o["val"][5] != "bare"]
            or [o for o in found if o["line"] in val_lines] or direct or found)


def _text_tails(lines, label, others=()):
    """이름이 있는 줄마다 값 표지(「=」·「:」) 뒤의 글 — 값이 낱말 · 문장인 항목(권 검사)이 읽습니다.

    이름 바로 뒤가 표지인 줄(「이름 = …」 · 「이름 (설명): …」)이 있으면 그 줄만, 없으면 이름이 있는 모든 줄입니다
    (표지가 없는 줄은 이름 뒤 전체). 글은 다른 항목 이름에서 끊지 않습니다 — 내가 적은 문장에 다른 항목의 이름이 나와도 그대로입니다.
    others 는 다른 항목의 이름입니다 — 이 이름을 품은 더 긴 이름(「1위 토픽」 에 대한 「1위 토픽 비율」)의 줄을 가려냅니다.
    이름 앞에 다른 낱말이 붙은 줄(「전임 판단 : …」 — 다른 이름의 줄)과 꼬리표가 붙은 줄(「(전임 코드) 판단 : …」)은
    같은 급의 줄 가운데 이름으로 시작하는 줄이 따로 있으면 쓰지 않습니다(값 칸의 `_value_use` 와 같은 규칙). 「고른 품목 = 헬스장」
    처럼 그런 줄뿐이면 그 줄을 읽습니다 — 표지 없는 줄(「… · 품목 민원 4,000건」)이 있다고 버리지 않습니다.
    """
    whole, rest = [], []
    for o in _occurrences(lines, label, others):
        cut = _mark_at(o["after"])
        if cut >= 0 and _bare_gap(o["after"][:cut]):
            whole.append((o["after"][cut + 1:], o))
        else:
            rest.append((o["after"][cut + 1:] if cut >= 0 else o["after"], o))
    pick = whole or rest
    for mark in ("pre", "tag"):
        if _bi.any(not o[mark] for _, o in pick):
            pick = [(t, o) for t, o in pick if not o[mark]]
    return [t for t, _ in pick]


def _value_lines(txt, units):
    """값 줄 — 단위가 붙은 행, 또는 「이름 = 값」 꼴의 행.

    이름은 출력의 모든 행에서 찾습니다(「이름: 값」 꼴 포함). 이름 바로 뒤 값 줄이 하나도 없을 때만 이 행들의 값을
    먼저 씁니다 — 이 행들에서 값이 읽혔다고 「이름: 값」 줄을 보지 않으면, 틀린 값을 「:」 로 찍고 맞는 값을 「=」 로 찍은
    출력의 충돌을 놓칩니다. 이름이 있는 행이 아예 없으면 숫자를 읽지 않고 ❌ 로 알립니다(아래 `judge`).
    """
    return [line for line in (txt or "").splitlines()
            if line.strip() and ("=" in line or _bi.any(unit and unit in line for unit in units))]


def _short_digits(raw, value, lo, hi):
    """찍힌 자릿수가 모자라 범위 밖으로 읽혔으면 그 자릿수, 아니면 None.

    「0.10」 · 「205」 처럼 찍힌 자리로 반올림하면 기준 범위와 겹치는데 값 자체는 범위 밖이면, 계산이 아니라
    출력 자릿수를 짚어야 학생이 엉뚱한 원인을 찾지 않습니다. 소수 한 자리(「0.1」)는 자릿수를 줄인 값인지
    조회표 끝의 p 값인지 가를 수 없어 보지 않습니다.

    겹침은 경계를 빼고 봅니다 — 반올림 구간의 끝이 기준 범위의 끝에 닿기만 한 값은 자릿수 문제가 아닙니다.
    개수처럼 정수인 칸은 기준 범위를 n±0.5 로 잡는데, 바로 옆 정수(n−1 · n+1)의 반올림 구간이 그 끝에 딱 닿습니다.
    닿은 것까지 겹침으로 치면 틀린 개수(167 · 169)가 「자릿수 모자람」 이 되고 같은 이름 줄의 충돌에서도 빠집니다.
    """
    raw = (raw or "").replace(",", "").lstrip("+-")
    if not raw or "e" in raw.lower() or lo <= value <= hi:
        return None
    places = len(raw.split(".", 1)[1]) if "." in raw else 0
    if places == 1:
        return None
    half = 0.5 * 10 ** (-places)
    edge = half * 1e-6                      # 부동소수 오차로 「닿음」 이 「겹침」 으로 읽히지 않게 두는 여유
    return places if value - half < hi - edge and value + half > lo + edge else None


# ───────────────────────── 문구 · 축 이름 ─────────────────────────
_NEG_TAIL = ("하지못", "하지않", "하지는못", "하지는않", "되지않", "되지못", "할수없", "못", "아님", "아니",
             "안되", "안됐", "안돼", "안됨", "안됩", "안함", "안했", "불가", "에실패", "실패",
             "에이르지못", "에이르지않", "까지는못", "까지못")
_NEG_MARK = r"(?:False|false|FALSE|No|no|NO|N|X|x|×|✗|✘|❌)(?![A-Za-z])"
_YES_MARK = r"[=:]?(?:True|true|TRUE|Yes|yes|YES|O|예|네|있음|함)(?![A-Za-z])"


def _flat(txt):
    """빈칸을 걷고 겉보기가 같은 대시·가운뎃점을 맞춘 글자"""
    return "".join(_GLYPH.get(ch, ch) for ch in _re.sub(r"\s+", "", txt or ""))


def _word_in(row, word):
    """빈칸을 걷은 줄 `row` 에 낱말이 **긍정으로** 있는가.

    「비기각」·「미기각」·「기각하지 못함」·「기각 안 됨」·「기각에 이르지 못함」·「기각 X」·「기각 = False」 는
    「기각」 을 글자로 품지만 뜻은 반대입니다. 「기각 여부」 뒤에 답(True·예)이 없으면 무엇도 말하지 않은 것이고,
    「비정상 쪽 증거 없음」 도 반대입니다. 그 꼴을 지운 뒤에도 낱말이 남아야 채운 것으로 봅니다
    (「비정상 쪽」 안의 「정상 쪽」 도 같은 방식으로 지웁니다). 금지 문구(forbid)는 이 규칙을 쓰지 않습니다 —
    머리말·옛 이름에 낱말용 부정 꼴을 대면 문구째 지워집니다(`_forbid_in`).
    """
    w = _flat(word)
    if not w:
        return True
    e = _re.escape(w)
    neg = "|".join((
        "(?:비|미|불)" + e,                                            # 비기각 · 미기각 · 불기각
        e + "(?:" + "|".join(_NEG_TAIL) + ")",                          # 기각하지 못함 · 기각 안 됨 · 기각에 이르지 못함
        e + r"(?:여부)?[=:]?" + _NEG_MARK,                              # 기각 X · 기각 = False · 기각 여부 = False
        e + r"의?증거(?:는|가|도)?(?:없|부족|불충분|아님|아니|못)",           # 비정상 쪽 증거 없음
        e + r"여부(?!" + _YES_MARK + ")",                               # 「기각 여부」 뒤에 답이 없다
    ))
    return w in _re.sub(neg, "", _flat(row))


#  금지 문구(forbid) 뒤에서 그 문구를 서술로 부정한 꼴 — 「…넣어도 되지 않습니다」·「운영 가능하지 않다」·「…이 아니다」
_FORBID_NOT = (r"(?:은|는|이|가|도|고|라고|다고)?(?:(?:하|되)지(?:는|도)?(?:않|못)|할수(?:는|도)?없|안(?:되|된|됩|돼|됨|됐)"
               r"|아니|아님|아닙)")


#  줄 머리말(「판단 :」) 바로 뒤에서 그 줄을 지웠다고 알리는 말 — 「판단 : 행은 삭제했습니다」·「판단 : 줄은 지웠습니다」
_FORBID_GONE = (r"(?:행|줄|문장|결론)(?:은|을|도|이|만)?(?:모두|전부|이제)?"
                r"(?:지웠|지움|지우|지운|삭제|뺐|뺌|빼고|뺀|없앴|없앰|없애|제거|남기지않|출력하지않|찍지않)")


def _forbid_in(row, phrase):
    """빈칸을 걷은 줄 `row` 에 금지 문구가 남아 있는가 — 함정 미션의 `forbid`.

    「판단 :」 처럼 「:」 로 끝나는 문구는 줄 머리말입니다. 줄의 맨 앞이나 쉼표·화살표 같은 구분 기호 바로 뒤에 올 때만
    남은 줄입니다 — 「누수 여부 판단: …」 처럼 다른 말 뒤에 이어 쓴 것, 따옴표 안에 옮겨 적은 것은 머리말이 아닙니다.
    머리말 뒤에 오는 말은 가리지 않습니다. 「판단 : 불가」·「판단 : X」·「판단 : 아니다」 도 판단 줄이고, 전각 「：」 도
    같은 표지입니다. 하나만 가립니다 — 머리말 바로 뒤가 그 줄을 지웠다는 말(「(판단 : 행은 삭제했습니다)」)이면 PROMPT 의
    「판단 : 행은 지워」 를 되받아 알린 글이라 남은 줄이 아닙니다(`_FORBID_GONE`).

    그 밖의 문구(옛 이름 · 전임 담당자의 결론)는 글자로 대조합니다. 문구 바로 뒤에서 서술로 부정한 인용(「…넣어도 되지
    않습니다」)과 답 없는 「… 여부」 만 남은 줄이 아닙니다. 낱말용 부정 꼴(`_word_in` 의 비·미·불 접두, 못·불가·실패,
    X·No·N)은 쓰지 않습니다 — 「계절 나이브 대비 제로샷 평균 MASE」 의 「비」, 「제로샷 평균 MASE: N/A」 의 「N」,
    「판단 : 불가」 의 「불가」 가 문구를 지워 남은 줄이 통과하던 자리입니다.
    """
    w = _flat(phrase).replace("：", ":")
    if not w:
        return False
    row = _re.sub(r"[*`]", "", row or "").replace("：", ":")
    e = _re.escape(w)
    if w.endswith(":") and len(w) > 1:
        # 「(판단 : 행은 삭제했습니다)」 — 머리말 바로 뒤가 그 줄을 지웠다는 말이면 남은 줄이 아니라 지웠다고 알리는 글입니다
        return _bi.bool(_re.search(r"(?:^|[^0-9A-Za-z가-힣_'\"「『“‘])" + e + "(?!" + _FORBID_GONE + ")", row))
    return w in _re.sub(e + "(?:" + _FORBID_NOT + "|여부(?!" + _YES_MARK + "))", "", row)


def _axes(fig):
    """Figure · Axes 어느 쪽을 받아도 축 목록으로 폅니다"""
    if fig is None:
        return []
    axs = getattr(fig, "axes", None)
    if isinstance(axs, list):
        return axs
    return [fig] if hasattr(fig, "get_xlabel") else []


def _sup_label(ax, which):
    """그림 수준 축 이름(fig.supxlabel · fig.supylabel)이 있는가 — 패널이 여럿인 그림에서 축마다 이름을 다는 대신 씁니다"""
    fig, seen = getattr(ax, "figure", None), set()
    while fig is not None and id(fig) not in seen:   # SubFigure 면 바깥 그림까지 올라갑니다
        seen.add(id(fig))
        text = getattr(fig, "_sup" + which + "label", None)
        try:
            if text is not None and text.get_text().strip():
                return True
        except Exception:
            pass
        fig = getattr(fig, "figure", None)
    return False


def _mate_label(ax, which):
    """가로축(which="x")·세로축("y")을 나눠 쓰는 다른 축에 그 이름이 있는가.

    `ax.twinx()` 로 겹친 두 축은 가로축 이름을 한쪽에, 세로축 이름을 다른 쪽에 다는 일이 흔하고, `sharex=True` 로 쌓은
    그림은 맨 아래 축에만 가로축 이름을 답니다. 화면에는 두 이름이 다 보이므로 짝의 이름도 이 축의 이름으로 봅니다.
    나눠 쓰지 않는 옆 칸의 이름(`subplots(1, 2)` 의 한 칸에 가로축 이름, 다른 칸에 세로축 이름)은 보지 않습니다.
    """
    mates = []
    for find in (lambda: getattr(ax, "get_shared_" + which + "_axes")().get_siblings(ax),
                 lambda: ax._twinned_axes.get_siblings(ax)):
        try:
            mates += [a for a in find() if a is not ax]
        except Exception:
            pass
    for a in mates:
        try:
            if getattr(a, "get_" + which + "label")().strip():
                return True
        except Exception:
            pass
    return False


def _plot_gaps(ax, spec):
    """축 하나를 기준과 대조해 어긋난 곳을 문장 목록으로 돌려줍니다 (빈 목록이면 통과).

    그래프 종류(막대·가로 막대·선·점, pandas·seaborn 으로 그린 것)는 가리지 않고 가로축·세로축 이름만 봅니다.
    `fig.supxlabel`·`fig.supylabel` 로 그림 전체에 단 이름, 축을 나눠 쓰는 짝(twinx · sharex)에 단 이름도 축 이름입니다.
    기준이 `가로축라벨` 이면(세로 눈금에 막대 이름이 붙는 가로 막대 그림) 가로축 이름만 봅니다.
    """
    x_name = ax.get_xlabel().strip() or _sup_label(ax, "x") or _mate_label(ax, "x")
    y_name = ax.get_ylabel().strip() or _sup_label(ax, "y") or _mate_label(ax, "y")
    if spec.get("축라벨") and not (x_name and y_name):
        return ["가로축·세로축 이름 가운데 빈 것이 있습니다"]
    if spec.get("가로축라벨") and not x_name:
        return ["가로축 이름이 비어 있습니다"]
    return []


def _order_of(tail):
    """「공식 차수」 줄 꼬리의 비계절 차수 — ((p, d, q), 이름 바로 뒤 값 줄인가) · 못 읽으면 None.

    값 표지(괄호 밖 첫 「=」·「:」) 뒤에서 **정수 3개가 든 첫 괄호 묶음**을 먼저 찾습니다(계절 차수까지 한 묶음에 적은
    7개도). 「= 1위 조합 (1,0,1)(0,1,0,24)」·「= AIC 1위 (1, 0, 1) x (0, 1, 0, 24)」 처럼 괄호 앞에 숫자가 든 말이 와도
    괄호 안을 읽습니다. 그런 묶음이 없으면 표지 뒤 앞 숫자 3개입니다 — 「= 1, 0, 1」. (1,0,1) · (1.0, 0, 1.0) ·
    SARIMA(1,0,1)(…) 를 같게 읽고, 이름 뒤 괄호 설명(「(p=1, d=0, q=1 로 적합)」)은 건너뜁니다.
    진행 막대 줄(「17%|█▋ | 1/6 [00:00<…」)은 차수 줄이 아닙니다.
    """
    t, depth, cut = _unwrap_np(tail), 0, -1
    if _re.search(r"\d+%\|", t):
        return None
    for i, ch in enumerate(t):
        if ch in "([":
            depth += 1
        elif ch in ")]":
            depth = _bi.max(0, depth - 1)
        elif ch in "=:：＝" and depth == 0:
            cut = i
            break
    rest = t[cut + 1:] if cut >= 0 else t
    direct = cut >= 0 and _bare_gap(t[:cut])
    num = r"(?<![A-Za-z_\d.])-?\d+(?:\.\d+)?"
    picks = [_re.findall(num, a or b) for a, b in _re.findall(r"\(([^()\[\]]*)\)|\[([^()\[\]]*)\]", rest)]
    nums = next((p for p in picks if len(p) in (3, 7)), None) or _re.findall(num, rest)
    nums = nums[:3]
    if len(nums) < 3 or _bi.any(float(x) != int(float(x)) for x in nums):
        return None
    return tuple(int(float(x)) for x in nums), direct


# ───────────────────────── 채점 상태 · 그림 ─────────────────────────
_FIGS = []                                 # 「PROMPT 셀」이 만든 그림을 모으는 자리
_STATE = {"tag": None, "buf": None, "out": None, "ids0": set(), "src": "", "prompt": None, "pub": None,
          "lost": None, "imgs": 0}
_TRY = {}                                  # 미션마다 ❌ 가 몇 번째인지 — 2단 힌트·기준 범위·빠진 낱말이 이 수를 봅니다
_orig_figure = getattr(_plt.figure, "_orig", _plt.figure)   # 이 셀을 다시 실행해도 겹겹이 싸이지 않게


def _figure(*a, _orig=_orig_figure, _figs=_FIGS, **k):
    # 학생 코드가 그림을 만들 때 불립니다 — 채점 코드의 이름을 기본값으로 묶어 둡니다(학생 셀이 같은 이름을 만들어도 그대로)
    f = _orig(*a, **k)
    _figs.append(f)
    return f


_figure._orig = _orig_figure
_plt.figure = _figure                      # subplots · pandas · seaborn 도 이 경로를 거칩니다


def _get(name):
    """이름이 살아 있으면 그 값, 없으면 None — 아직 만들지 않은 이름에도 채점이 죽지 않게"""
    return globals().get(name)


def _device_spec(spec):
    """실행 경로(CPU · GPU)마다 기준이 다른 미션 — 기준의 `by_device` 에서 지금 DEVICE 의 한 벌을 덮어씁니다.

    준비 셀이 DEVICE 를 다시 정해도 채점 시점의 값을 따릅니다("cuda" 로 시작하면 GPU 기준, 그 밖은 CPU 기준).
    """
    per = (spec or {}).get("by_device")
    if not per:
        return spec
    dev = "cuda" if str(_get("DEVICE") or "cpu").startswith("cuda") else "cpu"
    return {**{k: v for k, v in spec.items() if k != "by_device"}, **dict(per.get(dev, {}))}


def _spec_of(key):
    """이 미션의 채점 기준 — 실행 경로별 기준을 맞추고(`_device_spec`), 권 설정의 `spec` 으로 이 세션의 기준을 다시 계산합니다"""
    spec = _device_spec(_ANS.get(key))
    hook = _BOOK.get("spec")
    if hook is not None and spec is not None:
        try:
            spec = hook(key, spec) or spec      # 이 세션에서 다시 계산한 기준(권 설정)
        except Exception:
            pass
    return spec


def _takes_ctx(fn):
    """권 검사 함수가 셋째 인자(출력 글 · 그림 · 실행 상태)를 받는 꼴인가 — `fn(key, spec, ctx)`. 옛 꼴은 `fn(key, spec)` 입니다"""
    code = getattr(fn, "__code__", None)
    if code is None:
        return False
    n = code.co_argcount - (1 if getattr(fn, "__self__", None) is not None else 0)
    return n >= 3 or _bi.bool(code.co_flags & 0x04)


def _ai_lines(src):
    """셀 원문에서 「여기부터 AI 코드」 표시선 아래의 코드 줄 — 주석은 뗍니다(표시선이 없으면 빈 목록).

    셀이 표시선 줄에서 끝나면(줄바꿈 없이 — 코드를 받기 전의 PROMPT 셀) 표시선 줄의 나머지 글자(「 ---」)를 코드 줄로 돌려주지 않습니다.
    """
    src = str(src or "")
    if "여기부터 AI 코드" not in src:
        return []
    rest = src.split("여기부터 AI 코드", 1)[1]
    body = rest.split("\n", 1)[1] if "\n" in rest else ""
    return [ln.split("#", 1)[0] for ln in body.splitlines()]


def _open_figs():
    """지금 열려 있는 그림들 — 래퍼를 거치지 않고 읽습니다"""
    try:
        return [_plt._pylab_helpers.Gcf.figs[n].canvas.figure for n in _plt.get_fignums()]
    except Exception:
        return []


def _num_txt(v, fmt=",.6g"):
    """채점 문장에 찍을 수 — 음수 부호는 권 설정(`minus`)을 따릅니다(본문이 − U+2212 를 쓰는 권)"""
    return format(v, fmt).replace("-", _BOOK.get("minus", "-"))


def _val_txt(val):
    """읽은 값(`_read_value` 결과)을 채점 문장에 찍을 글자 — 찍힌 자리까지 보여 줍니다.

    유효숫자 6자리로만 찍으면 학생이 출력한 「25,209.34」 가 「25,209.3」 으로, 「1,052,176.7」 이 「1.05218e+06」 으로 보여
    자리가 잘린 값으로 읽힙니다. 6자리로 줄였을 때 찍힌 자리의 숫자가 사라지면 찍힌 소수 자릿수대로 찍습니다.
    끝의 0 은 뗍니다(91.0 → 91). 소수 7자리 넘게 찍힌 값(반올림하지 않은 0.2812250064155164)은 지금처럼 유효숫자 6자리이고
    (지수 표기가 되는 큰 수만 소수 6자리), 지수 표기로 찍힌 값은 지수 표기입니다.
    """
    v, raw = val[0], str(val[1] or "")
    short = _num_txt(v)
    places = len(raw.split(".", 1)[1]) if "." in raw else 0
    if "e" in raw.lower() or v != v or v in (float("inf"), float("-inf")) or (places > 6 and "e" not in short):
        return short
    places = _bi.min(places, 6)
    full = format(v, ",." + str(places) + "f")
    full = full.rstrip("0").rstrip(".") if "." in full else full
    try:
        same = float(format(v, ",.6g").replace(",", "")) == float(full.replace(",", ""))
    except ValueError:
        return short
    return short if same else full.replace("-", _BOOK.get("minus", "-"))


_ANSI_RE = _re.compile(r"\x1b\[[0-9;?]*[ -/]*[@-~]")       # 굵게·색을 켜고 끄는 글자 — 화면에는 보이지 않습니다


def _clean_out(text, prompt=""):
    """채점이 읽을 출력 — 화면에 보이지 않는 글자를 지우고 PROMPT 를 그대로 찍은 글을 뺍니다.

    굵게 찍으려고 넣은 ANSI 코드(ESC[1m · ESC[0m)는 「[」 가 있어 열린 괄호로 읽히고, `print(PROMPT)` 로 찍은 지시문의
    「"이름 = 값" 한 행으로 출력하고, Q_ZS 의 0.5 분위수로 …」 는 이름 줄이 되어 0.5 가 값으로 읽힙니다.
    PROMPT 글은 값 줄이 아닙니다.
    """
    txt = _ANSI_RE.sub("", text or "")
    echo = str(prompt or "").strip()
    if len(echo) >= _PROMPT_MIN and echo in txt:
        txt = txt.replace(echo, "\n")
    return txt


# ───────────────────────── PROMPT 칸 ─────────────────────────
def _one_line(e, n=120):
    """예외를 한 줄로 — 트레이스백 대신 학생이 읽을 문장 하나만 남깁니다.

    문법 오류는 임시 파일 이름 대신 셀의 몇째 줄인지 적고, 예외의 문장을 읽다가 다시 오류가 나도
    채점 표가 닫히도록 그 자리는 안내 문구로 채웁니다.
    """
    try:
        if isinstance(e, SyntaxError) and getattr(e, "msg", None):
            txt = str(e.msg) + (f" (셀 {e.lineno}번째 줄)" if getattr(e, "lineno", None) else "")
        else:
            txt = str(e)
    except Exception:
        txt = "(오류 문장을 읽지 못했습니다)"
    txt = txt.replace("\n", " ").strip()
    return (txt[:n] + "…") if len(txt) > n else txt


def _prompt_in_cell(src):
    """셀 원문의 PROMPT 따옴표 안 글자 — 셀을 실행하기 전에 읽어 둡니다.

    이름 공간의 PROMPT 를 읽으면 PROMPT 블록을 지운 셀이 앞 미션에서 채운 PROMPT 로 통과하고, 문법 오류로
    셀이 돌지 않은 날은 채운 PROMPT 도 앞 값으로 보입니다. 셀에 PROMPT 대입이 없으면 "" 이고, 따옴표 꼴이 아니라
    읽지 못하면 None 입니다. 셀이 돌았으면 따옴표 안 글자 대신 실행된 PROMPT 값을 씁니다(`_post`).
    """
    src = str(src or "")
    if not _re.search(r"(?m)^[ \t]*PROMPT[ \t]*=(?!=)", src):
        return ""
    lit = _re.search(r"^[ \t]*PROMPT[ \t]*=[ \t]*[rRuU]?(\"\"\"|'''|\"|')(.*?)\1", src, _re.M | _re.S)
    return lit.group(2) if lit else None


def _prompt_now(given=None):
    """채점할 PROMPT 글자 — 셀 원문에서 읽은 것(given)이 있으면 그것, 없으면 이름 공간의 PROMPT"""
    if given is not None:
        return str(given)
    try:
        return str(_ip.user_ns.get("PROMPT", _get("PROMPT")) or "")
    except Exception:
        return str(_get("PROMPT") or "")


def _prompt1(n=70, given=None):
    """지금 셀의 지시 첫 줄 — 튜터에 보낼 질문에 그대로 싣습니다"""
    head_lines = [l.strip() for l in _prompt_now(given).split("\n") if l.strip()]
    return (head_lines[0][:n] + "…") if head_lines else "____"


_PROMPT_MIN = 30
_PROMPT_PLACEHOLDERS = ("", "목표", "쓸 변수", "출력", "목표 —", "쓸 변수 —", "출력 —", "TODO",
                        "여기에 내 말로 적습니다")


def _strip(txt):
    """빈칸·줄표·가운뎃점을 떼어 낸 알맹이 — 자리표시자 대조는 이 꼴로 합니다"""
    return _re.sub(r"[\s—·:,\-]+", "", str(txt or ""))


_PLACEHOLDERS_STRIPPED = {_strip(t) for t in _PROMPT_PLACEHOLDERS}


def _prompt_missing(given=None):
    """지시 입력란이 아직 내 말로 채워지지 않았으면 사유 한 문장, 채워졌으면 None"""
    txt = _prompt_now(given).strip()
    stripped = _strip(txt)                          # 길이보다 먼저 본다 — 견본을 그대로 둔 자리가 있다
    if not stripped:
        return "PROMPT 입력란이 비어 있습니다"
    if stripped in _PLACEHOLDERS_STRIPPED:
        return "PROMPT 입력란이 처음 적혀 있던 견본 그대로입니다"
    if len(txt) < _PROMPT_MIN:
        return f"PROMPT 입력란이 {len(txt)}자뿐입니다 (최소 {_PROMPT_MIN}자)"
    return None


def _reveal_show(txt):
    """완성본 코드와 해설 — **평문 그대로** 찍는다.

    LMS 렌더러 계약: 「🔓 정답이 열렸습니다 …」 줄부터 채점 표를 닫는 `─` 구분선 줄까지를
    출력 영역이 「🔓 완성본 풀이 보기」 상자로 접는다. 접힘의 표지가 이 **평문 마커**라
    HTML `details` 로 싣지 않는다 — 마커도 이 함수도 바꾸지 않는다.
    """
    print(txt)


# ───────────────────────── 오류 · 흔한 실수 ─────────────────────────
def _error_hint(key, error):
    """오류에 맞는 (힌트, 물을 것).

    문법 오류(SyntaxError·IndentationError)는 셀이 한 줄도 돌지 않은 것이라 붙여 넣은 글자를 짚습니다.
    노트북에 있는 이름을 잘못 옮겨 적은 NameError 면 그 이름을 짚고, 권 설정의 `error_hint(key, error)` 가 (힌트, 물을 것)을
    돌려주면 그것, 권의 힌트에 오류 종류별 힌트(`error_kinds`)가 있으면 그것, 없으면 미션 공통 힌트입니다
    (준비 셀·설치 셀이 만든 이름과 다시 실행할 셀).
    """
    kind, msg = type(error).__name__, _one_line(error, 400)
    tbl = _HINT.get(key, {})
    if isinstance(error, SyntaxError):
        line_no = getattr(error, "lineno", None)
        at = f"셀 {line_no}번째 줄" if line_no else "오류 문장이 가리키는 줄"
        head = "문법 오류라 이 셀의 코드가 한 줄도 실행되지 않았습니다. "
        if "```" in (_STATE.get("src") or ""):
            return (head + "AI 답에서 코드를 옮길 때 「```python」·「```」 같은 코드 블록 표시 행(펜스 줄)이 함께 들어왔습니다."
                    " 표시 행을 지우고 그 안쪽 코드만 남긴 뒤 다시 실행하세요",
                    "붙여 넣은 코드의 첫 행과 끝 행에 파이썬이 아닌 행이 있는지")
        if isinstance(error, (IndentationError, TabError)):
            return (head + f"들여쓰기가 어긋났습니다 — {at}과 바로 위 행의 들여쓰기 칸 수를 비교하고,"
                    " 같은 블록의 행은 같은 칸 수로 맞추세요",
                    "이 행의 들여쓰기가 위 행과 어떻게 어긋났는지")
        return (head + f"{at}에서 괄호·따옴표가 닫혔는지, 코드가 아닌 설명 문장이 섞였는지,"
                ' PROMPT 에 적을 문장을 PROMPT = """ """ 따옴표 밖에 두지 않았는지 보세요',
                "이 문법 오류가 어느 행에서 왜 났는지")
    if kind == "NameError":                # 노트북에 있는 이름을 잘못 옮겨 적은 경우(FIG_W → FGW)는 그 이름을 짚습니다
        import difflib
        bad = _re.search(r"name '(\w+)'", msg)
        near = bad and difflib.get_close_matches(
            bad.group(1), [n for n in globals() if not n.startswith("_") and n != "FIGW"], n=1, cutoff=0.7)
        if near:
            return (f"「{bad.group(1)}」 이름은 이 노트북에 없습니다. 비슷한 이름은 「{near[0]}」 입니다."
                    f" 잘못 옮겨 적었다면 코드에서 「{bad.group(1)}」 를 「{near[0]}」 로 고쳐 다시 실행하세요",
                    "이 오류가 어느 행에서 왜 났는지")
    hook = _BOOK.get("error_hint")          # 권 설정 — 셀 위쪽에 내가 적은 값의 꼴이 어긋나 난 오류처럼 권만 아는 까닭
    if hook is not None:
        try:
            own = hook(key, error)
        except Exception:
            own = None
        if own and own[0]:
            return own[0], (own[1] if len(own) > 1 and own[1] else "이 오류가 어느 행에서 왜 났는지")
    for rule in tbl.get("error_kinds", ()):
        if _bi.any(kind == k and _re.search(pat, msg) for k, pat in rule.get("match", ())):
            return rule.get("hint", ""), rule.get("ask") or "이 오류가 어느 행에서 왜 났는지"
    hint, question = tuple(tbl.get("error", ("", "이 오류가 어느 행에서 왜 났는지")))[:2]
    return hint, question


def _note_for(rules, values, tail_text):
    """값이 흔한 실수 하나의 값과 같으면 그 원인 문장 — 규칙은 (하한, 상한, 문장) · ("nan", 문장) (`_CAUSE` 블롭)"""
    for rule in rules or ():
        if rule[0] == "nan":
            if "nan" in (tail_text or "").lower():
                return rule[1]
        elif _bi.any(float(rule[0]) <= v <= float(rule[1]) for v in values):
            return rule[2]
    return ""


def _stale_code(rule, src, lo, hi):
    """값은 기준 범위 안인데 그 값을 코드에 손으로 적었고 shift 인자는 전임 코드의 틀린 값뿐이면 그 사실 한 문장.

    함정 미션에서 전임 코드의 `shift(24)` 를 그대로 둔 채 기준 범위의 숫자를 코드에 적어 출력하면 값만 맞고
    고친 곳이 없습니다. 그 꼴만 거릅니다(`_CAUSE["code"]` 규칙이 있는 미션만).
    """
    if not rule or not src:
        return ""
    body = src.split("여기부터 AI 코드", 1)[-1]           # PROMPT 안의 말은 보지 않습니다
    body = "\n".join(l.split("#", 1)[0] for l in body.splitlines())
    args = [_re.sub(r"\s+", "", a) for a in _re.findall(r"\.shift\(([^()]*)\)", body)]
    if not (args and _bi.all(a in rule[0] for a in args)):
        return ""
    if not _bi.any(lo <= v <= hi for v in _nums(body)):
        return ""
    return rule[1]


def _ask_line(form, name, prompt, question):
    """튜터에 보낼 질문 한 줄 — 권의 `_ASK_FORM` 틀에 채웁니다"""
    tmpl = _ASK_FORM.get(form) or _ASK_FORM.get("full") or "{name} — {ask}"
    return "  " + tmpl.format(name=name, prompt=_prompt1(given=prompt), ask=question)


_FORM_HINT = (                              # 이름·숫자를 못 찾은 ❌ — 값 계산이 아니라 출력 꼴을 짚습니다
    "값 계산보다 출력 꼴이 먼저입니다 — 이름과 값을 「이름 = 숫자」 꼴로 한 줄에 찍고, 이름은 「만들 것」 의 굵은 이름과"
    " 글자까지 같게 적도록 PROMPT 의 출력 형식에 적으세요. 표(DataFrame)·딕셔너리·배열로 띄웠다면 항목마다 print 한 줄로 바꿉니다",
    "이 값을 출력하는 행이 이름과 값을 어떤 꼴로 찍는가")
_DIGITS_HINT = (                            # 반올림하면 기준 범위에 닿는 값 — 계산보다 자릿수를 짚습니다
    "값은 맞았을 수 있습니다 — 출력 형식의 자릿수(:.2f 같은 서식)가 「만들 것」 보다 적어 반올림된 값이 찍혔습니다."
    " PROMPT 의 출력 형식에 「만들 것」 의 소수점 아래 자릿수를 적고 다시 실행하세요",
    "이 값을 소수점 아래 몇 자리로 출력하도록 코드가 정했는가")
_DUP_HINT = (                               # 같은 이름 줄이 여럿이고 값이 다르면 — 어느 줄이 최종 답인지 모릅니다
    "같은 이름으로 값을 두 번 이상 찍었습니다 — 전임 코드나 시험 삼아 찍은 옛 print 를 지우고, 최종 값 한 줄만 남기도록"
    " PROMPT 에 적으세요",                  # 함정 미션의 힌트(전임 담당자 코드가 있는 미션)
    "같은 이름을 찍는 print 가 코드 어디 어디에 있는가")
_DUP_HINT_PLAIN = (                         # 함정이 아닌 미션 — 비교 · 중간 결과를 출력하며 같은 이름을 쓴 경우를 짚습니다(0.4.2, 분야 낱말 없이)
    "비교 결과나 중간 결과를 출력할 때도 이 이름을 쓰면, 함께 출력한 수도 이 항목의 값으로 읽습니다. 이름은 최종 값을 출력할 때만"
    " 쓰도록 PROMPT 에 적으세요",
    _DUP_HINT[1])


def _hint_key(key, kind, label):
    """❌ 의 종류에 맞는 힌트 열쇠 — 권의 힌트(`_HINT`)에 그 종류의 열쇠가 있으면 그것, 없으면 공통 힌트"""
    own = _HINT.get(key, {})
    pick = {"form": ("이름_형식", "label"), "dup": ("줄_중복", "dup"), "digits": ("round",), "unit": ("unit",)}
    for cand in pick.get(kind, ()):
        if cand in own:
            return cand
    return {"form": "_form", "dup": "_dup", "digits": "_digits"}.get(kind, label)


def _is_trap(key):
    """함정 미션인가 — 빈 PROMPT 에도 잠그지 않고 표시선 아래 전임 담당자 코드를 진단하는 미션(권 토막의 `_blocks`)"""
    try:
        return not _blocks(key, "PROMPT 입력란이 비어 있습니다")
    except Exception:
        return False


def _hint_parts(tiers):
    """권의 힌트 한 칸 → (힌트 단 목록, 튜터에 물을 것) — 원소가 둘이면 (힌트, 물을 것), 셋 이상이면 마지막이 물을 것"""
    tiers = (tiers,) if isinstance(tiers, str) else tuple(tiers or ())
    steps = tiers[:-1] if len(tiers) > 2 else tiers[:1]
    return [t for t in steps if t], (tiers[-1] if len(tiers) > 1 else "")


# ───────────────────────── 채점 표 ─────────────────────────
def judge(key, text="", figs=(), error=None, prompt=None, shown=None):
    """출력 텍스트와 그림을 기준과 대조해 채점 표를 찍습니다 — 통과하면 완성본 풀이를 엽니다.

    prompt 는 셀 원문에서 읽은 PROMPT(없으면 이름 공간의 PROMPT), shown 은 그림이 화면에 떴는지
    (True·False, 알 수 없으면 None — inline 백엔드에서만 셉니다)입니다.
    """
    spec = _spec_of(key)                   # 실행 경로별 기준(by_device) · 이 세션에서 다시 계산한 기준(권 설정)
    if spec is None:                       # 기준이 아직 없는 미션은 조용히 넘어갑니다
        return None
    head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
    failed, passed = [], True
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과")
    if spec.get("path_note"):              # 실행 경로(CPU · GPU)마다 기준이 다른 미션 — 어느 기준으로 채점하는지 먼저 알립니다
        print(f"ⓘ 채점 기준: {spec['path_note']}")
    reason = _prompt_missing(prompt)                  # 값·그래프보다 먼저 — 내 말로 시킨 지시가 있는가
    _locked = _bi.bool(reason)
    if _blocks(key, reason):                       # 권 토막의 `_blocks` — 함정 미션은 빈 PROMPT 에도 진단합니다
        print(f"❌ {reason} — 짧게 시켜도 됩니다. 다만 무엇을 어떤 단위로 출력할지까지 적으면 한 번에 통과할 확률이 높습니다")
        print("   PROMPT 입력란을 채우기 전에는 값·그래프 채점도, 완성본 풀이도 열지 않습니다")
        print('   AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적었는지 확인하세요')
        print("─" * 54)
        return False
    elif reason:                                   # 함정 미션은 PROMPT 가 비어도 표시선 아래 코드를 진단합니다(지시서 §9-10 · ⓘ 한 줄)
        print(f"ⓘ {reason} — 아래 채점 결과는 전임 담당자 코드의 진단입니다. 완성본 풀이는 PROMPT 를 채운 뒤에 열립니다")
        print("   (표시선 아래를 이미 고쳤다면 아래는 AI 코드 표시선 아래 코드의 진단입니다)"
              ' 고칠 지시는 AI 튜터 대화창이 아니라 이 셀의 PROMPT = """ """ 따옴표 안에 적습니다')
    form = _ASK.get(key, "full")
    if error is not None:                   # 오류가 나면 값·그래프 채점은 뜻이 없습니다
        kind = type(error).__name__                  # 괄호가 안 닫힌 셀은 3.13 에서 _IncompleteInputError 입니다
        kind = "SyntaxError" if isinstance(error, SyntaxError) and kind.startswith("_") else kind
        msg = _one_line(error)
        print(f"❌ 실행 오류 — {kind}" + (f": {msg}" if msg else ""))
        hint, question = _error_hint(key, error)          # 비슷한 이름이 있으면 그 이름, 없으면 미션 공통 힌트
        if hint:
            print(f"   힌트 : {hint}")
        print(_ASK_HEAD.get(form, ""))
        print(_ask_line(form, "실행 오류", prompt, question))
        print("─" * 54)
        return False
    txt = _clean_out(text, _prompt_now(prompt))
    trap = _is_trap(key)
    cell_src = str(_STATE.get("src") or "")
    if ("여기부터 AI 코드" in cell_src and not _bi.any(l.strip() for l in _ai_lines(cell_src))
            and not txt.strip() and not _bi.any(_axes(f) for f in figs)):
        # PROMPT 만 적고 ▶ 한 셀 — 틀린 것이 아니라 아직 코드를 받지 않은 것입니다. 값 · 그래프 ❌ 와 힌트를 늘어놓지 않고
        # 다음 할 일 한 줄만 알립니다(❌ 횟수에 넣지 않습니다)
        print("ⓘ 출력이 없습니다 — 「여기부터 AI 코드」 표시선 아래에 코드가 없습니다. "
              + ("전임 담당자 코드를 지웠다면 실행 취소로 되돌린 뒤, " if trap else "")
              + 'AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 하면 그 코드의 출력과 그림을 채점합니다')
        print("─" * 54)
        return False
    if not txt.strip():
        print("❌ 출력이 없습니다 — 값을 print 하도록 프롬프트의 출력 형식에 적으세요")
    values = spec.get("values", {})
    units = [str(ref[2]) for ref in values.values() if len(ref) > 2]
    val_lines = set(_value_lines(txt, units))               # 단위·「=」 가 붙은 행 — 이름 바로 뒤 값 줄이 없을 때 먼저 씁니다
    all_lines = [l for l in txt.splitlines() if l.strip()]
    series = _bi.bool(_SERIES_RE.search(txt))               # Series 를 통째로 찍은 흔적
    need_words = spec.get("need_words", {})        # 함정 미션 — {이름: [낱말, …]} 그 항목의 토막에 꼭 있어야 하는 낱말
    cause = _get("_CAUSE") or {}
    fail_notes = cause.get("fail", {}).get(key, {})            # 흔한 실수의 값 → 원인 문장
    pass_notes = cause.get("pass", {}).get(key, {})
    code_notes = cause.get("code", {}).get(key, {})
    find = {l: (r[3] if len(r) > 3 and r[3] else l) for l, r in values.items()}   # 출력에서 찾을 이름(짧은 이름 포함)
    names = list(find.values()) + (["공식 차수"] if spec.get("차수_표기") else [])
    skip = _BOOK.get("skip", {})
    own_too = []                                   # own_too: 출력 꼴 힌트에 그 항목의 권 힌트를 함께 낼 이름(아래 `not occ`)
    form_only = spec.get("form_only")              # 이름 줄이 없는 칸에 출력 꼴 힌트만 내는 미션(권 힌트 첫 단이 읽은 값을 전제로 쓴 글)
    forms = spec.get("forms", {})                  # {이름: "낱말 숫자 %"} 값과 낱말을 함께 찍는 칸의 출력 꼴
    paren_unit = spec.get("paren_unit", ())        # 「k 건 (r %)」 꼴로 찍는 칸 — 괄호 안의 기준 단위 수를 읽습니다(`_unit_value`)
    traps = spec.get("traps", {})                  # {이름: [(하한, 상한, 문장), …]} 흔한 실수의 값이면 그 실수를 짚는 힌트
    seen = {}                                      # 값 칸에서 읽은 값 — 권 검사가 값끼리 맞물리는지 볼 때 씁니다(ctx["values"])
    made = "고칠 것" if trap else "만들 것"          # 채점할 이름이 적힌 본문 소제목 — 함정 미션은 「고칠 것」, 다른 미션은 「만들 것」
    for label, ref in values.items():
        skip_if, skip_msg = skip.get((key, label), (None, ""))
        if skip_if is not None and skip_if():           # 대체 탐색을 쓴 날의 AIC 처럼 채점하지 않는 칸
            print(f"ⓘ {label} — {skip_msg}")
            continue
        lo, hi = float(ref[0]), float(ref[1])
        unit = ref[2] if len(ref) > 2 else ""
        name = find[label]
        want_form = f"「{name} = {forms.get(label) or ('숫자' + ((' ' + unit) if unit else ''))}」"
        where = "이 이름이 있는 줄" if name == label else f"이 이름(「{name}」)이 있는 줄"
        # 이 이름이 있는 줄에서 이름 뒤의 값 하나만 읽습니다. 이름이 있는 줄이 없을 때 출력 전체의 숫자로
        # 물러나면 다른 행의 값(부호를 뒤집은 차, 다른 모델의 값)으로 틀린 값이 ✅ 가 됩니다
        occ = _occurrences(all_lines, name, names, series, unit)   # 「이름 = 값」·「이름: 값」·「이름 값」 — 모든 행에서
        # 같은 이름 줄이 여럿이면 이름 바로 뒤 값 줄만 봅니다(`_value_use`). 그 줄들의 값이 모두 맞아야 통과입니다 — 하나만
        # 맞아도 통과시키면 옛 print 를 남긴 채 덧붙인 줄·부호를 뒤집은 줄이 헛통과합니다.
        use = _value_use(occ, val_lines)
        vals = [o["val"] for o in use]
        if label in paren_unit:                         # 「이름 = 14 건 (1.4 %)」 — 괄호 안의 기준 단위 수가 값입니다
            vals = [_unit_value(v, unit) for v in vals]
        in_range = [v for v in vals if lo <= v[0] <= hi]
        clash = [v for v in vals if not (lo <= v[0] <= hi) and _short_digits(v[1], v[0], lo, hi) is None]
        # 이름 바로 뒤 값 줄이 하나도 없어 「이름 = 값」 꼴이 아닌 줄에서 읽은 값 — 이름 앞에 다른 말이 붙은 줄(더 긴 이름일 수 있는 줄:
        # 「상태별 리뷰 수 : 통과 178 …」)이나, 값 표지 없이 이름과 숫자 사이에 다른 말이 있는 줄(결론 문장 「…하루 평균은 9월의
        # 0.933배로…」 의 9). 범위 안이면 그 값을 받고(「전체 리뷰 수 = 200 건」 · 「리뷰 수는 200 건입니다」), 범위 밖이면 이 이름의
        # 값으로 읽지 않습니다 — 다른 항목의 값이나 문장 속의 수로 「읽은 값은 … 범위 밖」 과 계산 힌트를 내면 거짓 원인이 됩니다.
        # 이름과 값 표지 사이에 설명이 낀 줄(「이름 — period=24, robust=False → 0.8115」)은 지금처럼 그 값을 읽습니다
        weak = _bi.bool(use) and (use[0]["pre"] or (not use[0]["val"][3] and use[0]["val"][5] == "bare"))
        foreign = use[0]["line"].strip() if weak and not in_range else ""
        if vals and not foreign:
            seen[label] = (in_range or vals)[0][0]
        # 개수 자리에 찍은 비율 — 기준 단위가 % 로 시작하는 칸(% · %p)은 값 뒤의 % 가 제 단위입니다
        pct = (next((v for v in in_range if v[2].lstrip().startswith("%")), None)
               if unit and not unit.startswith("%") else None)
        stale = _stale_code(code_notes.get(label), _STATE.get("src"), lo, hi) if in_range else ""
        words = need_words.get(label, ())
        words = (words,) if isinstance(words, str) else tuple(words)
        missing = []
        if in_range and words:          # 값은 기준 범위 안 — 그 항목의 토막에 필요한 낱말이 있는가
            # 값 칸 하나만 고치고 전임자의 틀린 결론 문구(「→ 정상」)를 그대로 둔 줄이 통과하지 않게 합니다.
            # 줄 전체가 아니라 이 항목의 토막 — 같은 줄의 다른 항목 낱말은 보지 않습니다
            rows = []
            for o in _occurrences(all_lines, name, names, series):
                flat = _re.sub(r"\s+", "", o["seg"])
                rows += [flat, _re.sub(r"\([^()]*\)", "", flat)]
            missing = [w for w in words if not _bi.any(_word_in(row, w) for row in rows)]
        if in_range and pct is not None:              # 개수 자리에 비율을 찍은 판 — 범위가 넓어 수만으로는 못 거릅니다
            passed = False
            failed.append((label, _hint_key(key, "unit", label)))
            print(f"❌ {label} — {where}에서 읽은 값 {_val_txt(pct)} 뒤에 % 가 붙어 있습니다."
                  f" 비율이 아니라 {want_form} 처럼 개수로 출력하세요")
        elif in_range and clash:                      # 같은 이름, 다른 값
            passed = False
            failed.append((label, _hint_key(key, "dup", label)))
            shown_vals = " · ".join(_val_txt(v) for v in vals)
            print(f"❌ {label} — {where}이 {len(vals)}개이고 값이 서로 다릅니다({shown_vals})."
                  f" 틀린 값을 찍는 옛 줄을 지우고 {want_form} 한 줄만 남기세요")
        elif in_range and stale:                      # 값만 손으로 적고 코드는 그대로 둔 판
            passed = False
            failed.append((label, label))
            print(f"❌ {label} — 출력에서 {_val_txt(in_range[0])} 를 찾았지만 {stale}")
        elif in_range and missing:
            passed = False
            failed.append((label, label))
            _prev_fail = _TRY.get(key, 0)              # 첫 ❌ 에는 빠진 낱말을 감춥니다
            print(f"❌ {label} — 값 {_val_txt(in_range[0])} 는 기준 범위 안이지만, 이 이름이 있는 줄의 문구가 기준과 다릅니다"
                  + (" (그 줄에 있어야 할 낱말 : " + " · ".join(f"「{w}」" for w in missing) + ")"
                     if _prev_fail else " — 무엇을 기각했고 어느 쪽 증거인지 같은 줄에 적었는지 보세요"))
        elif in_range:
            print(f"✅ {label} — 출력에서 찾은 값: {_val_txt(in_range[0])}")
            note = _note_for(pass_notes.get(label, ()), [in_range[0][0]], "")
            if note:
                print(f"   ⓘ {note}")
            if in_range[0][4] == "series":           # f-string 안의 Series 는 뒤에 적은 단위가 dtype 줄 끝으로 밀립니다 —
                # 단위 ⓘ 까지 겹치면 적은 단위를 안 적었다고 읽히므로, 출력 꼴을 고치라는 ⓘ 하나만 냅니다
                print("   ⓘ 표(Series)를 그대로 넣어 인덱스와 값이 함께 찍혔습니다 — 값만 읽었습니다."
                      " float(…) 로 숫자 하나만 출력하면 보고서에 그대로 옮길 수 있습니다")
            elif unit and unit not in " ".join(o["tail"] for o in use):   # 단위는 그 이름 줄에서 봅니다
                print(f"   ⓘ 단위 「{unit}」 도 값과 함께 출력하면 보고서에 그대로 옮길 수 있습니다")
        elif not occ or foreign:
            passed = False
            # 계산 힌트가 아니라 출력 꼴 힌트 — 함정 미션의 낱말 항목은 행 이름·문구를 함께 안내하는 항목 힌트 그대로
            form_key = _hint_key(key, "form", label)
            failed.append((label, label if words else form_key))
            # 함정 미션의 전임 담당자 코드가 이 이름의 줄을 아예 찍지 않는 권이 있습니다 — 공통 출력 꼴 힌트만 내면 그 함정을
            # 짚는 항목 힌트와 튜터에 물을 것이 사라지므로, 권이 그 항목에 적어 둔 힌트 첫 단과 물을 것을 함께 냅니다.
            # 두 경우는 지금처럼 출력 꼴 힌트만 냅니다 — 권에 출력 꼴 힌트(이름_형식 · label)가 따로 있으면 그 힌트와 물을
            # 것이 이미 권의 것이고, 낱말을 채점하는 함정 미션은 전임 담당자 코드가 이름 줄을 이미 찍으며 낱말 없는 항목의
            # 힌트가 그 줄의 결론 문구를 짚는 글이라 이름 줄이 없을 때 내면 사실과 다를 수 있습니다
            # 권 힌트 첫 단이 읽은 값을 전제로 쓴 글이면(기준값을 예로 든 글) 이름 줄이 없는 화면에 내지 않습니다 — 기준의 `form_only`
            if (trap and not need_words and not form_only and form_key == "_form"
                    and _hint_parts(_HINT.get(key, {}).get(label))[0]):
                own_too.append(label)
            print(f"❌ {label} — {where}을 출력에서 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
            if foreign:
                print(f"   ⓘ 「{foreign[:60]}」 은 「이름 = 값」 꼴의 줄이 아니라(이름 앞뒤에 다른 말이 있습니다) 이 이름의 값으로 읽지 않았습니다")
        elif not vals:
            passed = False
            note = _note_for(fail_notes.get(label, ()), [], " ".join(o["tail"] for o in occ))   # 「= nan」 같은 흔한 실수
            failed.append((label, label if words or note else _hint_key(key, "form", label)))
            if _bi.any(o["boxed"] for o in occ):     # 배열로 찍힌 값 — 값이 아니라 출력 꼴의 문제입니다
                print(f"❌ {label} — {where}에서 숫자를 찾지 못했습니다 — 값이 대괄호 [ ] 안의 배열이라 숫자 하나로 읽지 못했습니다")
                print(f"   ⓘ 값이 배열·Series 꼴로 찍혔습니다 — float(값) 이나 .item() 으로 꺼낸 숫자 하나를"
                      f" {want_form} 한 줄로 출력하세요")
            else:
                print(f"❌ {label} — {where}에서 숫자를 찾지 못했습니다. {want_form} 한 줄로 출력하세요")
                if _bi.any(_re.search(r"[(\[][^()\[\]]*\d", o["tail"]) for o in occ):
                    print("   ⓘ 이 줄의 숫자는 괄호 안에만 있습니다 — 괄호 안은 설명으로 보고 값으로 읽지 않습니다."
                          f" 값을 괄호 밖에 두고 {want_form} 꼴로 출력하세요")
            if note:
                print(f"   원인 : {note}")
        else:
            passed = False
            first = vals[0]
            _prev_fail = _TRY.get(key, 0)              # 첫 ❌ 에는 기준 범위를 감춥니다
            short = _short_digits(first[1], first[0], lo, hi)
            if short is not None:
                done = "정수로만" if short == 0 else f"소수점 아래 {short}자리까지만"
                got_txt = (f"{where}에서 읽은 값은 {_val_txt(first)} 인데 기준 범위 밖입니다 — 반올림한 자리가 모자라"
                           f" 범위 밖으로 읽혔습니다({done} 찍힘). 「{made}」 에 적힌 소수점 아래 자릿수대로 출력하세요")
            else:
                got_txt = f"{where}에서 읽은 값은 {_val_txt(first)} 인데 기준 범위 밖입니다"
            ref_txt = spec.get("range_text", {}).get(label)      # 한쪽이 열린 기준(「0 보다 큼」)은 범위 대신 말로 적습니다
            band = (f"기준: {ref_txt}" if ref_txt else f"기준 {_num_txt(lo, 'g')}" if lo == hi
                    else f"기준 범위 {_num_txt(lo, 'g')} ~ {_num_txt(hi, 'g')}")
            print(f"❌ {label} — {got_txt}" + (f" ({band})" if _prev_fail else ""))
            note = "" if short is not None else _note_for(fail_notes.get(label, ()), [v[0] for v in vals],
                                                          " ".join(o["tail"] for o in use))
            if note:                                 # 흔한 실수 하나와 같은 값이면 그 원인을 바로 말합니다
                print(f"   원인 : {note}")
            trap_note = "" if short is not None else _note_for(traps.get(label, ()), [v[0] for v in vals], "")
            if trap_note:                            # 기준에 적어 둔 흔한 실수의 값(traps)이면 그 실수를 짚습니다
                print(f"   힌트 : {trap_note}")
            if short is None and _bi.any(_re.search(r"\[|\barray\(|dtype|\bnp\.\w+\(", o["tail"]) for o in use):
                print("   ⓘ 값이 배열·Series 꼴로 찍혔습니다 — float(값) 이나 .item() 으로 숫자 하나만 출력하세요")
            elif short is None and first[5] == "bare":
                print(f"   ⓘ 이 줄은 이름 바로 뒤에 「=」·「:」 가 없어 이름 뒤의 숫자를 값으로 읽었습니다."
                      f" {want_form} 꼴로 출력하면 「=」 뒤 값을 읽습니다")
            elif short is None and first[5] == "marker" and _bi.any(lo <= v <= hi for v in _nums(first[2])):
                # 「= 182/200 = 91.0%」·「= 512일 가운데 0 일」·「= 2개 원점 평균 0.271」 — 값은 같은 줄 뒤쪽에 있습니다.
                # 계산이 아니라 출력 꼴의 문제일 수 있다는 것을 알려 계산부터 의심하지 않게 합니다. 뒤 숫자가 우연히 범위 안일
                # 수도 있어(「차 = -0.210 (MSE 0.536, MAE 0.326)」 의 0.326) 「값이 그 뒤에 있다면」 으로 적습니다
                print(f"   ⓘ 이 줄은 「=」·「:」 뒤의 첫 숫자({first[1]})를 값으로 읽었습니다 — 같은 줄의 뒤 숫자는 읽지 않습니다."
                      f" 값이 그 뒤에 있다면 계산식·설명에 든 숫자는 빼고 {want_form} 꼴로 값 하나만 출력하세요")
            failed.append((label, _hint_key(key, "digits", label) if short is not None else label))
    order_tag = spec.get("차수_표기")               # 적합만 고치고 출력 문자열을 그대로 두면 ✅ 가 열리던 자리
    if order_tag:
        order_label = "공식 차수"                   # 이 이름이 있는 줄의 「=」·「:」 뒤에서 고친 차수 표기를 찾습니다
        order_occ = _occurrences(all_lines, order_label, names)   # 「공식 차수 AIC」 줄은 건너뜁니다
        want_order = tuple(int(x) for x in _re.findall(r"\d+", order_tag))
        got = [r for r in (_order_of(o["tail"]) for o in order_occ) if r]
        # 이름 바로 뒤 값 줄(「공식 차수 = …」)이 있으면 그 줄만 — 「공식 차수 탐색: …」 같은 줄의 숫자는 차수가 아닙니다
        read = [r[0] for r in ([r for r in got if r[1]] or got)]
        many = list(dict.fromkeys(read))
        order_form = f"「{order_label} = (p,d,q)(P,D,Q,m)」 한 줄로 출력하세요"
        if len(many) == 1 and many[0] == want_order:
            print(f"✅ 차수 표기 — 「{order_label}」 줄이 고친 차수를 적고 있습니다")
        else:
            passed = False
            if len(many) > 1:
                failed.append(("차수 표기", _hint_key(key, "dup", "차수 표기")))
                print(f"❌ 차수 표기 — 「{order_label}」 줄이 {len(read)}개이고 적힌 차수가 서로 다릅니다"
                      f" ({' · '.join(str(o) for o in many)}). 최종 차수 한 줄만 남기세요")
            elif many:
                failed.append(("차수 표기", "차수_표기"))
                print(f"❌ 차수 표기 — 「{order_label}」 줄에 적힌 비계절 차수 {many[0]} 가 고친 차수와 다릅니다")
            else:
                failed.append(("차수 표기", _hint_key(key, "form", "차수 표기")))
                print("❌ 차수 표기 — " + (f"「{order_label}」 줄에서 숫자 3개로 된 차수를 읽지 못했습니다. {order_form}"
                                        if order_occ else f"이 이름이 있는 줄을 출력에서 찾지 못했습니다. {order_form}"))
    flat_lines = [_flat(l) for l in all_lines]
    forbid_all = [str(p) for p in spec.get("forbid", ())]
    forbid_seen = set()
    for phrase in forbid_all:                        # 함정 미션 — 전임자의 결론 줄이 남으면 통과시키지 않습니다
        # 빈칸만 다른 문구(「판단 :」·「판단:」)는 빈칸을 걷으면 같은 문구입니다 — 한 번만 알립니다
        same = _flat(phrase)
        if not same or same in forbid_seen:
            continue
        # 「…넣어도 되지 않습니다」 처럼 서술로 부정해 인용한 줄은 남은 줄이 아닙니다(`_forbid_in`)
        if _bi.any(_forbid_in(row, phrase) for row in flat_lines):
            forbid_seen.add(same)
            passed = False
            if ("남은 줄", "forbid") not in failed:
                failed.append(("남은 줄", "forbid"))
            # 알리는 글자는 출력에 찍힌 꼴 — 같은 문구의 여러 꼴 가운데 출력에 글자 그대로 있는 것, 없으면 먼저 적힌 것
            shown_as = next((p for p in forbid_all if _flat(p) == same and p in txt), phrase)
            print(f"❌ 남은 줄 — 출력에 「{shown_as}」 문구가 남아 있습니다")
    plot_spec = spec.get("plot")
    if plot_spec:
        axs = [a for f in figs for a in _axes(f)]
        if not axs:
            passed = False
            failed.append(("그래프", "plot"))
            print("❌ 그래프 — 그림을 찾지 못했습니다 (그래프 한 장도 같은 프롬프트에 함께 시킵니다."
                  " 채점은 matplotlib 으로 그린 그림을 읽습니다)")
        else:
            gaps = None
            for ax in axs:                 # 축이 여럿이면 기준에 맞는 축이 하나만 있어도 통과입니다
                try:
                    this_gaps = _plot_gaps(ax, plot_spec)
                except Exception as e:
                    this_gaps = [f"그림의 축 이름을 읽지 못했습니다 ({type(e).__name__})"]
                if not this_gaps:
                    gaps = []
                    break
                if gaps is None or len(this_gaps) < len(gaps):
                    gaps = this_gaps
            axis_words = "가로축·세로축 이름이" if plot_spec.get("축라벨") or not plot_spec.get("가로축라벨") else "가로축 이름이"
            if gaps:
                passed = False
                failed.append(("그래프", "plot"))
                print("❌ 그래프 — " + " · ".join(gaps))
            elif shown is False:                    # inline 백엔드인데 이 셀이 그림을 한 장도 화면에 내보내지 않았습니다
                print(f"✅ 그래프 — 그림 객체에 {axis_words} 있습니다")
                print("   ⓘ 그림이 화면에 뜨지 않았습니다 — plt.close() 나 savefig 로만 끝냈다면 plt.show() 로 띄워"
                      " 눈으로도 확인하세요")
            else:                                   # 그래프 종류는 가리지 않고 그림과 축 이름만 봅니다
                print(f"✅ 그래프 — 그림이 있고 {axis_words} 있습니다")
    ctx, notices = None, []
    for check in _BOOK.get("checks", ()):            # 권 설정의 검사(계획표의 열 · 낱말 · 저장한 파일처럼 값·그림 밖의 결과물)
        try:
            if _takes_ctx(check):                    # fn(key, spec, ctx) — 출력 글 · 그림 · 실행 상태를 받는 꼴
                if ctx is None:
                    ctx = {"text": txt, "lines": all_lines, "figs": list(figs), "axes": [a for f in figs for a in _axes(f)],
                           "values": seen, "failed": failed, "attempt": _TRY.get(key, 0), "locked": _locked, "trap": trap,
                           "src": str(_STATE.get("src") or ""), "code": _ai_lines(_STATE.get("src")),
                           "prompt": _prompt_now(prompt), "shown": shown,
                           "read": lambda name, others=(): [o["val"][0] for o in _value_use(
                               _occurrences(all_lines, name, names + list(others), series), val_lines)],
                           "tails": lambda name, others=(): _text_tails(all_lines, name, names + list(others))}
                rows = list(check(key, spec, ctx) or ())
            else:                                    # fn(key, spec) — 옛 꼴
                rows = list(check(key, spec) or ())
        except Exception as e:                       # 검사가 예외로 끝나도 채점 표는 끝까지 찍습니다
            rows = [("채점 검사", False, f"이 미션의 검사 하나를 마치지 못했습니다 ({type(e).__name__}: {_one_line(e, 80)})."
                     " 이 미션의 준비 셀을 실행했는지 보고, 그래도 같으면 맨 위 「[채점] 이 셀을 먼저 실행하세요」 셀을 다시 실행한 뒤"
                     " 이 셀을 다시 실행하세요", "_check")]
        for row in rows:
            name, ok, why, hkey = row[:4]
            more = row[4] if len(row) > 4 else ()
            if ok is None:                           # 알림 — 틀린 것이 아니라 다음 할 일(❌ 로 세지 않고 풀이는 열지 않습니다)
                notices.append(why if not name else f"ⓘ {name} — {why}")
                continue
            if ok:
                print(f"✅ {name} — {why}")
            else:
                passed = False
                if (name, hkey) not in failed:       # 같은 항목이 두 번 ❌ 여도 힌트는 한 번만
                    failed.append((name, hkey))
                print(f"❌ {name} — {why}")
            for line in ((more,) if isinstance(more, str) else more or ()):   # 그 줄에 덧붙이는 줄(ⓘ 참고 · 힌트)
                print(f"   {line}")
    tbl = dict(_HINT.get(key, {}))
    tbl.setdefault("_form", tuple(t.replace("「만들 것」", f"「{made}」") for t in _FORM_HINT))     # 공통 힌트가 가리키는
    tbl.setdefault("_digits", tuple(t.replace("「만들 것」", f"「{made}」") for t in _DIGITS_HINT))  # 소제목을 이 미션의 것으로
    tbl.setdefault("_dup", _DUP_HINT if trap else _DUP_HINT_PLAIN)
    attempt = _TRY[key] = _TRY.get(key, 0) + (1 if failed else 0)
    groups = {}                                    # 같은 힌트(출력 꼴 등)는 한 번만 — 그 힌트가 걸린 항목 이름을 모아 찍습니다
    for name, hkey in failed:
        groups.setdefault(hkey, []).append(name)
    for hkey, members in groups.items():
        tiers = tbl.get(hkey, ("",))
        tier = tiers[:-1] if len(tiers) > 2 else tiers[:1]   # 2단 힌트는 마지막 원소가 튜터에 물을 것입니다
        hint = tier[_bi.min(attempt, len(tier)) - 1] if tier else ""
        if hint:
            print(f"   힌트({'출력 꼴' if hkey == '_form' else ' · '.join(members)}) : {hint}")
        if hkey == "_form":                        # 함정 미션 — 출력 꼴 힌트 뒤에 그 항목의 권 힌트 첫 단
            for name in members:
                if name in own_too:
                    print(f"   힌트({name}) : {_hint_parts(tbl.get(name))[0][0]}")
    if failed:
        print(_ASK_HEAD.get(form, ""))
        tmpl = _ASK_FORM.get(form) or ""
        for hkey, members in groups.items():
            # 함정 미션에서 이름 줄이 없던 항목은 권이 그 항목에 적어 둔 물을 것을 씁니다(없는 줄의 출력 꼴을 묻지 않게)
            own_ask = [m for m in members if hkey == "_form" and m in own_too and _hint_parts(tbl.get(m))[1]]
            rest = [m for m in members if m not in own_ask]
            if rest:
                question = (tbl.get(hkey, ("", ""))[-1] if len(tbl.get(hkey, ("",))) > 1 else "") \
                    or "이 항목이 왜 ❌ 로 남는지"
                joined = "」·「".join(rest) if "「{name}」" in tmpl else " · ".join(rest)
                print(_ask_line(form, joined, prompt, question))
            for name in own_ask:
                print(_ask_line(form, name, prompt, _hint_parts(tbl.get(name))[1]))
    elif notices:                                  # 틀린 것은 없고 다음 할 일이 남았습니다(두 번에 나눠 푸는 미션의 1회차)
        for line in notices:
            print(line)
    elif _locked:                                  # 값·그림은 기준에 맞지만 PROMPT 칸이 아직 비었습니다
        print('ⓘ 값과 그림은 기준에 맞습니다 — 이 셀의 PROMPT = """ """ 따옴표 안에 고칠 지시를 내 말로 쓰고'
              " 다시 실행하면 완성본 풀이가 열립니다")
    else:
        print("🔓 정답이 열렸습니다 — 아래 「🔓 완성본 풀이 보기」를 펼쳐 내 코드와 비교해 보세요")
        _reveal_show(_b64.b64decode(_REVEAL[key]).decode("utf-8"))
    print("─" * 54)
    return passed and not _locked and not notices


# ───────────────────────── PROMPT 셀 훅 ─────────────────────────
class _Tee:
    """「PROMPT 셀」의 출력을 화면과 버퍼 양쪽으로 흘려보냅니다"""

    def __init__(self, orig, buf):
        self._o, self._b = orig, buf

    def write(self, s):
        self._b.write(s)
        return self._o.write(s)

    def flush(self):
        self._o.flush()

    def __getattr__(self, n):
        return getattr(self._o, n)


def _tag(src):
    """머리 행이 `# 미션 N — 프롬프트`·`# 과제 — 프롬프트` 인 셀만 채점 대상입니다.

    머리 행은 비어 있지 않은 앞 3행 가운데 처음 나오는 `# 미션`·`# 과제` 행입니다 — 맨 앞에 빈 줄이나
    import 한 줄이 끼어도 채점 표가 찍히게 합니다. 「준비 셀」(`# 미션 N · 준비`)은 대상이 아닙니다.
    """
    lines = [l for l in (src or "").split("\n") if l.strip()]
    if lines and lines[0].lstrip().startswith("#@title"):
        return None
    for head_line in lines[:3]:
        m = _re.match(r"\s*#\s*(미션\s*(\d+)|과제)\b", head_line)
        if not m:
            continue
        if "프롬프트" not in head_line:
            return None
        return "m" + m.group(2) if m.group(2) else "과제"
    return None


def _lost_tag(src):
    """머리줄이 사라진 「PROMPT 셀」 — 미션 이름이 적힌 코드면 그 미션, PROMPT·표시선만 있으면 "?", 아니면 None.

    AI 답 전체를 셀에 덮어쓰면 첫 행의 머리줄이 사라져 채점이 아무것도 찍지 않습니다. 조용히 지나가지 않고
    「머리줄이 없어 채점하지 못했다」 고 채점 표로 알립니다(`_lost_notice`).
    """
    first = next((l for l in (src or "").split("\n") if l.strip()), "")
    if first.lstrip().startswith("#@title") or _re.match(r"\s*#\s*(미션\s*\d+|과제)\s*·\s*준비", first):
        return None                                       # 채점 셀 · 준비 셀
    flat = _flat(src)
    hits = {k: _bi.sum(_flat(l) in flat for l in (_device_spec(s) or {}).get("values", {})) for k, s in _ANS.items()}
    best = _bi.max(hits.values()) if hits else 0
    keys = [k for k, n in hits.items() if n == best and n > 0]
    if len(keys) == 1:
        return keys[0]
    if "여기부터 AI 코드" in (src or "") or _re.search(r"^[ \t]*PROMPT[ \t]*=", src or "", _re.M):
        return "?"
    return None


def _lost_notice(key):
    """머리줄이 없어 채점하지 못한 셀 — 채점 표 꼴로 알리고 닫습니다"""
    if key == "?":
        head_label, header = "채점하지 못한 셀", "「# 미션 N — 프롬프트」"
    else:
        head_label = ("미션 " + key[1:]) if key[:1] == "m" and key[1:].isdigit() else key
        trap = _is_trap(key)                               # 빈 PROMPT 에도 잠그지 않는 미션이 함정 미션입니다
        header = f"「# {head_label} — 프롬프트{' (함정)' if trap else ''}」"
    print("\n" + "─" * 54)
    print(f"◼ {head_label} 채점 결과" if key != "?" else f"◼ {head_label}")
    print(f"❌ 머리줄 — 셀 첫 행에 {header} 행이 없어 이 셀을 채점하지 못했습니다")
    print("   힌트 : AI 답을 붙여 넣으며 셀 전체를 덮어쓰면 첫 행의 머리줄, PROMPT 입력란, 「# --- 여기부터 AI 코드 ---」"
          " 표시선이 함께 사라집니다. 셀을 되돌리거나(실행 취소) 그 세 가지를 셀 맨 위에 다시 적고,"
          " AI 코드는 표시선 아래에만 붙여 넣은 뒤 다시 실행하세요")
    print("─" * 54)


def _shown_text(data):
    """display 출력 묶음에서 화면에 보이는 글자 — text/plain 이 자리표시면 markdown · html 의 글자.

    자리표시는 `<IPython.core.display.Markdown object>` · `<pandas.io.formats.style.Styler at 0x…>` 같은 객체 표기입니다
    (`display(표.style…)` 의 text/plain 에는 「object」 낱말이 없습니다). HTML 표는 칸마다 줄이 바뀌어 적혀 있으므로
    한 행을 한 줄로 모으고, 화면에 보이지 않는 `<style>`·`<script>` 는 뺍니다.
    """
    import html as _html
    plain = str(data.get("text/plain", ""))
    if plain and not _re.fullmatch(r"\s*<[\w.]+(?: object)?(?: at 0x[0-9a-fA-F]+)?>\s*", plain):
        return plain
    if "text/markdown" in data:                  # **굵게** · `코드` 표시 기호는 화면에 안 보입니다
        return _re.sub(r"\*\*|__|`", "", str(data["text/markdown"]))
    if "text/html" in data:
        page = _re.sub(r"(?is)<(style|script)\b.*?</\1\s*>", "", str(data["text/html"]))
        page = _re.sub(r"(?is)<table\b.*?</table\s*>", lambda m: _re.sub(r"\s+", " ", m.group(0)), page)
        page = _re.sub(r"(?i)</t[dh]\s*>", "\t", page)
        page = _re.sub(r"(?i)<br\s*/?>|</(?:p|div|tr|li|h\d)\s*>", "\n", page)
        return _html.unescape(_re.sub(r"<[^>]+>", "", page))
    return plain


def _pub_hook():
    """display() 로 띄운 표·값도 채점이 읽게 — Colab·LMS(ipykernel)는 display 출력을 stdout 이 아닌 채널로 보냅니다.

    「PROMPT 셀」이 도는 동안만 display_pub.publish 를 감싸, 화면에 보이는 글자(text/plain · Markdown · HTML)를
    채점 버퍼에 적고 화면에 뜬 그림(image/*) 수를 셉니다. 터미널처럼 표시가 stdout 에도 쓰이는 자리에서 두 번
    담기지 않게, 원래 표시를 부르는 동안은 화면으로만 내보냅니다. `_post` 가 원래 publish 로 되돌립니다.
    """
    _pub_unhook()
    try:
        dp = _ip.display_pub
    except Exception:
        return
    had = "publish" in vars(dp)
    orig = dp.publish
    state, sysm = _STATE, _sys                 # 학생 코드가 도는 동안 불립니다 — 이름을 여기서 묶어 둡니다

    def note(d):
        if _bi.any(str(m).startswith("image/") for m in d):
            state["imgs"] = state.get("imgs", 0) + 1
        elif d and state.get("buf") is not None:
            state["buf"].write("\n" + _shown_text(d) + "\n")

    own = _get("_OWN")
    if isinstance(own, dict):                  # 학생 셀이 채점 코드와 같은 이름을 만든 뒤 display 를 불러도 읽게
        note = _guard(note, own, globals())

    def publish(data=None, *a, **k):
        try:
            note(data if isinstance(data, dict) else {})
        except Exception:
            pass
        cur = sysm.stdout
        sysm.stdout = state.get("out") or cur
        try:
            return orig(data, *a, **k)
        finally:
            sysm.stdout = cur

    dp.publish = publish
    _STATE["pub"] = (dp, had, orig, publish)


def _pub_unhook():
    """`_pub_hook` 이 감싼 publish 를 되돌립니다"""
    saved, _STATE["pub"] = _STATE.get("pub"), None
    if not saved:
        return
    dp, had, orig, mine = saved
    try:
        if vars(dp).get("publish") is mine:
            if had:
                dp.publish = orig
            else:
                del dp.publish
    except Exception:
        pass


def _pre(info=None):
    """「PROMPT 셀」이면 출력 버퍼를 걸고 그림 목록을 비웁니다"""
    try:
        src = str(getattr(info, "raw_cell", "") or "")
        _STATE.update(tag=None, lost=None, src=src, prompt=None, imgs=0)
        tag = _tag(src)
        hook = _BOOK.get("pre")
        if hook is not None:
            try:
                hook(tag if tag in _ANS else None)     # 권 설정 — 셀 실행 전 상태를 적어 둡니다
            except Exception:
                pass
        if tag is None:
            _STATE["lost"] = _lost_tag(src)
            return
        if tag not in _ANS:
            return
        _STATE["tag"] = tag
        _FIGS.clear()
        _STATE["ids0"] = {id(f) for f in _open_figs()}
        _STATE["prompt"] = _prompt_in_cell(src)    # 앞 미션의 PROMPT 가 남아 있어도 이 셀 것만
        _STATE["buf"], _STATE["out"] = _io.StringIO(), _sys.stdout
        _sys.stdout = _Tee(_sys.stdout, _STATE["buf"])
        _pub_hook()
    except Exception:
        pass


def _post(result=None):
    """「PROMPT 셀」이 끝나면 출력을 되돌리고 그 셀의 출력·그림으로 채점합니다 — 무슨 일이 있어도 채점 표를 닫습니다"""
    tag, buf, out, prompt = _STATE.get("tag"), _STATE.get("buf"), _STATE.get("out"), _STATE.get("prompt")
    _STATE.update(tag=None, buf=None, out=None, prompt=None)
    _pub_unhook()
    if out is not None:
        _sys.stdout = out
    lost, _STATE["lost"] = _STATE.get("lost"), None
    if lost:
        try:
            _lost_notice(lost)
        except Exception:
            print("─" * 54)
        return
    if tag is None or tag not in _ANS:
        return
    try:
        seen = {id(f) for f in _FIGS}
        figs_all = list(_FIGS) + [f for f in _open_figs()
                                  if id(f) not in seen and id(f) not in _STATE.get("ids0", set())]
        txt = buf.getvalue() if buf is not None else ""
        last_val = getattr(result, "result", None)       # 셀 마지막 표현식의 값 — print 없이 표로 띄운 값도 읽습니다
        if last_val is not None:
            try:
                txt = txt + "\n" + str(last_val)
            except Exception:
                pass
        # 문법 오류(SyntaxError·IndentationError)는 실행 전에 나서 error_before_exec 에만 남습니다 — 이것도 실행 오류입니다
        before = getattr(result, "error_before_exec", None)
        error = getattr(result, "error_in_exec", None) or before
        if before is None and prompt:
            prompt = None                                # 셀이 돌았으면 PROMPT 는 방금 실행된 값을 읽습니다
        elif before is not None and prompt is None:
            prompt = ""                                  # 돌지 않은 셀의 PROMPT 를 따옴표 꼴로 못 읽었습니다
        try:
            import matplotlib as _mpl
            shown = (_STATE.get("imgs", 0) > 0) if "inline" in _mpl.get_backend().lower() else None
        except Exception:
            shown = None
        judge(tag, txt, figs_all, error=error, prompt=prompt, shown=shown)
    except Exception as e:
        print(f"(채점을 마치지 못했습니다 — {type(e).__name__}: {_one_line(e)})")
        print("   맨 위 「[채점] 이 셀을 먼저 실행하세요」 셀을 다시 실행한 뒤 이 셀을 다시 실행하세요")
        print("─" * 54)


def _own_names(before):
    """이 셀이 만든 이름 → 값 — 밑줄로 시작하는 이름과 judge. IPython 이 쓰는 이름(_ · _i · _oh · _3 …)은 뺍니다.

    셀을 실행하기 전과 값이 달라졌거나 새로 생긴 이름입니다. 이 셀을 다시 실행하면 모듈 별칭처럼 값이 그대로인 이름은
    달라진 것으로 보이지 않으므로, 앞서 실행 때 적어 둔 이름도 함께 넣습니다.
    """
    prev, g = _get("_OWN"), globals()
    prev = prev if isinstance(prev, dict) else {}
    return {n: v for n, v in list(g.items())
            if (n[:1] == "_" or n == "judge") and n[:2] != "__" and n != "_G0"
            and not _re.fullmatch(r"_(?:i{0,3}|ih|oh|dh|i?\d+|exit_code)", n)
            and (n not in before or before[n] is not v or n in prev)}


def _guard(fn, own, g):
    """훅(`_pre`·`_post`)이 도는 동안만 채점 코드의 이름을 제자리에 둡니다.

    채점 코드의 도우미(`_axes`·`_flat`·`_nums`·`_get` …)는 학생 셀과 같은 이름 공간에 있습니다. 학생 코드가
    `_fig, _axes = plt.subplots(1, 2)` 처럼 같은 이름을 만들면 채점이 그 값을 함수로 부르다 멎고, 그 뒤 모든 미션이
    「채점을 마치지 못했습니다」 로 끝납니다. 그래서 훅에 들어갈 때 채점 코드의 이름을 되돌려 놓고, 나올 때 학생이 만든
    값을 돌려놓습니다 — 채점도, 그 이름을 쓰는 학생의 다음 셀도 그대로 돕니다.
    """
    def run(*a, **k):
        mine = {n: g[n] for n in own if n in g and g[n] is not own[n]}     # 학생 셀이 같은 이름에 넣어 둔 값
        g.update(own)
        try:
            return fn(*a, **k)
        finally:
            own.update({n: g[n] for n in own if n in g and g[n] is not own[n]})   # 채점 코드가 바꿔 단 값은 그대로
            g.update(mine)

    run.__name__ = fn.__name__
    return run


try:
    _OWN = _own_names(_G0)
    del _G0
    _ip = get_ipython()
    _OWN.update(_ip=_ip, _OWN=_OWN)
    for _ev, _fn in (("pre_run_cell", _pre), ("post_run_cell", _post)):
        for _old in list(_ip.events.callbacks[_ev]):
            if getattr(_old, "__name__", "") == _fn.__name__:
                _ip.events.unregister(_ev, _old)     # 이 셀을 다시 실행해도 채점은 한 번만 찍힙니다
        _ip.events.register(_ev, _guard(_fn, _OWN, globals()))
    print("채점 준비 완료 — 미션의 「PROMPT 셀」을 실행하면 실행 결과(출력)에 채점 결과가 찍힙니다")
except Exception:
    print("채점 준비 완료 (IPython 밖이라 훅 없이 judge() 만 올렸습니다)")

## 실행 환경

이 실습은 모델을 호출하지 않고 형태소 분석기로 낱말만 집계하므로 CPU 런타임으로 충분합니다. 맨 위 환경 확인 셀이 `DEVICE : cpu` 를 출력해도 그대로 진행합니다. GPU 런타임이면 `DEVICE : cuda` 가 출력되지만 결과는 같습니다.

원자료는 처음 실행할 때 한 번만 내려받습니다. 미션 1 · 2 와 과제의 준비 셀이 민원 파일 67.9MB 를, 미션 3 준비 셀이 리뷰 파일 20.6MB 를 가져옵니다. 받은 파일은 자료 폴더 `nlp_data` 에 저장되어 다음 실행부터는 다시 내려받지 않습니다.

두 파일은 먼저 과정이 GitHub 에 원 출처와 같은 바이트로 복제해 둔 사본에서 가져옵니다. 사본을 받지 못하면 원 출처(data.go.kr, GitHub bab2min/corpus)에서 받습니다.

준비 셀은 받은 파일을 SHA-256 으로 교안의 파일과 대조합니다. SHA-256 은 파일의 모든 바이트로 계산한 16진 숫자 64개이고 한 바이트만 달라도 값이 바뀝니다. 값이 다르면 준비 셀이 「원 출처 파일이 바뀌었습니다」 한 문장을 출력하고 계속 진행합니다. 이때는 결과 수가 교안과 다를 수 있습니다.

## 미션 1 — 민원 제목 2만 건의 키워드 문서 비율

#### 할 일 — 미션 1 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 시킬 일을 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다.
3. 아래 「먼저 예상해 보기」에서 하나를 고르고 근거를 한 문장 적습니다.
4. AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. 10초 안팎 걸립니다.
5. 채점 결과를 봅니다. ❌ 면 힌트대로 PROMPT 를 고치고 그래도 막히면 아래 「막히면」의 질문을 보냅니다.

**이 미션의 질문**: 2020년 민원 제목 무작위 2만 건의 형태소 명사를 집계하면, 「환불」을 쓴 제목은 몇 %이고 상위 20 은 어떤 낱말일까요?

이론 「1. 문자 인코딩」의 읽기 오류를 원본에서 확인하고 「3. 어절 · 형태소 · 품사 선택」의 형태소 명사로 집계합니다.

**쓸 수 있는 것**: 「준비 셀」의 변수와 함수입니다.

| 이름 | 무엇 | 쓰는 법 |
|---|---|---|
| `COMPLAINTS_PATH` | 민원 원본 파일의 경로. 준비 셀이 과정 복제본(원 출처와 같은 바이트)에서 받아 둔 cp949 파일입니다 | 다시 만들지 않는다 |
| `read_complaints(path)` | cp949 로 읽어 시험 입력 · 빈 제목 행을 빼고 전화번호 형태를 `[전화번호]` 로 바꾼 표를 만든다. 열은 `row_id`(원 파일 행 번호) · `사건제목` · `품목명` · `접수일자` 등 | 그대로 호출한다 |
| `sample_rows(df, n, seed=0)` | 시드를 고정한 무작위 n행. 같은 표 · 같은 시드면 같은 행이 같은 순서로 나온다 | 그대로 호출한다 |
| `kiwi_nouns(texts, kiwi)` | 문서마다 명사(NNG · NNP) 목록을 만든다. 한 문서 안의 순서와 반복을 그대로 둔다 | 그대로 호출한다 |
| `KIWI` | Kiwi 형태소 분석기(기본 사전, 문서를 동시에 두 개씩 분석하는 `num_workers=2`) | 다시 만들지 않는다 |
| `pct2(k, n)` | k ÷ n 의 백분율을 소수 둘째 자리까지 구한다. 교안과 같은 방식으로 반올림해 파이썬 `round` 와 끝 숫자가 다를 수 있다 | 그대로 호출한다 |

#### 만들 것

아래 5개를 모두 「PROMPT 셀」에서 출력합니다. 굵은 이름은 채점이 찾으니 글자 그대로 씁니다.

1. **기본값 읽기 오류**: 오류 이름(글자)
   - `pd.read_csv(COMPLAINTS_PATH)` 를 인코딩 인자 없이 `try` 안에서 읽을 때 난 오류의 이름입니다
2. **민원 행 수**: 단위 행 · 정수
   - `read_complaints(COMPLAINTS_PATH)` 로 다시 읽은 표의 행 수입니다
3. **환불 문서 비율**: 단위 % · 소수 둘째 자리
   - `sample_rows(표, 20000, seed=0)` 표본에서 명사 「환불」을 가진 제목 수 ÷ 20,000 입니다
   - 제목마다 명사 목록을 `set` 으로 바꿔 한 제목을 한 번만 집계합니다
4. **원문 9건**: 상위 3 낱말 × 원문 3건
   - 낱말마다 `sample(3, random_state=0)` 으로 뽑은 제목을 「원문 [낱말] 행 row_id · 제목」 형식으로 출력합니다
   - 상위 20 · 상위 3 은 불용어를 빼지 않은 순위입니다(1～3위 문의 · 환불 · 요청)
5. **그래프 한 장**: 종류는 자유이고 가로축 「문서 비율(%)」, 세로축 「낱말」 이름을 답니다
   - 권장은 상위 20 가로 막대입니다. 1위가 맨 위, 「환불」 막대만 다른 색입니다

**통과 기준**: 1～3번 값이 「이름 = 값」 형식으로 기준 범위 안에 있고 원문이 낱말마다 3건이어야 합니다. 그래프에도 가로축·세로축 이름이 있으면 통과입니다.

#### 함께 볼 것

1. 상위 20 의 낱말 · 문서 수 · 문서 비율: 1위부터 차례로
2. 원문 9건에서 제목의 주제가 드러나는 낱말과 불용어 구별하기

#### 프롬프트 힌트

- **이미 있는 변수와 함수**: 준비 셀의 변수 6개는 다시 만들지 말라고 적습니다.
- **집계 단위**: 「set 으로 바꿔 한 제목을 한 번만 집계해 줘」와 분모 20,000.

**막히면**: AI 튜터에게 「한 제목이 같은 명사를 두 번 가지면 문서 수를 집계할 때 어떻게 되나요」 처럼 질문합니다.

<details class="lms-extra"><summary>자세히 — AI 튜터에 맡길 일과 내가 확인할 일</summary>

| AI 튜터에 맡길 일 | 내가 반드시 확인할 일(틀려도 오류가 나지 않는 곳) |
|---|---|
| 기본값 읽기와 오류 확인, `read_complaints` 로 다시 읽기 | 첫 읽기에 `encoding` 인자가 **없는가**. 인자가 있으면 오류가 나지 않아, 기본값으로 읽을 때 무엇이 깨지는지 보지 못한다 |
| 표본 뽑기와 명사 목록 만들기 | 표본이 `sample_rows(표, 20000, seed=0)` 인가. `head(20000)` 처럼 앞 2만 행을 쓰면 무작위 표본이 아니다 |
| 낱말별 문서 수와 막대 그림 | 문서마다 **`set` 으로 바꿨는가**. 빼면 한 제목이 두 번 쓴 낱말을 2 로 집계한다. 분모가 20,000 인가 |

</details>

<details class="lms-extra"><summary>배경 — 낱말 순위와 빈도를 로그로 그린 그림</summary>

<img src="https://commons.wikimedia.org/wiki/Special:FilePath/Zipf_30wiki_en_labels.png" alt="가로축 log(rank), 세로축 log(frequency)인 그래프에 위키백과 30개 언어의 선이 왼쪽 위에서 오른쪽 아래로 거의 겹쳐 내려가는 그림" width="560" height="420"/>
<p align="center"><em>▲ 위키백과 30개 언어의 앞 1천만 낱말에서 순위와 빈도를 모두 로그로 그렸다. 언어가 달라도 순위가 뒤로 갈수록 빈도가 같은 기울기로 줄어든다(지프 법칙). 미션 1 의 상위 20 막대는 이 곡선의 왼쪽 부분이다 (출처: Wikimedia Commons, Zipf 30wiki en labels.png, CC BY-SA 4.0, 2015년 10월 덤프)</em></p>

</details>

#### 먼저 예상해 보기

> **[예측] 표본 2만 건에서 「환불」의 문서 비율은 50만 건 전체의 17.39% 와 비교해 어떨까요?**
>
> 1. 17.39% 에서 0.5%p 안쪽으로 가깝다
> 2. 17.39% 보다 1%p 넘게 높다
> 3. 17.39% 보다 1%p 넘게 낮다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>예상 확인 — 미션 1 결과를 본 뒤 펼칩니다</summary>

답은 첫째 보기입니다. 표본의 「환불」 17.75% 는 50만 건의 17.39% 보다 0.36%p 높습니다. 0.36%p 는 0.5%p 안쪽입니다. 표본 2만 건의 오차 한계 0.53%p 보다도 작은 차이입니다.

$$
1.96\sqrt{0.1775 \times 0.8225 \div 20{,}000} \approx 0.0053 \;\; (0.53\%\text{p})
$$

50만 건의 구간 17.28～17.49% 는 이보다 짧습니다.

</details>

In [ ]:
# 미션 1 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 1 에 쓸 이름을 직접 만듭니다
#  과정 복제본(원 출처와 같은 바이트)에서 민원 파일을 받아 두고(처음 한 번만), 읽기 · 표본 · 형태소 함수를 만듭니다.
#  표를 읽거나 형태소를 세지는 않습니다 — 그 일은 PROMPT 셀에서 합니다. 받은 뒤에는 1초 안팎입니다
import hashlib
import math
import os
import re
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal

import pandas as pd
from kiwipiepy import Kiwi

# ---------------------------------------------------------------- 원 출처 (조회 2026-09-24)
RAW_COMPLAINTS_URL = ("https://www.data.go.kr/cmm/cmm/fileDownload.do?"
                      "atchFileId=FILE_000000002610054&fileDetailSn=1&insertDataPrcus=N")
RAW_REVIEWS_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
RAW_FILES = {  # 이름: (원 출처 URL, 내 컴퓨터에 둘 파일 이름, 교안의 수를 계산한 파일의 SHA-256, 데이터 페이지)
    "complaints": (RAW_COMPLAINTS_URL, "kftc_complaints_1.csv",
                   "50c098fea2ec1d05191d07c45cd0f75b4d0ac3b5e5ac36910fe40f2aab699ebe",
                   "https://www.data.go.kr/data/15098314/fileData.do"),
    "reviews": (RAW_REVIEWS_URL, "naver_shopping.txt",
                "dc4d1aca0a148671cbe80bcb81962eee297370acab42be93c1617ce9336479c0",
                "https://github.com/bab2min/corpus/tree/master/sentiment"),
}
MIRROR_URL = "https://github.com/studio-js/korean-cx-text/raw/main/raw"  # 과정 복제본: 원 출처와 같은 바이트, 먼저 받는 곳
SUBDIRS = ("", "cache", "raw", "processed")  # 자료 폴더 아래에서 찾는 순서


def data_dir():
    # 자료 폴더: 환경변수 NLP_DATA_DIR 이 있으면 그 폴더, 없으면 ./nlp_data
    return os.environ.get("NLP_DATA_DIR", "nlp_data")


def find_local(name):
    # 자료 폴더 → cache/ → raw/ → processed/ 순서로 찾아 처음 나온 경로, 없으면 None
    for sub in SUBDIRS:
        path = os.path.join(data_dir(), sub, name)
        if os.path.isfile(path):
            return path
    return None


def sha256_of(path, chunk=1 << 20):
    # 파일의 SHA-256 16진 문자열 — 받은 파일이 교안의 파일과 같은지 대조합니다
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _download(url, path, retries=3):
    # url 을 path.part 로 받아 두고 성공하면 True (실패하면 세 번까지 다시 시도)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    for i in range(retries):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            return True
        except Exception as e:  # network error: retry
            print(f"받기 실패 {i + 1}/{retries}: {e}")
            time.sleep(2 * (i + 1))
    return False


def _fetch(url, name):
    # 자료 폴더에 있으면 그 경로, 없으면 url 에서 받아 자료 폴더/cache/ 에 두고 경로를 반환합니다(받지 못하면 None)
    path = find_local(name)
    if path is None:
        path = os.path.join(data_dir(), "cache", name)
        print(f"{name} 을 받습니다 (처음 한 번만)")
        if not _download(url, path):
            return None
        os.replace(path + ".part", path)
    return path


def fetch_raw(name):
    # 원자료 경로: 자료 폴더를 먼저 보고, 없으면 과정 복제본(MIRROR_URL) → 원 출처 URL 순서로 받습니다
    url, fname, sha, page = RAW_FILES[name]
    path = find_local(fname)
    if path is None:
        path = _fetch(f"{MIRROR_URL}/{fname}", fname)
        if path is not None and sha256_of(path) != sha:  # 복제본이 교안의 파일과 다르면 지우고 원 출처에서 받습니다
            os.remove(path)
            path = None
    if path is None:
        path = _fetch(url, fname)
    if path is None:
        raise RuntimeError(f"원 출처에서 파일을 받지 못했습니다. 데이터 페이지: {page}")
    if sha256_of(path) != sha:
        print(f"원 출처 파일이 바뀌었습니다. 결과 수가 교안과 다를 수 있습니다(데이터 페이지: {page})")
    return path


# ---------------------------------------------------------------- 반올림 (교안의 수와 같게)
def rnd(x, digits):
    # 십진 값 기준 반올림(마지막 숫자가 5 면 올림) — 파이썬 round 는 4.045 를 4.04 로 적습니다
    x = float(x)
    if not math.isfinite(x):
        return x
    return float(Decimal(f"{x:.12g}").quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


def pct2(k, n, digits=2):
    # 문서 k 건 ÷ 전체 n 건을 백분율로, 소수 둘째 자리: pct2(809, 20000) -> 4.05
    return rnd(100 * k / n, digits)


def half95(k, n, digits=2):
    # 비율 k/n 의 95% 구간 오차 한계(구간 길이의 절반, ± 뒤의 값)를 %p 로: 1.96 × √(p̂(1 − p̂)/n) × 100
    p = k / n
    return rnd(100 * 1.96 * math.sqrt(p * (1 - p) / n), digits)


# ---------------------------------------------------------------- 민원 파일 읽기
# 상담 시스템에 시험 삼아 입력한 행(소비자 민원이 아님)을 찾는 정규식
TEST_INPUT_RE = re.compile(
    r"(?i)^\s*(?:test|테스트)\s*(?:입니다)?\s*\.?\s*$"
    r"|IN-TEST"
    r"|(?:인바운드|아웃바운드|녹취|수화기).*테스트|테스트.*(?:인바운드|아웃바운드|녹취|수화기)"
    r"|^\s*자동차\s*(?:상담\s*)?테스트"
    r"|^\s*\d{8}\s*[-_ ].*테스트"
    r"|테스트\s*데이터|상담\s*(?:전화\s*)?테스트|전화\s*테스트|헬프데스크\s*시험|^\s*일반\s*테스트")
EMPTY_TITLE_RE = re.compile(r"^[^가-힣A-Za-z]*$")  # 한글 음절도 로마자도 없는 제목(「-」「.」「0」「ㅇ」)
PHONE_RE = re.compile(r"(?<!\d)(?:01[016789]|0\d{1,2})[-.)\s]{0,2}\d{3,4}[-.\s]\d{4}(?!\d)")  # 전화번호 형태
PHONE_MASK = "[전화번호]"  # 원 출처가 이미 가린 번호와 같은 표지


def mask_phones(text):
    # 전화번호 형태를 [전화번호] 로 가림
    return PHONE_RE.sub(PHONE_MASK, text)


def tidy_columns(df):
    # 열 이름 정리: '사건제목(ACCIDENT_TITLE)' -> '사건제목'
    return df.rename(columns=lambda c: c.split("(")[0].strip())


def read_complaints(path, verbose=True):
    # 민원 파일(cp949) 읽기: 시험 입력·빈 제목 행 제거, 전화번호 형태 가림, 열 이름 정리, 날짜 변환
    raw = pd.read_csv(path, encoding="cp949", dtype=str)
    df = tidy_columns(raw)
    df.insert(0, "row_id", range(len(df)))  # 원 파일의 행 번호(0부터)
    title = df["사건제목"].fillna("").str.strip()
    is_test = title.str.contains(TEST_INPUT_RE)
    is_empty = title.str.match(EMPTY_TITLE_RE) & ~is_test
    df = df[~is_test & ~is_empty].copy()
    title = title[df.index]
    df["사건제목"] = title.map(mask_phones)
    df["번호가림"] = df["사건제목"].ne(title)  # 전화번호 형태를 가린 행이면 True
    df["접수일자"] = pd.to_datetime(df["접수일자"], format="%Y-%m-%d")
    df = df.reset_index(drop=True)
    if verbose:
        print(f"민원 원본 {len(raw):,}행 → 시험 입력 {int(is_test.sum())}행·빈 제목 {int(is_empty.sum())}행 제거 "
              f"→ {len(df):,}행, 전화번호 형태 가림 {int(df['번호가림'].sum())}행")
    return df


# ---------------------------------------------------------------- 표본과 형태소
def sample_rows(df, n, seed=0):
    # 고정 표본: 같은 입력 행과 같은 seed 면 같은 행이 같은 순서로 나옵니다
    return df.sample(n=n, random_state=seed)


NOUN_TAGS = ("NNG", "NNP")  # 일반 명사 + 고유 명사 (Kiwi 기본 사전)


def kiwi_nouns(texts, kiwi):
    # 문서마다 명사(NNG·NNP) 목록 — 나온 순서와 반복을 그대로 둡니다(문서 수를 집계할 때는 set 으로)
    return [[t.form for t in toks if t.tag in NOUN_TAGS] for toks in kiwi.tokenize(list(texts))]


COMPLAINTS_PATH = fetch_raw("complaints")   # 쓸 이름 — 민원 원본 파일 경로 (cp949)
import os
os.environ.setdefault("KIWI_ARCH_TYPE", "balanced")  # CPU 종류(Mac·Colab)와 상관없이 형태소 분석 결과를 같게
KIWI = Kiwi(num_workers=2)                  # 쓸 이름 — 형태소 분석기(기본 사전, 작업자 2)
print(f"민원 원본 : {os.path.basename(COMPLAINTS_PATH)} · {os.path.getsize(COMPLAINTS_PATH):,} 바이트")
print("준비 완료 — COMPLAINTS_PATH · read_complaints · sample_rows · kiwi_nouns · KIWI · pct2")

In [ ]:
# 미션 1 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 1 의 완성 코드는 제목마다 명사 목록을 `set` 으로 바꾼 뒤 낱말마다 문서 수를 집계합니다. `set` 을 빼고 집계하면 무엇을 집계하게 되나요?**
>
> 1. 토큰 빈도: 한 제목이 같은 명사를 두 번 쓰면 2 가 된다
> 2. 문서 수 그대로: 결과가 달라지지 않는다
> 3. 어절 수: 공백을 기준으로 나눈 단위의 개수

## 미션 2 — 위약금이 해지보다 많다는 표 고치기 (함정 미션)

#### 할 일 — 미션 2 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 실행하기 전에 「PROMPT 셀」의 전임 코드를 읽고 맨 아래 「틀린 곳 찾기」에서 하나를 고릅니다. 고친 뒤 해지의 문서 비율이 커질지 작아질지 한 문장 적습니다.
3. 「PROMPT 셀」을 **그대로 ▶** 해 출력과 ❌ 를 확인합니다. PROMPT 가 비어 있어도 채점됩니다.
4. 아래 「고칠 것」대로 고칠 지시를 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다. 비어 있으면 완성본 풀이가 잠깁니다.
5. AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. ❌ 면 힌트대로 PROMPT 를 고칩니다.

**이 미션의 질문**: 전임 담당자는 위약금 4.05%, 해지 3.52% 를 근거로 「위약금 민원이 해지 민원보다 많다」고 결론 내렸습니다. 같은 표본 2만 건을 형태소 명사로 다시 집계하면 두 낱말의 순위는 어떻게 될까요?

#### 고칠 것

고친 뒤 아래 7개를 출력합니다. 굵은 이름은 채점이 찾으니 글자 그대로 씁니다.

1. **다섯 낱말 표**: 낱말마다 「낱말 · 어절 % · 형태소 %」
   - 어절 쪽은 전임 코드 그대로, 형태소 쪽은 `NOUNS_20K` 에서 그 낱말을 가진 제목 수 ÷ 20,000 입니다(제목마다 `set`)
2. **해지 형태소 비율**: 단위 % · 소수 둘째 자리
3. **위약금 형태소 비율**: 단위 % · 소수 둘째 자리
4. **배송 형태소 비율**: 단위 % · 소수 둘째 자리
5. **형태소 순위**: 해지와 위약금 가운데 형태소 문서 비율이 큰 쪽을 앞에 둔 「A > B」
6. **그래프 한 장**: 종류는 자유이고 가로축 「문서 비율(%)」, 세로축 「낱말」 이름을 답니다
   - 권장은 낱말마다 어절 · 형태소 막대가 나란한 가로 막대 10개입니다
7. **판단 행**: 전임 담당자의 「판단 :」 줄을 형태소 두 값의 비교 결과로 다시 씁니다
   - 비율이 큰 쪽 낱말 바로 뒤에 그 비율을 씁니다. 이어서 작은 쪽 낱말과 비율을 씁니다
     > 판단 : A 민원(○○.○○%)이 B 민원(○.○○%)보다 많다

**통과 기준**: 2～4번 값이 「이름 = 값」 형식으로 기준 범위 안에 있고 5·7번이 형태소로 구한 두 값의 비교여야 합니다. 그래프에도 가로축·세로축 이름이 있으면 통과입니다.

**바꿀 범위**: 집계 방법 하나만 바꾸고 표본 `C20K` · 낱말 목록 `FIVE_WORDS` · 반올림 `pct2` 는 그대로 둡니다. 그래야 두 표의 차이가 집계 단위에서만 나옵니다.

**쓸 수 있는 것**: 「준비 셀」의 변수와 함수입니다.

| 이름 | 무엇 | 쓰는 법 |
|---|---|---|
| `C20K` | 미션 1 과 같은 표본 2만 건(시드 0). 열은 `row_id` · `품목명` · `사건제목` | 다시 만들지 않는다 |
| `NOUNS_20K` | `C20K` 의 제목마다 Kiwi 명사(NNG · NNP) 목록. `NOUNS_20K[i]` 가 `C20K` 의 i 번째 제목의 목록이고, 한 제목 안의 순서와 반복을 그대로 둔다 | 다시 만들지 않는다 |
| `FIVE_WORDS` | 전임 담당자가 비교한 낱말 5개 `["환불", "해지", "위약금", "배송", "취소"]` | 다시 만들지 않는다 |
| `pct2(k, n)` | k ÷ n 의 백분율을 소수 둘째 자리까지 구한다. 반올림은 교안과 같은 방식이다 | 그대로 호출한다 |
| `KIWI` · `kiwi_nouns` · `read_complaints` · `sample_rows` | 미션 1 과 같은 형태소 분석기와 함수 | 그대로 호출한다 |

#### 함께 볼 것

1. 환불 · 배송의 어절 대 형태소: 집계 단위를 바꾸자 몇 배가 되었는가
2. 「해지」 글자는 있는데 `NOUNS_20K` 에는 「해지」가 없는 제목 수와 그 원문 3건

#### 프롬프트 힌트

- **고칠 곳**: `w in t.split()` 이 빠뜨리는 제목과, 무엇으로 바꿀지.

**막히면**: AI 튜터에게 「"헬스장 계약해지 관련 민원" 을 공백으로 자르면 "해지" 와 똑같은 어절이 있나요」 처럼 질문합니다.

<details class="lms-extra"><summary>자세히 — AI 튜터에 맡길 일과 내가 확인할 일</summary>

| AI 튜터에 맡길 일 | 내가 반드시 확인할 일(틀려도 오류가 나지 않는 곳) |
|---|---|
| 낱말 5개의 두 비율과 표 | 형태소 쪽을 제목 문자열이 아니라 `NOUNS_20K` 에서 집계했는가. `w in t`(t 는 제목 문자열)로 글자가 있는지만 보면 「배송비」 안의 배송까지 들어가도 오류가 나지 않는다 |
| 「형태소 순위」 · 「판단 :」 줄 | 둘 다 형태소로 집계한 두 값을 비교한 결과인가. 결론 문장만 직접 바꾸면 집계 방법은 그대로 남는다 |
| 어절 · 형태소 막대 | 어절 막대와 형태소 막대가 같은 낱말 순서인가. 순서가 어긋나 막대가 엉뚱한 낱말 옆에 그려져도 오류가 나지 않는다 |

</details>

#### 틀린 곳 찾기

전임 코드에서 낱말 5개의 문서 수를 집계하는 한 줄을 봅니다.

> **[예측] AI 코드 표시선 아래 코드는 오류 없이 실행되지만 「위약금 민원이 해지 민원보다 많다」는 결론은 믿을 수 없습니다. 틀린 곳은 어디이고 왜일까요?**
>
> 1. `w in t.split()`: 공백으로 자른 어절이 낱말과 똑같은 제목만 집계해 「계약해지」 · 「해지요청」 같은 어절을 빠뜨린다
> 2. `C20K`: 표본 2만 건은 50만 건의 4% 뿐이라 두 낱말의 순위를 믿을 수 없다
> 3. `FIVE_WORDS`: 낱말 5개 목록이 틀려 해지를 쓴 제목이 빠졌다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>예상 확인 — 「PROMPT 셀」을 그대로 실행한 뒤 펼칩니다</summary>

답은 첫째 보기입니다. 고치면 해지는 3.52% 에서 10.72% 로 커져 위약금 5.10% 의 두 배를 넘습니다.

- `C20K`: 50만 건에서도 순위가 똑같이 반대라 표본 크기 탓이 아닙니다.
- `FIVE_WORDS`: 목록에 「해지」가 있습니다. 빠진 것은 「계약해지」 같은 어절의 제목입니다.

</details>

In [ ]:
# 미션 2 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 2 에 쓸 이름을 직접 만듭니다
#  미션 1 과 같은 함수로 민원 파일을 읽고, 같은 표본 2만 건(시드 0)과 그 명사 목록을 다시 만듭니다.
#  앞 미션의 결과에 기대지 않습니다. 원자료를 받아 둔 뒤에는 5초 안팎입니다(읽기 2초 · 형태소 분석 2만 건 2.5초)
import hashlib
import math
import os
import re
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal

import pandas as pd
from kiwipiepy import Kiwi

# ---------------------------------------------------------------- 원 출처 (조회 2026-09-24)
RAW_COMPLAINTS_URL = ("https://www.data.go.kr/cmm/cmm/fileDownload.do?"
                      "atchFileId=FILE_000000002610054&fileDetailSn=1&insertDataPrcus=N")
RAW_REVIEWS_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
RAW_FILES = {  # 이름: (원 출처 URL, 내 컴퓨터에 둘 파일 이름, 교안의 수를 계산한 파일의 SHA-256, 데이터 페이지)
    "complaints": (RAW_COMPLAINTS_URL, "kftc_complaints_1.csv",
                   "50c098fea2ec1d05191d07c45cd0f75b4d0ac3b5e5ac36910fe40f2aab699ebe",
                   "https://www.data.go.kr/data/15098314/fileData.do"),
    "reviews": (RAW_REVIEWS_URL, "naver_shopping.txt",
                "dc4d1aca0a148671cbe80bcb81962eee297370acab42be93c1617ce9336479c0",
                "https://github.com/bab2min/corpus/tree/master/sentiment"),
}
MIRROR_URL = "https://github.com/studio-js/korean-cx-text/raw/main/raw"  # 과정 복제본: 원 출처와 같은 바이트, 먼저 받는 곳
SUBDIRS = ("", "cache", "raw", "processed")  # 자료 폴더 아래에서 찾는 순서


def data_dir():
    # 자료 폴더: 환경변수 NLP_DATA_DIR 이 있으면 그 폴더, 없으면 ./nlp_data
    return os.environ.get("NLP_DATA_DIR", "nlp_data")


def find_local(name):
    # 자료 폴더 → cache/ → raw/ → processed/ 순서로 찾아 처음 나온 경로, 없으면 None
    for sub in SUBDIRS:
        path = os.path.join(data_dir(), sub, name)
        if os.path.isfile(path):
            return path
    return None


def sha256_of(path, chunk=1 << 20):
    # 파일의 SHA-256 16진 문자열 — 받은 파일이 교안의 파일과 같은지 대조합니다
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _download(url, path, retries=3):
    # url 을 path.part 로 받아 두고 성공하면 True (실패하면 세 번까지 다시 시도)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    for i in range(retries):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            return True
        except Exception as e:  # network error: retry
            print(f"받기 실패 {i + 1}/{retries}: {e}")
            time.sleep(2 * (i + 1))
    return False


def _fetch(url, name):
    # 자료 폴더에 있으면 그 경로, 없으면 url 에서 받아 자료 폴더/cache/ 에 두고 경로를 반환합니다(받지 못하면 None)
    path = find_local(name)
    if path is None:
        path = os.path.join(data_dir(), "cache", name)
        print(f"{name} 을 받습니다 (처음 한 번만)")
        if not _download(url, path):
            return None
        os.replace(path + ".part", path)
    return path


def fetch_raw(name):
    # 원자료 경로: 자료 폴더를 먼저 보고, 없으면 과정 복제본(MIRROR_URL) → 원 출처 URL 순서로 받습니다
    url, fname, sha, page = RAW_FILES[name]
    path = find_local(fname)
    if path is None:
        path = _fetch(f"{MIRROR_URL}/{fname}", fname)
        if path is not None and sha256_of(path) != sha:  # 복제본이 교안의 파일과 다르면 지우고 원 출처에서 받습니다
            os.remove(path)
            path = None
    if path is None:
        path = _fetch(url, fname)
    if path is None:
        raise RuntimeError(f"원 출처에서 파일을 받지 못했습니다. 데이터 페이지: {page}")
    if sha256_of(path) != sha:
        print(f"원 출처 파일이 바뀌었습니다. 결과 수가 교안과 다를 수 있습니다(데이터 페이지: {page})")
    return path


# ---------------------------------------------------------------- 반올림 (교안의 수와 같게)
def rnd(x, digits):
    # 십진 값 기준 반올림(마지막 숫자가 5 면 올림) — 파이썬 round 는 4.045 를 4.04 로 적습니다
    x = float(x)
    if not math.isfinite(x):
        return x
    return float(Decimal(f"{x:.12g}").quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


def pct2(k, n, digits=2):
    # 문서 k 건 ÷ 전체 n 건을 백분율로, 소수 둘째 자리: pct2(809, 20000) -> 4.05
    return rnd(100 * k / n, digits)


def half95(k, n, digits=2):
    # 비율 k/n 의 95% 구간 오차 한계(구간 길이의 절반, ± 뒤의 값)를 %p 로: 1.96 × √(p̂(1 − p̂)/n) × 100
    p = k / n
    return rnd(100 * 1.96 * math.sqrt(p * (1 - p) / n), digits)


# ---------------------------------------------------------------- 민원 파일 읽기
# 상담 시스템에 시험 삼아 입력한 행(소비자 민원이 아님)을 찾는 정규식
TEST_INPUT_RE = re.compile(
    r"(?i)^\s*(?:test|테스트)\s*(?:입니다)?\s*\.?\s*$"
    r"|IN-TEST"
    r"|(?:인바운드|아웃바운드|녹취|수화기).*테스트|테스트.*(?:인바운드|아웃바운드|녹취|수화기)"
    r"|^\s*자동차\s*(?:상담\s*)?테스트"
    r"|^\s*\d{8}\s*[-_ ].*테스트"
    r"|테스트\s*데이터|상담\s*(?:전화\s*)?테스트|전화\s*테스트|헬프데스크\s*시험|^\s*일반\s*테스트")
EMPTY_TITLE_RE = re.compile(r"^[^가-힣A-Za-z]*$")  # 한글 음절도 로마자도 없는 제목(「-」「.」「0」「ㅇ」)
PHONE_RE = re.compile(r"(?<!\d)(?:01[016789]|0\d{1,2})[-.)\s]{0,2}\d{3,4}[-.\s]\d{4}(?!\d)")  # 전화번호 형태
PHONE_MASK = "[전화번호]"  # 원 출처가 이미 가린 번호와 같은 표지


def mask_phones(text):
    # 전화번호 형태를 [전화번호] 로 가림
    return PHONE_RE.sub(PHONE_MASK, text)


def tidy_columns(df):
    # 열 이름 정리: '사건제목(ACCIDENT_TITLE)' -> '사건제목'
    return df.rename(columns=lambda c: c.split("(")[0].strip())


def read_complaints(path, verbose=True):
    # 민원 파일(cp949) 읽기: 시험 입력·빈 제목 행 제거, 전화번호 형태 가림, 열 이름 정리, 날짜 변환
    raw = pd.read_csv(path, encoding="cp949", dtype=str)
    df = tidy_columns(raw)
    df.insert(0, "row_id", range(len(df)))  # 원 파일의 행 번호(0부터)
    title = df["사건제목"].fillna("").str.strip()
    is_test = title.str.contains(TEST_INPUT_RE)
    is_empty = title.str.match(EMPTY_TITLE_RE) & ~is_test
    df = df[~is_test & ~is_empty].copy()
    title = title[df.index]
    df["사건제목"] = title.map(mask_phones)
    df["번호가림"] = df["사건제목"].ne(title)  # 전화번호 형태를 가린 행이면 True
    df["접수일자"] = pd.to_datetime(df["접수일자"], format="%Y-%m-%d")
    df = df.reset_index(drop=True)
    if verbose:
        print(f"민원 원본 {len(raw):,}행 → 시험 입력 {int(is_test.sum())}행·빈 제목 {int(is_empty.sum())}행 제거 "
              f"→ {len(df):,}행, 전화번호 형태 가림 {int(df['번호가림'].sum())}행")
    return df


# ---------------------------------------------------------------- 표본과 형태소
def sample_rows(df, n, seed=0):
    # 고정 표본: 같은 입력 행과 같은 seed 면 같은 행이 같은 순서로 나옵니다
    return df.sample(n=n, random_state=seed)


NOUN_TAGS = ("NNG", "NNP")  # 일반 명사 + 고유 명사 (Kiwi 기본 사전)


def kiwi_nouns(texts, kiwi):
    # 문서마다 명사(NNG·NNP) 목록 — 나온 순서와 반복을 그대로 둡니다(문서 수를 집계할 때는 set 으로)
    return [[t.form for t in toks if t.tag in NOUN_TAGS] for toks in kiwi.tokenize(list(texts))]


COMPLAINTS_PATH = fetch_raw("complaints")   # 민원 원본 파일 경로 (cp949)
import os
os.environ.setdefault("KIWI_ARCH_TYPE", "balanced")  # CPU 종류(Mac·Colab)와 상관없이 형태소 분석 결과를 같게
KIWI = Kiwi(num_workers=2)                  # 형태소 분석기(기본 사전, 작업자 2)

# 쓸 이름 — 미션 1 과 같은 표본 2만 건(시드 0)과, 행 순서가 같은 명사 목록
C20K = sample_rows(read_complaints(COMPLAINTS_PATH), 20000, seed=0)[["row_id", "품목명", "사건제목"]]
C20K = C20K.reset_index(drop=True)
NOUNS_20K = kiwi_nouns(C20K["사건제목"], KIWI)   # NOUNS_20K[i] = C20K 의 i 번째 제목의 명사 목록
FIVE_WORDS = ["환불", "해지", "위약금", "배송", "취소"]   # 전임 담당자가 비교한 다섯 낱말

print(f"표본 C20K : {len(C20K):,}건 · 열 {list(C20K.columns)}")
print(f"명사 목록 NOUNS_20K : {len(NOUNS_20K):,}개 — 첫 제목 「{C20K['사건제목'][0]}」 → {NOUNS_20K[0]}")
print("준비 완료 — C20K · NOUNS_20K · FIVE_WORDS · pct2 · KIWI · kiwi_nouns · read_complaints · sample_rows")

In [ ]:
# 미션 2 — 프롬프트 (함정)   # ai: prompt
# 이 미션은 함정입니다 — 아래는 전임 담당자가 남긴 코드라 오류 없이 실행되지만 출력한 표로 내린 결론은 믿을 수 없습니다.
# 고칠 지시는 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 튜터가 AI 코드 표시선 아래를 다시 씁니다.
# 따옴표 안이 비어 있으면 아래 코드의 채점만 나오고 완성본 풀이는 잠깁니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---
# ↓ 전임 담당자의 코드 — 이 코드가 만든 표로 「위약금 민원이 해지 민원보다 많다」고 판단했습니다. 그대로 실행하면 채점이 ❌ 를 냅니다
titles = C20K["사건제목"]
share = {w: pct2(sum(w in t.split() for t in titles), len(C20K)) for w in FIVE_WORDS}   # 낱말마다 그 낱말을 쓴 제목의 비율
print(f"민원 제목 표본 {len(C20K):,}건 · 다섯 낱말의 문서 비율")
for w in FIVE_WORDS:
    print(f"  {w} · {share[w]:.2f} %")
print(f"판단 : 위약금 민원({share['위약금']:.2f}%)이 해지 민원({share['해지']:.2f}%)보다 많다")

fig, ax = plt.subplots(figsize=(FIG_W * 0.6, 3.4))
ax.barh(FIVE_WORDS, [share[w] for w in FIVE_WORDS], color=COL["연회색"])
ax.invert_yaxis()
ax.set_xlabel("문서 비율(%)")
ax.set_title(f"다섯 낱말의 문서 비율 — 민원 제목 {len(C20K):,}건")
plt.show()

> **[문제] 미션 2 의 고친 코드는 `NOUNS_20K` 로 집계해 배송이 5.58% 였고, 제목에 「배송」 글자가 있는지로 집계한 코드는 6.15% 였습니다. 두 값이 다른 이유는?**
>
> 1. 글자가 있는지로 집계하면 「배송비」처럼 다른 명사 안에 있는 「배송」까지 들어간다
> 2. 형태소 분석이 표본의 제목 수를 줄인다
> 3. 두 코드의 반올림 방식이 다르다

## 미션 3 — 리뷰 잡음을 정리하고 품사 선택 두 설정 비교

#### 할 일 — 미션 3 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 시킬 일을 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다.
3. 아래 「먼저 예상해 보기」에서 하나를 고르고 근거를 한 문장 적습니다.
4. AI 튜터에게 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶ 합니다. 5초 안팎 걸립니다.
5. 채점 결과를 봅니다. ❌ 면 나온 항목대로 PROMPT 를 고칩니다.

**이 미션의 질문**: 별점 1·2점 쇼핑 리뷰 1만 건을 정리해 명사만 표와 명사 + 용언 표를 만듭니다.
두 표의 상위 15 는 몇 개가 바뀔까요? 어느 설정이 「무엇에 대한 불만인가」에 답할까요?

이론 「2. 고객 글의 잡음」의 정리 방법을 코드로 옮기고 「3. 어절 · 형태소 · 품사 선택」의 두 설정을 상위 15 로 비교합니다.

#### 만들 것

아래 7개를 모두 「PROMPT 셀」에서 출력합니다. 굵은 이름은 채점이 찾으니 글자 그대로 씁니다.

1. **자모 나열 건수**: 단위 건 · 정수
   - 정리 전 본문으로 만든 「자모나열」 열이 True 인 리뷰 수입니다
2. **자모나열 열 값**: 그 열에 있는 서로 다른 값의 목록
3. **명사만 표 행 수**: 단위 행
   - 명사만 목록의 문서 비율 상위 15 표입니다. 분모는 10,000 이고 리뷰마다 목록을 `set` 으로 바꿉니다
4. **명사 + 용언 표 행 수**: 단위 행
   - 같은 방법으로 만든 명사 + 용언 상위 15 표입니다. 용언은 어간에 「다」를 더합니다
5. **들어오고 나간 낱말 수**: 단위 개
   - 명사 + 용언 표에만 있는 낱말의 개수입니다. 두 표 모두 낱말 15개라 들어온 수와 나간 수가 같습니다
6. **명사 + 용언 1위**: 낱말과 문서 비율(%)을 한 줄에
7. **그래프 한 장**: 종류는 자유이고 가로축 「문서 비율(%)」, 세로축 「낱말」 이름을 답니다
   - 권장은 두 설정의 상위 15 를 나란히 그린 가로 막대그래프 두 개입니다

**통과 기준**: 1～6번 값이 「이름 = 값」 형식으로 기준 범위 안에 있고 그래프에 가로축·세로축 이름이 있으면 통과입니다.

**잡음 정리**: 아래 표의 차례대로 씁니다.

| 잡음 | 찾는 조건 | 다루는 법 |
|---|---|---|
| 자모 나열 | 자모(ㄱ～ㅣ, ㆍ)만 4개 이상 이어진 구간에 서로 다른 자음이 4개 이상 | 지우기 전 본문에서 자모 나열이 있던 리뷰를 「자모나열」 열로 기록하고, 그 구간을 지운다 |
| 반복 자모 | 같은 자모가 둘 이상 이어짐(ㅋㅋ · ㅠㅠ) | 셋 이상 이어지면 두 글자로 줄여 남긴다 |
| 띄어쓰기 오류 | 띄어 쓰지 않은 어절(「구매할수있었어요」) | 고치지 않는다. 공백이 둘 이상 이어지면 하나로만 줄인다 |

**쓸 수 있는 것**: 「준비 셀」의 변수와 함수입니다.

| 이름 | 무엇 | 쓰는 법 |
|---|---|---|
| `LOW10K` | 별점 1·2점 리뷰 가운데 무작위 1만 건(시드 0). 열은 `row_id`(원 파일 행 번호) · `별점` · `본문`이고, 본문은 쓴 그대로다 | 다시 만들지 않는다 |
| `KIWI` | Kiwi 형태소 분석기(기본 사전, 문서를 동시에 두 개씩 분석하는 `num_workers=2`). `KIWI.tokenize(목록)` 은 문서마다 형태소 목록을 만든다. 형태소마다 글자 `form` 과 품사 `tag` 가 있다 | 다시 만들지 않는다 |
| `NOUN_TAGS` | 명사만 설정의 품사 `("NNG", "NNP")`: 일반 명사 · 고유 명사 | 다시 만들지 않는다 |
| `PRED_TAGS` | 용언의 품사 `("VV", "VA")`: 동사 · 형용사. Kiwi 는 「VA-I」 처럼 태그 뒤에 「-」 와 글자를 더 적기도 하므로 「-」 앞만 비교한다 | 다시 만들지 않는다 |
| `pct2(k, n)` | k ÷ n 의 백분율을 소수 둘째 자리까지 구한다. 반올림은 교안과 같은 방식이다 | 그대로 호출한다 |
| `read_reviews` · `sample_rows` | 리뷰 파일을 읽는 함수와 시드를 고정한 무작위 n행 | 그대로 호출한다 |

#### 함께 볼 것

1. 두 설정의 상위 15: 순위 · 낱말 · 리뷰 수 · 문서 비율
2. 들어온 낱말과 나간 낱말 목록
3. 들어온 「좋다」와 나간 「맛」을 쓴 리뷰 원문 3건씩: 별점 1·2점 리뷰에 「좋다」가 나오는 이유

#### 프롬프트 힌트

- **이미 있는 변수와 함수**: 준비 셀의 변수는 다시 만들지 말라고 적습니다.
- **정리 순서**: 「`자모나열` 열을 정리 전 본문으로 먼저 만들고 그다음 자모 나열 구간을 지워 줘」.
- **Kiwi 한 번**: 「정리한 본문을 `KIWI.tokenize` 로 한 번만 나누고 그 결과에서 두 목록을 함께 만들어 줘」.

**막히면**: AI 튜터에게 「"좋지 않아요" 를 Kiwi 로 나누면 명사 + 용언 설정에는 어떤 낱말이 남나요」 처럼 질문합니다.

<details class="lms-extra"><summary>배경 — 자모 나열에 자음이 몰리는 이유</summary>

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1d/KB_South_Korea.svg" alt="한국 표준 두벌식 자판. 자음 ㅂㅈㄷㄱㅅ · ㅁㄴㅇㄹㅎ · ㅋㅌㅊㅍ 이 왼쪽 세 줄에, 모음이 오른쪽에 있는 배열" width="560" height="187"/>
<p align="center"><em>▲ 한국 표준 두벌식 자판. 자음은 왼쪽, 모음은 오른쪽 키에 있다. 이론에서 본 별점 2점 리뷰의 「ㅇㄷㅇㅈㅇㄷㄱㅌㅅ」 9글자는 모두 왼쪽 자음 키(D · E · W · R · X · T)에서 나온다 (출처: Wikimedia Commons, KB South Korea.svg, Yes0song, CC BY-SA 3.0)</em></p>

자판 왼쪽만 누르면 자음만 이어져 음절이 되지 않습니다.

</details>

<details class="lms-extra"><summary>자세히 — AI 튜터에 맡길 일과 내가 확인할 일</summary>

| AI 튜터에 맡길 일 | 내가 반드시 확인할 일(틀려도 오류가 나지 않는 곳) |
|---|---|
| 정리 함수와 「자모나열」 열 | 열을 **지우기 전 본문으로** 만들었는가. 정리한 본문으로 만들면 자모 나열이 이미 지워져 모두 False 가 되어도 오류가 나지 않는다 |
| Kiwi 두 목록 | 태그를 「-」 앞만 비교했는가. `t.tag in PRED_TAGS` 로 바로 비교하면 받(VV-R) · 아쉽(VA-I) 처럼 태그에 「-」 가 있는 용언 형태소가 빠진다. 이 1만 건에서는 빠진 낱말이 상위 15 안에 없어 표가 그대로이므로, 오류도 표의 차이도 나지 않는다 |
| 상위 15 두 표와 막대 | 리뷰마다 `set` 으로 바꿨는가, 분모가 10,000 인가. 빼면 「하다」를 두 번 쓴 리뷰가 2 로 집계된다 |

</details>

#### 먼저 예상해 보기

> **[예측] 이론에서 상위 10 을 대조했을 때는 10개 가운데 8개가 바뀌었습니다. 상위 15 로 늘려 대조하면 바뀌는 낱말은 몇 개일까요?**
>
> 1. 8개보다 적다
> 2. 8개 그대로다
> 3. 8개보다 많다
>
> 답을 정한 뒤 아래를 읽어 보세요(정답 채점은 없습니다).

<details class="lms-extra"><summary>예상 확인 — 미션 3 결과를 본 뒤 펼칩니다</summary>

답은 셋째 보기입니다. 두 표에 함께 남은 낱말이 배송 · 구매 · 생각 · 제품 4개라 15 − 4 = 11개가 바뀝니다.

| 명사 + 용언 상위 15 의 낱말 | 개수 | 명사만 상위 15 에 있는가 |
|---|---:|---|
| 1～10위 용언: 하다 · 쓰다 · 되다 · 같다 · 좋다 · 없다 · 사다 · 오다 | 8 | 없음 · 바뀜 |
| 1～10위 명사: 배송 · 구매 | 2 | 있음 |
| 11～15위 용언: 나다 · 있다 · 먹다 | 3 | 없음 · 바뀜 |
| 11～15위 명사: 생각 · 제품 | 2 | 있음 |

- 「8개보다 적다」 · 「8개 그대로다」: 11～15위의 용언 3개가 더해져 8 + 3 = 11 입니다.

</details>

In [ ]:
# 미션 3 · 준비 — 읽고 실행만 합니다. 이 셀이 미션 3 에 쓸 이름을 직접 만듭니다
#  리뷰 원본을 과정 복제본(원 출처와 같은 바이트)에서 받아 두고(처음 한 번만), 별점 1·2점 리뷰 무작위 1만 건(시드 0)을 뽑아 둡니다.
#  잡음 정리와 형태소 분석은 하지 않습니다 — 그 일은 「PROMPT 셀」에서 합니다.
#  앞 미션의 결과에 기대지 않습니다. 원자료를 받아 둔 뒤에는 1초 안팎입니다
import hashlib
import math
import os
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal

import pandas as pd
from kiwipiepy import Kiwi

# ---------------------------------------------------------------- 원 출처 (조회 2026-09-24)
RAW_REVIEWS_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
RAW_FILES = {  # 이름: (원 출처 URL, 내 컴퓨터에 둘 파일 이름, 교안의 수를 계산한 파일의 SHA-256, 데이터 페이지)
    "reviews": (RAW_REVIEWS_URL, "naver_shopping.txt",
                "dc4d1aca0a148671cbe80bcb81962eee297370acab42be93c1617ce9336479c0",
                "https://github.com/bab2min/corpus/tree/master/sentiment"),
}
MIRROR_URL = "https://github.com/studio-js/korean-cx-text/raw/main/raw"  # 과정 복제본: 원 출처와 같은 바이트, 먼저 받는 곳
SUBDIRS = ("", "cache", "raw", "processed")  # 자료 폴더 아래에서 찾는 순서


def data_dir():
    # 자료 폴더: 환경변수 NLP_DATA_DIR 이 있으면 그 폴더, 없으면 ./nlp_data
    return os.environ.get("NLP_DATA_DIR", "nlp_data")


def find_local(name):
    # 자료 폴더 → cache/ → raw/ → processed/ 순서로 찾아 처음 나온 경로, 없으면 None
    for sub in SUBDIRS:
        path = os.path.join(data_dir(), sub, name)
        if os.path.isfile(path):
            return path
    return None


def sha256_of(path, chunk=1 << 20):
    # 파일의 SHA-256 16진 문자열 — 받은 파일이 교안의 파일과 같은지 대조합니다
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _download(url, path, retries=3):
    # url 을 path.part 로 받아 두고 성공하면 True (실패하면 세 번까지 다시 시도)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    for i in range(retries):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            return True
        except Exception as e:  # network error: retry
            print(f"받기 실패 {i + 1}/{retries}: {e}")
            time.sleep(2 * (i + 1))
    return False


def _fetch(url, name):
    # 자료 폴더에 있으면 그 경로, 없으면 url 에서 받아 자료 폴더/cache/ 에 두고 경로를 반환합니다(받지 못하면 None)
    path = find_local(name)
    if path is None:
        path = os.path.join(data_dir(), "cache", name)
        print(f"{name} 을 받습니다 (처음 한 번만)")
        if not _download(url, path):
            return None
        os.replace(path + ".part", path)
    return path


def fetch_raw(name):
    # 원자료 경로: 자료 폴더를 먼저 보고, 없으면 과정 복제본(MIRROR_URL) → 원 출처 URL 순서로 받습니다
    url, fname, sha, page = RAW_FILES[name]
    path = find_local(fname)
    if path is None:
        path = _fetch(f"{MIRROR_URL}/{fname}", fname)
        if path is not None and sha256_of(path) != sha:  # 복제본이 교안의 파일과 다르면 지우고 원 출처에서 받습니다
            os.remove(path)
            path = None
    if path is None:
        path = _fetch(url, fname)
    if path is None:
        raise RuntimeError(f"원 출처에서 파일을 받지 못했습니다. 데이터 페이지: {page}")
    if sha256_of(path) != sha:
        print(f"원 출처 파일이 바뀌었습니다. 결과 수가 교안과 다를 수 있습니다(데이터 페이지: {page})")
    return path


# ---------------------------------------------------------------- 반올림 (교안의 수와 같게)
def rnd(x, digits):
    # 십진 값 기준 반올림(마지막 숫자가 5 면 올림) — 파이썬 round 는 4.045 를 4.04 로 적습니다
    x = float(x)
    if not math.isfinite(x):
        return x
    return float(Decimal(f"{x:.12g}").quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


def pct2(k, n, digits=2):
    # 문서 k 건 ÷ 전체 n 건을 백분율로, 소수 둘째 자리: pct2(1874, 10000) -> 18.74
    return rnd(100 * k / n, digits)


# ---------------------------------------------------------------- 리뷰 파일 읽기
def read_reviews(path):
    # 리뷰 파일(UTF-8, 한 행 = 「별점<탭>본문」) 읽기 — 본문은 쓴 그대로 둡니다(잡음 정리는 PROMPT 셀에서)
    rows = [line.split("\t", 1) for line in open(path, encoding="utf-8").read().split("\n") if line]
    df = pd.DataFrame(rows, columns=["별점", "본문"])
    df.insert(0, "row_id", range(len(df)))  # 원 파일의 행 번호(0부터)
    df["별점"] = df["별점"].astype(int)
    return df


def sample_rows(df, n, seed=0):
    # 고정 표본: 같은 입력 행과 같은 seed 면 같은 행이 같은 순서로 나옵니다
    return df.sample(n=n, random_state=seed)


NOUN_TAGS = ("NNG", "NNP")  # 명사만: 일반 명사 + 고유 명사
PRED_TAGS = ("VV", "VA")    # 용언: 동사 + 형용사 — Kiwi 는 「VA-I」 처럼 태그 뒤에 「-」 와 글자를 더 적기도 하니 「-」 앞만 비교합니다

REVIEWS_PATH = fetch_raw("reviews")   # 리뷰 원본 파일 경로 (UTF-8)
REVIEWS = read_reviews(REVIEWS_PATH)
import os
os.environ.setdefault("KIWI_ARCH_TYPE", "balanced")  # CPU 종류(Mac·Colab)와 상관없이 형태소 분석 결과를 같게
KIWI = Kiwi(num_workers=2)            # 형태소 분석기(기본 사전, 작업자 2)

# 쓸 이름 — 별점 1·2점 리뷰 가운데 무작위 1만 건(시드 0). 행 순서는 뽑힌 순서 그대로입니다
LOW10K = sample_rows(REVIEWS[REVIEWS["별점"].isin([1, 2])], 10000, seed=0).reset_index(drop=True)

print(f"리뷰 원본 {len(REVIEWS):,}행 · 별점 1·2점 {int(REVIEWS['별점'].isin([1, 2]).sum()):,}행")
print(f"표본 LOW10K : {len(LOW10K):,}건 · 열 {list(LOW10K.columns)} · 별점 {sorted(LOW10K['별점'].unique().tolist())}")
print(f"첫 행 : 행 {LOW10K.at[0, 'row_id']} · 별점 {LOW10K.at[0, '별점']} · {LOW10K.at[0, '본문'][:40]}")
print("준비 완료 — LOW10K · KIWI · NOUN_TAGS · PRED_TAGS · pct2 · read_reviews · sample_rows")

In [ ]:
# 미션 3 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

> **[문제] 미션 3 의 명사 + 용언 설정에서 「좋지 않아요」와 「좋아요」가 똑같이 「좋다」 하나로 집계되는 이유는?**
>
> 1. Kiwi 가 부정문을 지우고 긍정문만 남긴다
> 2. 반복 자모 정리가 「않」을 지운다
> 3. 부정을 만드는 「않」이 보조 용언(VX)이라 VV · VA 로 고른 설정에서 빠진다

## 과제 — 국외여행 민원 키워드 표

> **[과제] 과제 제출 — 국외여행 민원 키워드 표**
>
> - **무엇을** 문서 비율 상위 15 와 95% 구간이 있는 국외여행 민원 키워드 표와, 사용자 사전 · 불용어 처리 전후의 순위 대조를 만듭니다.
> - **무엇으로** 민원 파일 1 의 국외여행 13,012건 전체를 쓰고, 원자료 읽기와 사용자 사전 후보 · 불용어 목록은 「준비 셀」이 만듭니다.
> - **인정** 채점이 값과 그림을 모두 통과시킨 노트북에 키워드 표 파일과 브리핑 세 단락을 갖춘 것입니다.
> 
> - 「PROMPT 셀」을 실행해 값과 그림 한 장이 채점을 통과한 상태로 남기고, 그 셀의 PROMPT 는 내가 쓴 그대로 둡니다.
> - 사용자 사전 · 불용어 처리 뒤 상위 15 를 5열로 저장한 day66_keyword_table.csv 를 함께 제출합니다.
> - 처리 전후 두 표에서 순위가 바뀐 낱말과, 사전에 추가하지 않았는데 문서 수가 바뀐 낱말을 하나 이상 적습니다.
> - 처리 뒤 표 상위 낱말을 포함한 원문 2건과 무작위 원문 3건을 행 번호와 함께 인용하고, 판매처 이름은 ○○ 로 씁니다.
> - 브리핑 세 단락을 과제 끝의 양식에 결론 · 위험 · 코드가 답을 만드는 방식 순으로 채웁니다.
>
> 마감: 2026-11-03T23:59+09:00
>
> **평가 기준**
>
> | 기준 | 이렇게 하면 충족 | 배점 |
> | --- | --- | --- |
> | 문제 정의 | PROMPT 에 대상 품목 · 사용자 사전 세 낱말 · 불용어 아홉 개 · 문서 비율의 분모가 내 말로 적혀 있다. | 25 |
> | 검증 설계 | 값과 그림이 채점을 통과했고, 저장한 키워드 표를 다시 읽어 15행 5열과 불용어가 없음을 확인했다. | 25 |
> | 오류 탐지 | 사전에 추가하지 않았는데 문서 수가 바뀐 낱말을 전후 표에서 찾았고, 그 낱말의 문서 수가 몇 건에서 몇 건이 됐는지 적었다. | 25 |
> | 해석·보고 | 결론 문장이 「가장 많이 말하는 것」을 문서 비율과 구간으로 답하고, 원문 다섯 건과 어긋나지 않는다. | 25 |
>
> 제출은 LMS 레슨 화면의 과제 카드에서 GitHub 또는 Google Drive 링크로 합니다(이 파일에 적으면 제출되지 않습니다).

#### 할 일 — 과제 (25분)

1. 아래 「준비 셀」을 실행합니다. 이 셀은 고치지 않습니다.
2. 아래 「만들 것」대로 시킬 일을 「PROMPT 셀」의 `PROMPT = """ """` 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다.
3. AI 튜터에게 지시하고 ▶ 합니다. 10초 안팎 걸립니다. ❌ 면 힌트대로 PROMPT 를 고칩니다.
4. 키워드 표 파일을 저장하고 과제 끝의 양식에 브리핑 세 단락을 채워 제출합니다.

**이 과제의 질문**: 2020년 국외여행 민원 제목은 무엇을 가장 많이 말할까요?
사용자 사전과 불용어를 처리하면 상위 15 는 어떻게 달라질까요?

Kiwi 기본 사전은 「취소수수료」를 취소 + 수수료로 쪼갭니다. 이런 낱말을 **사용자 사전**(한 낱말로 나누게 내가 더하는 낱말 목록)에 추가하면, 같은 제목의 다른 낱말 문서 수도 바뀝니다. 「취소수수료」를 추가하면 「취소」 문서 수가 줍니다. 그래서 처리 전후 두 표를 비교합니다.

**쓸 수 있는 것**: 「준비 셀」의 변수와 함수입니다.

| 이름 | 무엇 | 쓰는 법 |
|---|---|---|
| `OVERSEAS` | 품목이 국외여행인 민원 전체(시험 입력 · 빈 제목을 거른 뒤). 열은 `row_id`(원 파일 행 번호) · `접수일자` · `사건제목` | 다시 만들지 않는다 |
| `Kiwi` | Kiwi 형태소 분석기를 만드는 클래스. `Kiwi(num_workers=2)` 로 새로 만들고, `add_user_word("항공권", "NNG")` 로 사용자 사전에 낱말을 추가한다 | 그대로 호출한다 |
| `kiwi_nouns(texts, kiwi)` | 제목마다 명사(NNG · NNP) 목록을 만든다. 한 제목 안의 순서와 반복을 그대로 둔다 | 그대로 호출한다 |
| `USER_WORDS` | 사용자 사전에 추가할 세 낱말 `["항공권", "우한폐렴", "취소수수료"]` | 다시 만들지 않는다 |
| `FORM_WORDS` | 불용어 9개: 문의 · 요청 · 관련 · 요구 · 건 · 후 · 상담 · 불만 · 피해 | 다시 만들지 않는다 |
| `pct2(k, n)` · `half95(k, n)` | k ÷ n 의 백분율과 95% 구간의 오차 한계(± 뒤의 값, %p)를 소수 둘째 자리까지 구한다. 반올림은 교안과 같은 방식이다 | 그대로 호출한다 |
| `TABLE_FILE` · `TABLE_COLS` | 키워드 표 파일 이름 `day66_keyword_table.csv` · 열 이름 5개(`낱말, 문서 수, 문서 비율, 95% 반길이, 처리 전 순위`) | 다시 만들지 않는다 |

#### 만들 것

「PROMPT 셀」 한 곳에서 아래 값 7개를 출력합니다.

| 이름 | 단위 | 무엇 |
|---|---|---|
| 국외여행 행 수 | 건 | `OVERSEAS` 의 행 수. 모든 문서 비율의 분모다 |
| 처리 전 취소 문서 수 | 건 | 기본 사전 그대로 만든 Kiwi 의 명사 목록에 「취소」가 있는 제목 수 |
| 처리 뒤 취소 문서 수 | 건 | 새로 만든 Kiwi 에 `USER_WORDS` 세 낱말을 추가한 뒤 같은 방법으로 집계한 제목 수 |
| 키워드 표 행 수 | 행 | 저장한 키워드 표 파일을 다시 읽어 집계한 행 수 |
| 키워드 표 열 수 | 열 | 같은 파일의 열 수 |
| 처리 뒤 표의 불용어 수 | 개 | 다시 읽은 표의 낱말 열에 있는 `FORM_WORDS` 의 개수 |
| 처리 뒤 1위 | % | 키워드 표 1위 낱말과 그 문서 비율을 한 줄에 |

**키워드 표** `day66_keyword_table.csv`: 처리 뒤 목록에서 불용어를 뺀 문서 비율 상위 낱말 15개이고 열은 `TABLE_COLS` 5개입니다.

- 문서 비율은 제목마다 `set` 으로 바꿔 집계하고 분모는 국외여행 행 수입니다
- `95% 반길이` 열은 `half95` 로 구한 오차 한계입니다. 처리 전 순위는 불용어를 포함한 처리 전 명사 전체의 순위입니다. 처리 전에 없던 낱말은 0 입니다

**그래프 한 장**: 종류는 자유이고 가로축 「문서 비율(%)」, 세로축 「낱말」 이름을 답니다. 권장은 가로 막대 15개와 `xerr` 로 그린 95% 구간입니다.

**통과 기준**: 값 7개가 「이름 = 값」 형식으로 기준 범위 안에 있고 키워드 표가 이번 실행에서 15행 5열로 저장돼야 합니다. 그래프에도 가로축·세로축 이름이 있으면 통과입니다.

#### 함께 볼 것

1. 처리 전 상위 15(불용어 포함)
2. 처리 전 표에서 빠진 낱말과 처리 뒤 표에만 있는 낱말
3. 사전에 추가하지 않았는데 문서 수가 바뀐 낱말, 차이가 큰 차례로
4. 처리 뒤 표 상위 낱말을 쓴 제목 2건과 무작위 제목 3건, 행 번호와 함께

#### 프롬프트 힌트

- **이미 있는 변수와 함수**: 준비 셀의 변수는 다시 만들지 말라고 적습니다.
- **Kiwi 두 개**: 「처리 뒤에는 새 Kiwi 에 `add_user_word(낱말, "NNG")` 로 세 낱말을 추가해 줘」.
- **표와 파일**: 「불용어는 집계 뒤에 빼 줘. 저장한 표는 다시 읽어 줘」.

**막히면**: AI 튜터에게 「사용자 사전을 추가하기 전과 뒤의 명사 목록을 둘 다 만들려면 Kiwi 를 어떻게 만들어야 하나요」 처럼 질문합니다.

#### 배경과 확인할 일

<details class="lms-extra"><summary>배경 — 2020년 국외여행 민원이 나온 시기</summary>

민원 접수일자 2020-01-01～10-17 은 국제선 항공편이 크게 줄어든 시기와 겹칩니다. 이 시기를 민원 주제의 원인으로 쓰지 않습니다. 이 겹침은 비교할 해가 없다는 한계로 위험 단락에 적습니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/1/1b/HKIA_March_26%2C_2020_flights_cancelled.jpg" alt="2020년 3월 26일 홍콩 국제공항 출발 안내판. 여러 행 오른쪽에 붉은 Cancelled 가 있고, 서울/인천 행도 여러 편이 취소로 나온다" width="640" height="386"/>
<p align="center"><em>▲ 2020년 3월 26일 홍콩 국제공항의 출발 안내판이다. 붉은 글자가 취소(Cancelled)이고 서울/인천(Seoul/ICN) 행도 여러 편이 취소됐다 (출처: Wikimedia Commons, HKIA March 26, 2020 flights cancelled.jpg, OhanaUnited, CC BY-SA 4.0)</em></p>

<img src="https://upload.wikimedia.org/wikipedia/commons/0/0f/Aislelabs-Aviation-FlightTrafficChange-COVID-19.jpg" alt="2019년 같은 주와 비교한 2020년 항공편 운항 변화. 2월부터 4월까지 주별 · 지역별 막대, 국제선 막대가 3월 셋째 주부터 크게 아래로 내려간다" width="760" height="153"/>
<p align="center"><em>▲ 2020년 항공편 운항 수를 2019년 같은 주와 비교한 변화율이다. 막대 하나가 지역 하나다. 4월 국제선 막대는 대부분 −75% 아래로 내려간다 (출처: Wikimedia Commons, Aislelabs-Aviation-FlightTrafficChange-COVID-19.jpg, Aislelabs Inc, CC BY-SA 4.0)</em></p>

</details>

<details class="lms-extra"><summary>자세히 — 오차 한계와 겹치는 구간</summary>

오차 한계의 식은 이론 「5. 토큰 빈도와 문서 비율」과 같습니다. 분모가 2만보다 작아 미션 1 보다 값이 큽니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/9/9f/Normal_approximation_to_binomial.svg" alt="이항분포 Binom(48, 0.25) 의 확률 막대와 정규분포 Norm(12, 3) 곡선을 겹친 그래프. 막대 끝이 곡선을 따라간다" width="480" height="384"/>
<p align="center"><em>▲ 빨간 막대는 48번 가운데 확률 0.25 로 일어나는 일의 횟수별 확률(이항분포)이다. 파란 곡선은 평균 12 · 표준편차 3 의 정규분포다. 막대 끝이 곡선을 따라가므로 문서 수를 정규분포로 다루고 가운데 95% 는 평균 ± 1.96 표준편차 안에 있다 (출처: Wikimedia Commons, Normal approximation to binomial.svg, CC BY-SA 3.0)</em></p>

<img src="https://upload.wikimedia.org/wikipedia/commons/b/b6/Binomial_confidence_band.svg" alt="나이 20~80 세마다 후보 A 지지 비율의 추정값 선과 95% 구간 선들, 그리고 참값 선을 그린 그래프" width="480" height="360"/>
<p align="center"><em>▲ 모의 자료에서 나이마다 후보 A 지지 비율을 추정한 검은 선과 그 95% 구간(파란 선 두 개)이다. 녹색 참값 선은 대부분의 나이에서 파란 선 사이에 있다. 구간은 추정한 비율이 표본에 따라 이만큼 달라질 수 있다는 범위다 (출처: Wikimedia Commons, Binomial confidence band.svg, Skbkekas, CC BY 3.0)</em></p>

위아래 두 막대의 오차 막대가 크게 겹치면 두 낱말은 비슷한 수준입니다. 위험 단락에서는 둘의 차례를 단정하지 않습니다.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/f/f4/95%25_confidence_interval.svg/1280px-95%25_confidence_interval.svg.png" alt="막대 다섯 개에 95% 구간 오차 막대를 그린 그래프. 가로선이 왼쪽 검은 막대의 높이이고, 빨간 막대 둘은 구간이 가로선에 닿고 파란 막대 둘은 닿지 않는다" width="560" height="340"/>
<p align="center"><em>▲ 막대마다 95% 구간을 오차 막대로 그린 그래프다. 가로선은 왼쪽 검은 막대의 높이다. 빨간 막대 둘은 구간이 이 선에 닿아 차이가 있다고 말할 수 없다. 파란 막대 둘은 구간이 선에서 떨어져 있다. 막대 길이와 구간이 닿는 곳을 함께 본다 (출처: Wikimedia Commons, 95% confidence interval.svg, Leonhard Bamberg, CC BY-SA 4.0)</em></p>

</details>

<details class="lms-extra"><summary>자세히 — AI 튜터에 맡길 일과 내가 확인할 일</summary>

| AI 튜터에 맡길 일 | 내가 반드시 확인할 일(틀려도 오류가 나지 않는 곳) |
|---|---|
| Kiwi 두 개와 명사 목록 | 처리 전 목록을 사용자 사전을 추가하기 **전의** Kiwi 로 만들었는가. 한 Kiwi 에 사전부터 추가하고 두 표를 만들면 두 표가 같아진다 |
| 문서 수 집계와 키워드 표 | 분모가 국외여행 행 수 그대로인가. 불용어를 뺀 뒤 명사가 하나도 남지 않은 제목도 분모에 남는다 |
| 저장과 그래프 | 다시 읽은 표가 15행 5열이고 낱말 열에 불용어가 없는가. 그래프를 다시 읽은 표로 그렸는가 |

</details>

In [ ]:
# 과제 · 준비 — 읽고 실행만 합니다. 이 셀이 과제에 쓸 이름을 직접 만듭니다
#  과정 복제본(원 출처와 같은 바이트)에서 민원 파일을 받아 두고(처음 한 번만), 품목이 국외여행인 민원을 골라 둡니다.
#  형태소 분석은 하지 않습니다 — 그 일은 PROMPT 셀에서 합니다. 앞 미션의 결과에 기대지 않습니다.
#  원자료를 받아 둔 뒤에는 3초 안팎입니다
import hashlib
import math
import os
import re
import time
import urllib.request
from decimal import ROUND_HALF_UP, Decimal

import pandas as pd
from kiwipiepy import Kiwi

# ---------------------------------------------------------------- 원 출처 (조회 2026-09-24)
RAW_COMPLAINTS_URL = ("https://www.data.go.kr/cmm/cmm/fileDownload.do?"
                      "atchFileId=FILE_000000002610054&fileDetailSn=1&insertDataPrcus=N")
RAW_REVIEWS_URL = "https://raw.githubusercontent.com/bab2min/corpus/master/sentiment/naver_shopping.txt"
RAW_FILES = {  # 이름: (원 출처 URL, 내 컴퓨터에 둘 파일 이름, 교안의 수를 계산한 파일의 SHA-256, 데이터 페이지)
    "complaints": (RAW_COMPLAINTS_URL, "kftc_complaints_1.csv",
                   "50c098fea2ec1d05191d07c45cd0f75b4d0ac3b5e5ac36910fe40f2aab699ebe",
                   "https://www.data.go.kr/data/15098314/fileData.do"),
    "reviews": (RAW_REVIEWS_URL, "naver_shopping.txt",
                "dc4d1aca0a148671cbe80bcb81962eee297370acab42be93c1617ce9336479c0",
                "https://github.com/bab2min/corpus/tree/master/sentiment"),
}
MIRROR_URL = "https://github.com/studio-js/korean-cx-text/raw/main/raw"  # 과정 복제본: 원 출처와 같은 바이트, 먼저 받는 곳
SUBDIRS = ("", "cache", "raw", "processed")  # 자료 폴더 아래에서 찾는 순서


def data_dir():
    # 자료 폴더: 환경변수 NLP_DATA_DIR 이 있으면 그 폴더, 없으면 ./nlp_data
    return os.environ.get("NLP_DATA_DIR", "nlp_data")


def find_local(name):
    # 자료 폴더 → cache/ → raw/ → processed/ 순서로 찾아 처음 나온 경로, 없으면 None
    for sub in SUBDIRS:
        path = os.path.join(data_dir(), sub, name)
        if os.path.isfile(path):
            return path
    return None


def sha256_of(path, chunk=1 << 20):
    # 파일의 SHA-256 16진 문자열 — 받은 파일이 교안의 파일과 같은지 대조합니다
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def _download(url, path, retries=3):
    # url 을 path.part 로 받아 두고 성공하면 True (실패하면 세 번까지 다시 시도)
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    for i in range(retries):
        try:
            urllib.request.urlretrieve(url, path + ".part")
            return True
        except Exception as e:  # network error: retry
            print(f"받기 실패 {i + 1}/{retries}: {e}")
            time.sleep(2 * (i + 1))
    return False


def _fetch(url, name):
    # 자료 폴더에 있으면 그 경로, 없으면 url 에서 받아 자료 폴더/cache/ 에 두고 경로를 반환합니다(받지 못하면 None)
    path = find_local(name)
    if path is None:
        path = os.path.join(data_dir(), "cache", name)
        print(f"{name} 을 받습니다 (처음 한 번만)")
        if not _download(url, path):
            return None
        os.replace(path + ".part", path)
    return path


def fetch_raw(name):
    # 원자료 경로: 자료 폴더를 먼저 보고, 없으면 과정 복제본(MIRROR_URL) → 원 출처 URL 순서로 받습니다
    url, fname, sha, page = RAW_FILES[name]
    path = find_local(fname)
    if path is None:
        path = _fetch(f"{MIRROR_URL}/{fname}", fname)
        if path is not None and sha256_of(path) != sha:  # 복제본이 교안의 파일과 다르면 지우고 원 출처에서 받습니다
            os.remove(path)
            path = None
    if path is None:
        path = _fetch(url, fname)
    if path is None:
        raise RuntimeError(f"원 출처에서 파일을 받지 못했습니다. 데이터 페이지: {page}")
    if sha256_of(path) != sha:
        print(f"원 출처 파일이 바뀌었습니다. 결과 수가 교안과 다를 수 있습니다(데이터 페이지: {page})")
    return path


# ---------------------------------------------------------------- 반올림 (교안의 수와 같게)
def rnd(x, digits):
    # 십진 값 기준 반올림(마지막 숫자가 5 면 올림) — 파이썬 round 는 4.045 를 4.04 로 적습니다
    x = float(x)
    if not math.isfinite(x):
        return x
    return float(Decimal(f"{x:.12g}").quantize(Decimal(1).scaleb(-digits), rounding=ROUND_HALF_UP))


def pct2(k, n, digits=2):
    # 문서 k 건 ÷ 전체 n 건을 백분율로, 소수 둘째 자리: pct2(809, 20000) -> 4.05
    return rnd(100 * k / n, digits)


def half95(k, n, digits=2):
    # 비율 k/n 의 95% 구간 오차 한계(구간 길이의 절반, ± 뒤의 값)를 %p 로: 1.96 × √(p̂(1 − p̂)/n) × 100
    p = k / n
    return rnd(100 * 1.96 * math.sqrt(p * (1 - p) / n), digits)


# ---------------------------------------------------------------- 민원 파일 읽기
# 상담 시스템에 시험 삼아 입력한 행(소비자 민원이 아님)을 찾는 정규식
TEST_INPUT_RE = re.compile(
    r"(?i)^\s*(?:test|테스트)\s*(?:입니다)?\s*\.?\s*$"
    r"|IN-TEST"
    r"|(?:인바운드|아웃바운드|녹취|수화기).*테스트|테스트.*(?:인바운드|아웃바운드|녹취|수화기)"
    r"|^\s*자동차\s*(?:상담\s*)?테스트"
    r"|^\s*\d{8}\s*[-_ ].*테스트"
    r"|테스트\s*데이터|상담\s*(?:전화\s*)?테스트|전화\s*테스트|헬프데스크\s*시험|^\s*일반\s*테스트")
EMPTY_TITLE_RE = re.compile(r"^[^가-힣A-Za-z]*$")  # 한글 음절도 로마자도 없는 제목(「-」「.」「0」「ㅇ」)
PHONE_RE = re.compile(r"(?<!\d)(?:01[016789]|0\d{1,2})[-.)\s]{0,2}\d{3,4}[-.\s]\d{4}(?!\d)")  # 전화번호 형태
PHONE_MASK = "[전화번호]"  # 원 출처가 이미 가린 번호와 같은 표지


def mask_phones(text):
    # 전화번호 형태를 [전화번호] 로 가림
    return PHONE_RE.sub(PHONE_MASK, text)


def tidy_columns(df):
    # 열 이름 정리: '사건제목(ACCIDENT_TITLE)' -> '사건제목'
    return df.rename(columns=lambda c: c.split("(")[0].strip())


def read_complaints(path, verbose=True):
    # 민원 파일(cp949) 읽기: 시험 입력·빈 제목 행 제거, 전화번호 형태 가림, 열 이름 정리, 날짜 변환
    raw = pd.read_csv(path, encoding="cp949", dtype=str)
    df = tidy_columns(raw)
    df.insert(0, "row_id", range(len(df)))  # 원 파일의 행 번호(0부터)
    title = df["사건제목"].fillna("").str.strip()
    is_test = title.str.contains(TEST_INPUT_RE)
    is_empty = title.str.match(EMPTY_TITLE_RE) & ~is_test
    df = df[~is_test & ~is_empty].copy()
    title = title[df.index]
    df["사건제목"] = title.map(mask_phones)
    df["번호가림"] = df["사건제목"].ne(title)  # 전화번호 형태를 가린 행이면 True
    df["접수일자"] = pd.to_datetime(df["접수일자"], format="%Y-%m-%d")
    df = df.reset_index(drop=True)
    if verbose:
        print(f"민원 원본 {len(raw):,}행 → 시험 입력 {int(is_test.sum())}행·빈 제목 {int(is_empty.sum())}행 제거 "
              f"→ {len(df):,}행, 전화번호 형태 가림 {int(df['번호가림'].sum())}행")
    return df


# ---------------------------------------------------------------- 표본과 형태소
def sample_rows(df, n, seed=0):
    # 고정 표본: 같은 입력 행과 같은 seed 면 같은 행이 같은 순서로 나옵니다
    return df.sample(n=n, random_state=seed)


NOUN_TAGS = ("NNG", "NNP")  # 일반 명사 + 고유 명사 (Kiwi 기본 사전)


def kiwi_nouns(texts, kiwi):
    # 문서마다 명사(NNG·NNP) 목록 — 나온 순서와 반복을 그대로 둡니다(문서 수를 집계할 때는 set 으로)
    return [[t.form for t in toks if t.tag in NOUN_TAGS] for toks in kiwi.tokenize(list(texts))]


COMPLAINTS_PATH = fetch_raw("complaints")   # 민원 원본 파일 경로 (cp949)
os.environ.setdefault("KIWI_ARCH_TYPE", "balanced")  # PROMPT 셀에서 만드는 Kiwi 도 CPU 종류(Mac·Colab)와 상관없이 같은 형태소 결과를
_all = read_complaints(COMPLAINTS_PATH, verbose=False)
OVERSEAS = (_all.loc[_all["품목명"] == "국외여행", ["row_id", "접수일자", "사건제목"]]
            .reset_index(drop=True))        # 쓸 이름 — 품목이 국외여행인 민원 전체 (시험 입력 · 빈 제목을 거른 뒤)
del _all

USER_WORDS = ["항공권", "우한폐렴", "취소수수료"]   # 쓸 이름 — 사용자 사전에 넣을 낱말 셋 (기본 사전이 둘로 쪼개는 복합어 후보)
FORM_WORDS = ["문의", "요청", "관련", "요구", "건", "후", "상담", "불만", "피해"]   # 쓸 이름 — 불용어 아홉 개 (집계한 뒤 표에서 뺍니다)
TABLE_FILE = "day66_keyword_table.csv"      # 쓸 이름 — 키워드 표 파일 이름
TABLE_COLS = ["낱말", "문서 수", "문서 비율", "95% 반길이", "처리 전 순위"]   # 쓸 이름 — 키워드 표의 다섯 열(「95% 반길이」 = 오차 한계)
print(f"민원 원본 : {os.path.basename(COMPLAINTS_PATH)} · {os.path.getsize(COMPLAINTS_PATH):,} 바이트")
print("준비 완료 — OVERSEAS · USER_WORDS · FORM_WORDS · TABLE_FILE · TABLE_COLS · Kiwi · kiwi_nouns · pct2 · half95")

In [ ]:
# 과제 — 프롬프트   # ai: prompt
# PROMPT 를 쓰고 → AI 튜터에 "위 PROMPT 대로 코드를 써 줘" → 「이 셀에 넣기」 → ▶
# 시킬 일은 AI 튜터 대화창이 아니라 아래 PROMPT 따옴표 안에 적습니다. 비어 있으면 채점과 완성본 풀이가 잠깁니다
PROMPT = """

"""
# --- 여기부터 AI 코드 ---

**브리핑 세 단락**: 아래 세 단락과 원문 표를 이 셀에 그대로 채워 제출합니다. 채점을 받기 전에 내 출력만 보고 먼저 채웁니다. 그다음 채점 뒤 열리는 완성본과 대조해 어긋난 곳만 고칩니다.

> **결론**: 2020년 1～10월 국외여행 민원 제목은 13,012건입니다. 불용어를 뺀 명사 가운데 ____ 을 쓴 제목이 ____ %(95% 구간 ____ ～ ____ %)로 가장 많았습니다. 인용한 원문 ____ 건 가운데 ____ 건이 이 주제였습니다.
>
> **위험**: ____ (사용자 사전 때문에 문서 수가 바뀐 낱말과 그 문서 수 한 문장, 이 자료가 2020년 코로나 시기의 요약된 제목이라는 한계 한 문장)
>
> **코드가 답을 만드는 방식**: ____ (분모 · 사용자 사전 세 낱말 · 불용어를 집계한 뒤 뺐다는 것 · Kiwi 버전 한 문장)

결론의 첫 빈칸은 내가 정합니다. 1위 「여행」은 품목 이름 「국외여행」에서 나온 낱말입니다. 그래서 그대로 쓸지, 그 아래 첫 주제어를 쓸지 고르고 이유를 위험 단락에 한 구절로 적습니다.

| 행 번호 | 뽑은 방법 | 사건제목(판매처 이름은 ○○) |
|---:|---|---|
| ____ | 상위 낱말 「 ____ 」 | ____ |
| ____ | 상위 낱말 「 ____ 」 | ____ |
| ____ | 무작위 | ____ |
| ____ | 무작위 | ____ |
| ____ | 무작위 | ____ |

## 정리

민원 제목 표본 2만 건을 형태소 명사로 집계하면, 불용어를 뺀 1위는 **「환불」 17.75%(± 0.53%p)** 입니다. 같은 낱말을 공백 어절로 집계하면 10.48% 로 낮게 나옵니다. 전임 담당자의 「위약금 4.05% > 해지 3.52%」도 이렇게 어절로 집계한 결과였습니다. 형태소로 집계하면 **해지 10.72% 가 위약금 5.10% 의 두 배**를 넘습니다. 리뷰에서는 품사 선택이 상위 15 가운데 11개를 바꿨고 불만의 대상이 드러나는 쪽은 명사였습니다.

이 답은 상위 3 낱말마다 원문 3건을 읽고 「문의 · 요청」이 불용어임을 확인한 뒤의 답입니다. 예를 들어 「환불」 원문 3건 가운데 하나는 행 362013 「구매한 물건이 모두오지 않아서 환불을 원함.」입니다. 소비자가 원하는 것이 「환불」 한 낱말에 들어 있습니다.

- **문자 인코딩**: 민원 원본은 cp949 파일이라 기본값(UTF-8)으로 읽으면 UnicodeDecodeError 가 났습니다. cp949 로 다시 읽고 시험 입력 · 빈 제목을 거르자 499,844행이 남았습니다
- **집계 단위**: 어절 일치는 「계약해지」 · 「해지요청」 같은 어절을 빠뜨려 오류 없이 작은 값을 냈습니다. 형태소 명사로 다시 집계하자 해지와 위약금의 차례가 반대가 됐습니다
- **품사 선택**: 명사 + 용언 설정의 1위 「하다」 18.74% 에는 불만의 대상이 드러나지 않았습니다. 명사만 설정의 상위 15 에만 있는 맛 · 가격 · 사이즈 · 반품 · 냄새는 불만의 대상이 될 수 있는 낱말입니다. 「맛」은 원문 3건 가운데 2건에서 불만의 대상이었습니다(나머지 1건은 무른 과실이 불만). 가격 · 사이즈 · 반품 · 냄새는 원문을 읽기 전이라 아직 후보입니다
- **사용자 사전**: 「취소수수료」를 한 낱말로 추가하자 국외여행 민원의 「취소」가 6,670건에서 6,555건이 됐습니다. 사전을 바꾸면 추가하지 않은 낱말의 수도 바뀝니다

<img src="https://upload.wikimedia.org/wikipedia/commons/8/8c/Hebrew_stop_words_word_cloud.png" alt="히브리어 트위터에서 가장 많이 쓰인 낱말의 워드클라우드. 큰 글자 대부분이 조사 · 접속사 같은 기능어다" width="560" height="284"/>
<p align="center"><em>▲ 히브리어 트위터 글에서 가장 많이 나온 낱말을 글자 크기로 그린 워드클라우드다. 큰 글자는 대부분 「아니다 · 을/를 · 의 · 이것」 같은 기능어다. 불용어를 빼지 않은 키워드 표도 이렇게 보인다. 워드클라우드로는 문서 비율의 구간을 나타낼 수 없다 (출처: Wikimedia Commons, Hebrew stop words word cloud.png, Saifunny, CC BY-SA 4.0)</em></p>

### 낱말 표에 옮긴 답

이론 「7. 정리와 팀 라운드」에서 본 표에 이 실습의 답을 질문마다 옮깁니다. 분석 설정 열이 비어 있으면, 같은 자료로 다시 계산해도 같은 수가 나온다고 말할 수 없습니다.

| 질문 | 분모 · 기간 | 수 | 95% 구간 | 근거 원문 | 분석 설정 |
|---|---|---|---|---|---|
| 민원 제목은 무엇을 가장 많이 말하는가 | 민원 제목 무작위 2만 건(시드 0), 2020-01-01～10-17 | 「환불」 3,550건, 17.75% | 17.22～18.28% | 미션 1 의 「환불」 원문 3건(행 362013 외 2건) | Kiwi 0.23.1 기본 사전, 명사만, 불용어 9개 제외 |
| 국외여행 민원은 무엇을 가장 많이 말하는가 | 국외여행 민원 13,012건, 같은 기간 | 내 결론의 낱말과 문서 수 | 내 결론의 구간 | 과제의 원문 5건 | Kiwi 0.23.1 + 사용자 사전 세 낱말, 명사만, 불용어 9개 제외 |

<img src="https://bab2min.github.io/kiwipiepy/images/DisambAcc.PNG" alt="한국어 형태소 분석기별 모호성 해소 정확도 막대그래프. Kiwi 0.23.0 CoNg 평균 79.99%, CoNg Global 85.85%, Komoran 46.65%, Mecab 47.05%, Kkma 55.70%, Khaiii 54.33%, Bareun 69.85%" width="640" height="393"/>
<p align="center"><em>▲ 여러 뜻으로 나뉠 수 있는 문장을 모은 평가에서 형태소 분석기마다 맞게 나눈 비율이다. 초록 막대가 평균이고 같은 Kiwi 0.23.0 도 모델에 따라 79.99% 와 85.85% 로 다르다. 분석기와 버전이 바뀌면 같은 제목도 다르게 나뉘어 키워드 표의 수가 달라질 수 있다 (출처: bab2min/Kiwi README, 모호성 해소 성능, Apache-2.0)</em></p>

마지막으로, 채점을 통과한 뒤 열린 완성본과 내 코드가 달랐던 부분 한 곳을 고릅니다. 그 부분과 거기에서 배운 것을 두 문장으로 적습니다. 이 두 문장을 과제의 결론 문장과 함께 팀 채널에 공유하고 다른 사람의 글을 읽으며 복습을 마칩니다.

<div style="background:#F2F2F2;color:#000000;padding:28px;border-radius:12px;border:1px solid #E6E6E6;margin-top:16px;border-left:4px solid #051C2C">
<h2 style="color:#000000;margin:0 0 8px 0">수고했습니다 — 비정형 데이터(자연어) 문서 비율과 형태소 토큰 실습 완료</h2>
<div style="color:#656565">cp949 민원 원본을 읽어 형태소 명사로 키워드 표를 만들고 전임 담당자의 어절 기준 표를 고쳤습니다. 리뷰의 품사 선택 두 설정과 국외여행 민원의 사용자 사전 전후도 대조했습니다.<br/>아래 마지막 한 문항만 풀고 마칩니다.</div>
</div>

> **[문제] 과제에서 사용자 사전에 「취소수수료」를 추가하자 「취소」 문서 수가 6,670 에서 6,555 로 줄었습니다. 이 수가 줄어든 이유와, 다시 집계할 때 사전 목록을 맞춰야 하는 이유로 맞는 것은?**
>
> 1. 사용자 사전이 추가하지 않은 낱말의 문서 수까지 바꿔, 사전 목록이 다르면 같은 자료에서도 수가 달라지기 때문
> 2. 「취소」가 불용어가 됐기 때문
> 3. Kiwi 가 표본을 바꿨기 때문